# AI Meeting Assistant — full neural-symbolic pipeline

**Runtime → Change runtime type → T4 GPU**, then *Run all*. Every model proposes; deterministic code decides what is kept.

| Stage | Neural part | Symbolic part (decides) | Files |
|---|---|---|---|
| 1 Transcribe | `faster-whisper large-v3` | hallucination firewall (VAD, no-speech/log-prob/compression/loop/phantom filters), confidence flags | `raw_transcript.*`, `stage1_record.json` |
| 1b Speakers | `pyannote/speaker-diarization-3.1` | word-level speaker assignment, split at speaker changes, stable labels | speakers in every line |
| 1c Cross-check | NVIDIA Parakeet TDT 0.6B (ONNX, CPU) | word alignment against Whisper → disagreement spans fed to Stage 2, never auto-applied | `asr2` block |
| 2 Refine | `Qwen3-8B` 4-bit (LLM #1) | 12 validators per edit (protected tokens, polarity, names, phonetics, LM rescoring, circuit breaker) | `refined_transcript*`, `edit_log.json` |
| 3 Document | `Qwen2.5-7B-Instruct` 4-bit (LLM #2) | verifier: verbatim quote in consecutive lines, negation context, grounding, owner/deadline rules, entity screen | `meeting_record.json/.md`, `verification_log.json` |
| 4 Calendar + report | — | spoken deadline → date with the rule recorded; RFC 5545 `.ics` with citations; citation-linked HTML | `meeting_calendar.ics`, `meeting_record.html` |
| UI | — | Gradio: report where every item links to its transcript lines, search that returns only verbatim cited lines | share link |

**Nothing is hard-coded:** settings live in `config.py` (override with `MA_<SECTION>_<FIELD>` environment variables or the
Inputs cell); the Hugging Face token comes from a Colab secret; meeting context, glossary, date, time zone and speaker
count come from the recording or from you. **Re-runnable:** every cell can be run again in any order after the setup
cells; models are released after each stage; a failing model degrades its stage with a warning instead of an error.


In [ ]:
# 1. Install — idempotent: a re-run installs nothing that is already present, and it never downgrades
#    torch / numpy / huggingface_hub / transformers / gradio (downgrading the hub is what broke the previous version).
#@markdown Runtime → Change runtime type → **T4 GPU**. If this cell restarts the runtime (only when a package that was
#@markdown already loaded changed version), run it once more and continue.
SKIP_INSTALL = False  #@param {type:"boolean"}
AUTO_RESTART = True   #@param {type:"boolean"}

import importlib.metadata as _md, importlib.util as _iu, os, subprocess, sys, tempfile
SKIP_INSTALL = SKIP_INSTALL or os.environ.get("MA_SKIP_INSTALL", "") == "1"


def _ver(dist):
    try:
        return _md.version(dist)
    except Exception:
        return None


def _vt(v):
    out = []
    for p in str(v).split("+")[0].split(".")[:3]:
        num = "".join(ch for ch in p if ch.isdigit())
        out.append(int(num) if num else 0)
    return tuple(out + [0] * (3 - len(out)))


# (pip requirement, distribution to check, minimum version or None)
REQUIREMENTS = [
    ("faster-whisper", "faster-whisper", None),
    ("nvidia-cublas-cu12", "nvidia-cublas-cu12", None),    # CTranslate2 (faster-whisper) loads CUDA 12 cuBLAS/cuDNN
    ("nvidia-cudnn-cu12", "nvidia-cudnn-cu12", None),
    ("transformers>=4.51", "transformers", "4.51"),         # Qwen3
    ("accelerate", "accelerate", None),
    ("bitsandbytes", "bitsandbytes", None),
    ("rapidfuzz", "rapidfuzz", None), ("jellyfish", "jellyfish", None), ("wordfreq", "wordfreq", None),
    ("json_repair", "json_repair", None), ("jiwer", "jiwer", None), ("num2words", "num2words", None),
    ("soundfile", "soundfile", None),
    ("pyannote.audio==3.3.2", "pyannote.audio", "3.3.2"),  # runs the pyannote/speaker-diarization-3.1 pipeline
    ("onnx-asr[hub]", "onnx-asr", None),                    # NVIDIA Parakeet TDT (ONNX) for the cross-check
    ("spacy", "spacy", None), ("g2p_en", "g2p_en", None),
    ("gradio", "gradio", None),
]
PROTECT = ["torch", "torchaudio", "torchvision", "numpy", "huggingface_hub", "transformers", "gradio"]


def _missing():
    todo = []
    for req, dist, minv in REQUIREMENTS:
        v = _ver(dist)
        if v is None or (minv and _vt(v) < _vt(minv)):
            todo.append(req)
    if _ver("onnxruntime") is None and _ver("onnxruntime-gpu") is None:
        todo.append("onnxruntime!=1.24.1,!=1.25.*,!=1.26.0")   # the versions onnx-asr excludes
    return todo


def _pip(args, constraints):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check"]
    if constraints:
        cf = tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False)
        cf.write("\n".join(constraints)); cf.close()
        cmd += ["-c", cf.name]
    r = subprocess.run(cmd + args, capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)[-1500:]


before = {d: _ver(d) for d in PROTECT}
todo = [] if SKIP_INSTALL else _missing()
if todo:
    print("installing:", ", ".join(todo))
    needed_min = {dist: minv for _, dist, minv in REQUIREMENTS if minv}
    keep_all = [f"{d}=={v}" for d, v in before.items()
                if v and not (d in needed_min and _vt(v) < _vt(needed_min[d]))]
    keep_torch = [f"{d}=={v}" for d, v in before.items() if v and d in ("torch", "torchaudio", "torchvision")]
    code, log = _pip(todo, keep_all)
    if code != 0:
        print("could not keep every installed version; retrying with only torch pinned")
        code, log = _pip(todo, keep_torch)
    if code != 0:
        print("some packages could not be installed (the pipeline degrades without them):\n", log)
    else:
        print("packages OK")
else:
    print("install skipped" if SKIP_INSTALL else "all packages already present")

if not SKIP_INSTALL and _iu.find_spec("spacy") is not None and _iu.find_spec("en_core_web_sm") is None:
    r = subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], capture_output=True, text=True)
    print("spaCy model:", "installed" if r.returncode == 0 else "not available (heuristic name detection will be used)")

after = {d: _ver(d) for d in PROTECT}
changed = [d for d in PROTECT if before[d] != after[d]]
for d in changed:
    print(f"version change: {d} {before[d]} -> {after[d]}")
loaded_changed = [d for d in changed if d in sys.modules]
if loaded_changed:
    if AUTO_RESTART:
        print("Restarting the runtime so the new versions load. Run this cell again, then continue.")
        sys.stdout.flush()
        os.kill(os.getpid(), 9)
    else:
        print("Restart the runtime (Runtime -> Restart session) before continuing:", loaded_changed)

print(subprocess.run("nvidia-smi -L 2>/dev/null || echo 'no GPU visible (CPU mode: use a small Whisper model)'; "
                     "ffmpeg -version 2>/dev/null | head -1 || echo 'ffmpeg missing'",
                     shell=True, capture_output=True, text=True).stdout.strip())


In [ ]:
# 2. Setup: project folder, optional Google Drive cache, Hugging Face token (from a secret, never typed into code)
#@markdown **Hugging Face token** (needed for pyannote): Colab left bar → 🔑 *Secrets* → add `HF_TOKEN`, enable notebook
#@markdown access. Accept the conditions of `pyannote/speaker-diarization-3.1` and `pyannote/segmentation-3.0` on
#@markdown huggingface.co with the same account. Without a token everything still runs; speakers are just not separated.
PROJECT_DIR = "/content/meeting-assistant"   #@param {type:"string"}
USE_DRIVE = True                              #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/meeting-assistant"  #@param {type:"string"}
ASK_FOR_TOKEN_IF_MISSING = True               #@param {type:"boolean"}

import ctypes, glob, os, site, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")
if not IN_COLAB and PROJECT_DIR.startswith("/content"):
    # outside Colab: next to the notebook (and the same folder again when this cell is re-run)
    PROJECT_DIR = os.environ.get("MA_PROJECT_DIR") or os.path.join(os.getcwd(), "meeting-assistant")
for d in ("stages", "outputs", "models", "clips"):
    os.makedirs(os.path.join(PROJECT_DIR, d), exist_ok=True)
Path(PROJECT_DIR, "stages", "__init__.py").touch()
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

MODELS_DIR = Path(PROJECT_DIR) / "models"
CLIPS_DIR = Path(PROJECT_DIR) / "clips"
OUT_DIR = Path(PROJECT_DIR) / "outputs"
if USE_DRIVE and IN_COLAB:
    try:
        from google.colab import drive
        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
        MODELS_DIR = Path(DRIVE_DIR) / "models"; CLIPS_DIR = Path(DRIVE_DIR) / "clips"
        os.environ.setdefault("HF_HOME", str(Path(DRIVE_DIR) / "hf_cache"))   # Whisper/pyannote/Parakeet cache survives sessions
    except Exception as e:
        print("Drive not mounted, using local folders:", str(e)[:120])
for p in (MODELS_DIR, CLIPS_DIR, OUT_DIR):
    p.mkdir(parents=True, exist_ok=True)
os.environ["MA_PROJECT_DIR"] = PROJECT_DIR
os.environ["MA_MODELS_DIR"] = str(MODELS_DIR)
os.environ["MA_OUTPUTS_DIR"] = str(OUT_DIR)

# ---- Hugging Face token: environment -> Colab secret -> (optional) hidden prompt. Stored only in this process.
def _find_token():
    for k in ("HF_TOKEN", "HUGGINGFACE_HUB_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k, "").strip():
            return os.environ[k].strip(), "environment"
    try:
        from google.colab import userdata
        v = (userdata.get("HF_TOKEN") or "").strip()
        if v:
            return v, "Colab secret"
    except Exception:
        pass
    return None, None

_tok, _src = _find_token()
if not _tok and ASK_FOR_TOKEN_IF_MISSING:
    try:
        from getpass import getpass
        _tok = getpass("Hugging Face token for pyannote (press Enter to skip diarization): ").strip() or None
        _src = "prompt" if _tok else None
    except Exception:
        _tok = None
if _tok:
    os.environ["HF_TOKEN"] = _tok
print("Hugging Face token:", f"found ({_src})" if _tok else "not set -> diarization will be skipped")
del _tok

# ---- make pip-installed cuBLAS/cuDNN (CUDA 12) visible to CTranslate2 inside THIS process (safe to repeat)
def preload_cuda_libs():
    roots = list(dict.fromkeys(site.getsitepackages() + [site.getusersitepackages()]))
    dirs, loaded = [], []
    for r in roots:
        for sub in ("cublas", "cudnn"):
            d = os.path.join(r, "nvidia", sub, "lib")
            if os.path.isdir(d):
                dirs.append(d)
                for pat in ("libcublasLt.so*", "libcublas.so*", "libcudnn.so*"):
                    for p in sorted(glob.glob(os.path.join(d, pat))):
                        try:
                            ctypes.CDLL(p, mode=ctypes.RTLD_GLOBAL); loaded.append(os.path.basename(p))
                        except OSError:
                            pass
    if dirs:
        cur = os.environ.get("LD_LIBRARY_PATH", "")
        os.environ["LD_LIBRARY_PATH"] = ":".join([d for d in dirs if d not in cur.split(":")] + ([cur] if cur else []))
    return sorted(set(loaded))

print("preloaded GPU libs:", preload_cuda_libs() or "none")
try:
    import ctranslate2
    print("GPUs visible to CTranslate2:", ctranslate2.get_cuda_device_count())
except Exception as e:
    print("ctranslate2 not importable yet:", type(e).__name__)
print("project:", PROJECT_DIR, "| model cache:", MODELS_DIR, "| clips:", CLIPS_DIR, "| outputs:", OUT_DIR)


## Modules (written to disk — this is the repository)

Shared settings, errors and the version-compatibility layer; Stage 1 (`ingest`, `stt`, `diarize`, `asr2`, `pipeline_stt`); Stage 2 (`refine`); Stage 3 (`document`); Stage 4 (`calendar_ics`, `report_html`); the orchestrator (`pipeline`) and the web app (`app`). `python pipeline.py meeting.mp3` runs everything from a terminal.

In [ ]:
%%writefile config.py
"""Every setting of the pipeline lives here. Nothing is hard-wired in the stage modules.

Three ways to change a value (later wins):
  1. the dataclass default below,
  2. an environment variable  MA_<SECTION>_<FIELD>  (e.g. MA_STT_MODEL_SIZE=small, MA_DIAR_NUM_SPEAKERS=3),
  3. at runtime: config.apply_overrides(cfg, {"field": value})  (the notebook Inputs cell and the UI do this).
Secrets (the Hugging Face token) are never stored in code or in the record; see get_hf_token().
"""
import copy
import os
from dataclasses import dataclass, field, fields, replace

SR = 16000
SCHEMA_VERSION = "1.1"


def _has_cuda() -> bool:
    try:
        import ctranslate2
        return ctranslate2.get_cuda_device_count() > 0
    except Exception:
        try:
            import torch
            return bool(torch.cuda.is_available())
        except Exception:
            return False


_CUDA = _has_cuda()


# ----------------------------------------------------------------------------------------------- paths
def project_dir() -> str:
    return os.environ.get("MA_PROJECT_DIR", os.getcwd())


def models_dir() -> str:
    """Where 4-bit LLM copies are cached. Notebook sets MA_MODELS_DIR (Drive when mounted)."""
    return os.environ.get("MA_MODELS_DIR", os.path.join(project_dir(), "models"))


def outputs_dir() -> str:
    return os.environ.get("MA_OUTPUTS_DIR", os.path.join(project_dir(), "outputs"))


# ----------------------------------------------------------------------------------------------- secrets
_TOKEN_ENV = ("HF_TOKEN", "HUGGINGFACE_HUB_TOKEN", "HUGGING_FACE_HUB_TOKEN")


def get_hf_token():
    """Hugging Face token from the environment, else from Colab secrets (key HF_TOKEN), else None.
    Never hard-code a token in a notebook: it is visible to anyone who receives the file."""
    for k in _TOKEN_ENV:
        v = os.environ.get(k, "").strip()
        if v:
            return v
    try:
        from google.colab import userdata          # Colab: key icon in the left bar -> add secret HF_TOKEN
        v = (userdata.get("HF_TOKEN") or "").strip()
        if v:
            os.environ["HF_TOKEN"] = v
            return v
    except Exception:
        pass
    return None


# ======================================================================================= Stage 1: STT
@dataclass
class STTConfig:
    # ---- model ----
    model_size: str = "large-v3"
    device: str = "cuda" if _CUDA else "cpu"
    compute_type: str = "float16" if _CUDA else "int8"
    language: str = "en"
    beam_size: int = 5
    temperature: tuple = (0.0, 0.2, 0.4)
    # ---- hallucination firewall ----
    firewall: bool = True                 # False = "bare Whisper" baseline for ablations
    vad_threshold: float = 0.5
    vad_min_silence_ms: int = 500
    vad_speech_pad_ms: int = 400
    no_speech_drop: float = 0.6
    logprob_drop: float = -1.0
    compression_drop: float = 2.4
    repetition_penalty: float = 1.0       # 1.0 = off
    no_repeat_ngram_size: int = 0         # 0 = off
    hallucination_silence_threshold: float = 0.0   # seconds; 0 = off
    consistency: bool = False             # L3 self-consistency re-decode (2x time)
    merge_short_words: int = 3            # merge segments shorter than this into their neighbour (0 = off)
    consistency_min_similarity: float = 0.6
    # ---- vocabulary biasing ----
    prompt_style: str = "sentence"        # "sentence" | "list" | "none"
    prompt_max_chars: int = 400
    use_hotwords: bool = False
    # ---- confidence flags ----
    low_conf_word_prob: float = 0.5
    critical_prob: float = 0.8
    # ---- ingest ----
    max_duration_s: int = 3 * 3600
    max_file_mb: int = 500
    min_speech_s: float = 1.5
    ffmpeg_filter: str = ""               # e.g. "highpass=f=80"; empty = none (recommended default)
    # ---- optional second ASR (L4 cross-check: NVIDIA Parakeet TDT, ONNX, CPU) ----
    use_asr2: bool = True
    asr2_model: str = "nemo-parakeet-tdt-0.6b-v2"
    asr2_quantization: str = "int8"
    asr2_max_chunk_s: float = 25.0
    asr2_providers: str = "CPUExecutionProvider"   # comma-separated onnxruntime providers


# ======================================================================================= Stage 1b: diarization
@dataclass
class DiarConfig:
    enabled: bool = True
    model_id: str = "pyannote/speaker-diarization-3.1"
    num_speakers: int = 0                 # 0 = estimate automatically
    min_speakers: int = 0                 # 0 = no bound
    max_speakers: int = 0                 # 0 = no bound
    device: str = "auto"                  # auto | cuda | cpu
    split_segments: bool = True           # split a Whisper segment where the speaker changes inside it
    min_words_per_turn: int = 2           # a speaker run shorter than this is merged into its neighbour
    max_gap_to_turn_s: float = 1.0        # a word outside every turn takes the nearest turn within this distance
    unknown_label: str = "UNKNOWN"        # label used when diarization does not run


# ======================================================================================= Stage 2: refiner
DEFAULT_CASING = ["PyTorch", "TensorFlow", "PostgreSQL", "MySQL", "MongoDB", "Redis", "Kubernetes", "Docker", "Terraform",
                  "GitHub", "GitLab", "Jenkins", "Grafana", "Prometheus", "Kafka", "Elasticsearch", "AdamW", "BERT", "CUDA",
                  "LoRA", "GPT", "LLM", "API", "CI/CD", "AWS", "GCP", "Azure", "JSON", "YAML", "SQL", "NoSQL", "REST", "gRPC",
                  "OAuth", "JWT", "HTTP", "HTTPS", "DNS", "VPN", "SSH", "GPU", "CPU", "RAM", "SDK", "UI", "UX", "QA", "KPI",
                  "OKR", "ROI", "EBITDA", "CapEx", "OpEx", "SOX", "GDPR", "HIPAA", "ICU", "MRI", "ECG", "iOS", "macOS", "Linux",
                  "JavaScript", "TypeScript", "Python", "Node.js", "Django", "FastAPI", "Streamlit", "Gradio", "Qwen",
                  "Jira", "Figma", "PowerPoint"]   # ambiguous words (react, excel, zoom, slack, whisper, java) are left out


@dataclass
class RefineConfig:
    # ---- model ----
    model_id: str      = "Qwen/Qwen3-8B"            # or "meta-llama/Llama-3.1-8B-Instruct" (gated)
    load_in_4bit: bool = True
    save_4bit_copy: bool = True                      # cache the quantised copy in models_dir() (Drive) for fast reloads
    max_new_tokens: int = 900                        # 8 edits with reasons fit in ~500; headroom for long chunks
    repetition_penalty: float = 1.05                 # guards greedy decoding against repetition loops
    # ---- chunking ----
    chunk_segments: int = 40
    chunk_overlap: int  = 5
    # ---- validators ----
    max_edits_per_100_words: float = 3.0  # cap scales with chunk length (a 600-word chunk -> 18 edits)
    max_edits_per_chunk: int = 8          # kept for backward compatibility; the per-100-words cap is what applies
    max_original_words: int  = 4          # an edit may touch at most this many words
    max_word_delta: int      = 2          # replacement may differ in length by at most this many words
    min_confidence: float    = 0.6        # model's own confidence; below this -> rejected (soft signal)
    fuzzy_min_ratio: int     = 70         # per-word / glued rapidfuzz ratio for the MEDIUM evidence tier
    phoneme_max_dist: float  = 0.35       # normalised phoneme edit distance (g2p_en) for the MEDIUM tier
    common_word_zipf: float  = 3.5        # wordfreq zipf >= this = ordinary English (memo, cache, atom, zoom are; prod, docker are not)
    # ---- evidence tiers (Whisper word probability of the original span) ----
    low_conf_p: float  = 0.5              # below: acoustically suspect -> looser phonetic threshold
    high_conf_p: float = 0.85             # above: Whisper was sure -> needs glossary / spelled letters / LM win
    # ---- extra evidence ----
    use_ner: bool = True                  # spaCy PERSON entities protect names (fallback: heuristic)
    spacy_model: str = "en_core_web_sm"
    use_lm_rescoring: bool = True         # accept only if the LLM finds the edited sentence likelier
    lm_margin: float = 1.0                # nats of total log-probability the edit must gain
    use_domain_discovery: bool = True     # one LLM call: domain + likely jargon -> context + seed glossary
    discovery_max_words: int = 1500
    # ---- canonical casing ----
    use_builtin_casing: bool = True       # DEFAULT_CASING above; extend per meeting with casing_terms
    casing_terms: list = field(default_factory=list)
    # ---- safety ----
    max_edit_ratio: float = 0.06          # circuit breaker: more than 6% of words changed -> fall back to raw
    # ---- context ----
    meeting_context: str = ""             # filled by domain discovery; set manually to override
    glossary: list = field(default_factory=list)    # user-supplied terms (names, products); vouched spellings


# ======================================================================================= Stage 3: documenter
@dataclass
class DocConfig:
    model_id: str       = "Qwen/Qwen2.5-7B-Instruct"   # distinct from the Stage 2 refiner
    load_in_4bit: bool  = True
    save_4bit_copy: bool = True
    max_new_tokens: int = 3000                       # a chunk's full JSON (quotes included) must fit; see max_chunk_words
    repetition_penalty: float = 1.05
    # chunking (map-reduce) — a chunk is sent as one call
    max_chunk_words: int = 1500                      # ~2k tokens in; 15 items x ~60-token quotes still fit the output budget
    chunk_overlap_lines: int = 4
    # verifier
    quote_min_ratio: int = 85        # rapidfuzz partial_ratio for quote-in-evidence (tolerates punctuation)
    name_min_ratio: int  = 90        # owner must appear near the quote (fuzzy, for case/punctuation)
    deadline_min_ratio: int = 85     # deadline must appear near the quote (fuzzy)
    max_quote_words: int = 40
    max_action_evidence_lines: int = 2   # an action item's evidence must be 1-2 consecutive lines
    evidence_neighbour_lines: int = 1    # also search this many lines around the cited ones (quote still verbatim)
    owner_window_words: int = 8          # owner/deadline must occur inside the quote or within this many words of it
    statement_overlap_min: float = 0.34  # share of the statement's content words that must appear in its quote
    use_nli: bool = False                # optional: cross-encoder entailment check (quote must entail statement)
    nli_model: str = "cross-encoder/nli-deberta-v3-small"
    dedupe_ratio: int = 85               # fuzzy statement similarity treated as the same item across chunks
    meeting_context: str = ""            # filled from stage2_record.json (domain discovery); set manually to override


# ======================================================================================= Stage 4: calendar (.ics)
@dataclass
class CalendarConfig:
    enabled: bool = True
    meeting_date: str = ""          # YYYY-MM-DD; "" = recording metadata -> date in file name -> processing date
    meeting_time: str = ""          # HH:MM (24h) local time; "" = recording metadata, else all-day meeting event
    timezone: str = ""              # IANA name, e.g. "Asia/Kolkata"; "" = TZ env var -> system local time zone
    date_order: str = "DMY"         # how to read numeric dates like 10/11: DMY or MDY
    week_end_day: int = 4           # 0=Mon ... 4=Fri: what "end of the week" means
    include_meeting_event: bool = True
    include_vtodo: bool = True      # VTODO for every action item (Outlook/Apple/Thunderbird); Google imports VEVENT only
    alarm_minutes_before: int = 0   # >0 adds a reminder to each action-item event
    default_event_minutes: int = 30 # length of a timed action-item event ("by Friday 5pm")
    prodid: str = "-//meeting-assistant//citation-grounded minutes//EN"
    uid_domain: str = "meeting-assistant.local"


CFG = STTConfig()
DIAR_CFG = DiarConfig()
REFINE_CFG = RefineConfig()
DOC_CFG = DocConfig()
CAL_CFG = CalendarConfig()

_SECTIONS = {"STT": CFG, "DIAR": DIAR_CFG, "REFINE": REFINE_CFG, "DOC": DOC_CFG, "CAL": CAL_CFG}


# ----------------------------------------------------------------------------------------------- overrides
def _coerce(value, current):
    """Convert a string / scalar to the type of the field's current value."""
    if isinstance(current, bool):
        if isinstance(value, str):
            return value.strip().lower() in ("1", "true", "yes", "y", "on")
        return bool(value)
    if isinstance(current, int) and not isinstance(current, bool):
        return int(float(value))
    if isinstance(current, float):
        return float(value)
    if isinstance(current, tuple):
        if isinstance(value, str):
            return tuple(float(x) for x in value.split(",") if x.strip())
        return tuple(value)
    if isinstance(current, list):
        if isinstance(value, str):
            return [x.strip() for x in value.split(",") if x.strip()]
        return list(value)
    return value if value is None or isinstance(value, str) else str(value)


def apply_overrides(cfg, overrides: dict, strict: bool = False):
    """Set fields on a config dataclass in place from a dict; unknown keys are ignored (or raise if strict)."""
    names = {f.name for f in fields(cfg)}
    for k, v in (overrides or {}).items():
        if k not in names:
            if strict:
                raise KeyError(f"{type(cfg).__name__} has no field {k!r}")
            continue
        setattr(cfg, k, _coerce(v, getattr(cfg, k)))
    return cfg


def apply_env(prefix: str = "MA_"):
    """MA_<SECTION>_<FIELD>=value  ->  cfg.field = value. Called once at import."""
    for sec, cfg in _SECTIONS.items():
        for f in fields(cfg):
            key = f"{prefix}{sec}_{f.name.upper()}"
            if key in os.environ:
                try:
                    setattr(cfg, f.name, _coerce(os.environ[key], getattr(cfg, f.name)))
                except Exception as e:     # a bad env value must not break import
                    print(f"config: ignoring {key}={os.environ[key]!r} ({type(e).__name__})")
    # legacy names used by earlier notebooks
    for env, attr in (("WHISPER_MODEL", "model_size"), ("WHISPER_DEVICE", "device"), ("WHISPER_COMPUTE", "compute_type")):
        if os.environ.get(env):
            setattr(CFG, attr, os.environ[env])


apply_env()


def with_changes(**kw) -> STTConfig:
    """Copy of the default STT config with some fields changed (used by the comparison grid)."""
    return replace(CFG, **kw)


def fresh(cfg):
    """Independent copy for one run (stages mutate meeting_context etc.; runs must not leak into each other)."""
    return copy.deepcopy(cfg)


def snapshot() -> dict:
    """All current settings (for the run record). Contains no secrets."""
    from dataclasses import asdict
    return {sec.lower(): asdict(cfg) for sec, cfg in _SECTIONS.items()}


In [ ]:
%%writefile errors.py
class PipelineError(Exception):
    code = "pipeline_error"

    def __init__(self, user_message: str):
        super().__init__(user_message)
        self.user_message = user_message


class UnsupportedFormatError(PipelineError):
    code = "unsupported_format"


class EmptyAudioError(PipelineError):
    code = "empty_audio"


class DecodeError(PipelineError):
    code = "decode_error"


class NoSpeechError(PipelineError):
    code = "no_speech"


class TooLargeError(PipelineError):
    code = "too_large"


class ModelError(PipelineError):
    code = "model_error"


class RefineError(PipelineError):
    code = "refine_error"


class DocumentError(PipelineError):
    code = "document_error"


In [ ]:
%%writefile compat.py
"""Compatibility layer: lets pyannote.audio 3.x (the speaker-diarization-3.1 pipeline) run on the current Colab stack
WITHOUT downgrading anything that transformers / gradio depend on.

  * torchaudio >= 2.9 removed AudioMetaData, list_audio_backends and info(). pyannote 3.x references them at import
    time. We add equivalents (audio itself is passed to pyannote in memory, so torchaudio I/O is never used).
  * huggingface_hub >= 1.0 removed the `use_auth_token` argument. pyannote 3.x still passes it. We translate it to
    `token` in a thin wrapper instead of pinning huggingface_hub < 0.29 (which breaks transformers 5 and gradio 6).
  * torch >= 2.6 loads checkpoints with weights_only=True. pyannote checkpoints pickle a few pyannote classes, so
    loading them needs weights_only=False. This is done ONLY inside `trusted_torch_load()` while the pyannote models
    are being loaded, never globally.
  * numpy 2 removed np.NaN / np.NAN aliases that some pyannote helpers still use.
Every function here is idempotent: running it again (or re-running the notebook) changes nothing.
"""
import contextlib
import functools
import inspect
import sys
from dataclasses import dataclass


def patch_numpy():
    try:
        import numpy as np
        for name in ("NaN", "NAN"):
            if not hasattr(np, name):
                setattr(np, name, np.nan)
        if not hasattr(np, "Inf"):
            np.Inf = np.inf
    except Exception:
        pass


def patch_torchaudio():
    """Add the torchaudio symbols pyannote 3.x expects. Safe to call many times; returns a list of what was added."""
    added = []
    try:
        import torchaudio
    except Exception:
        return added
    if not hasattr(torchaudio, "AudioMetaData"):
        AudioMetaData = None
        try:
            from torchaudio.backend.common import AudioMetaData        # torchaudio < 2.9
        except Exception:
            @dataclass
            class AudioMetaData:                                          # noqa: F811
                sample_rate: int
                num_frames: int
                num_channels: int
                bits_per_sample: int = 16
                encoding: str = "PCM_S"
        torchaudio.AudioMetaData = AudioMetaData
        if "torchaudio.backend.common" not in sys.modules:
            backend_mod = type(sys)("torchaudio.backend.common")
            backend_mod.AudioMetaData = AudioMetaData
            sys.modules["torchaudio.backend.common"] = backend_mod
        added.append("AudioMetaData")
    if not hasattr(torchaudio, "list_audio_backends"):
        torchaudio.list_audio_backends = lambda: ["soundfile"]
        added.append("list_audio_backends")
    if not hasattr(torchaudio, "info"):
        def _info(uri, *args, **kwargs):
            import soundfile as sf
            i = sf.info(uri if not hasattr(uri, "read") else uri)
            return torchaudio.AudioMetaData(sample_rate=int(i.samplerate), num_frames=int(i.frames),
                                            num_channels=int(i.channels))
        torchaudio.info = _info
        added.append("info")
    return added


def _accepts(fn, name: str) -> bool:
    try:
        params = inspect.signature(fn).parameters
        return name in params or any(p.kind == inspect.Parameter.VAR_KEYWORD for p in params.values())
    except (TypeError, ValueError):
        return True


def _wrap_hub_fn(fn):
    """Return fn wrapped so that use_auth_token=... becomes token=... and unknown kwargs are dropped."""
    if getattr(fn, "_ma_compat", False):
        return fn
    try:
        params = inspect.signature(fn).parameters
        has_varkw = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in params.values())
        names = set(params)
    except (TypeError, ValueError):
        has_varkw, names = True, set()

    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        if "use_auth_token" in kwargs and "use_auth_token" not in names:
            tok = kwargs.pop("use_auth_token")
            if tok is not None and "token" not in kwargs:
                kwargs["token"] = tok
        if not has_varkw and names:
            kwargs = {k: v for k, v in kwargs.items() if k in names}
        return fn(*args, **kwargs)

    wrapper._ma_compat = True
    wrapper._ma_original = fn
    return wrapper


def patch_huggingface_hub():
    """Make hf_hub_download / snapshot_download accept use_auth_token again (translated to token)."""
    patched = []
    try:
        import huggingface_hub
    except Exception:
        return patched
    for fname in ("hf_hub_download", "snapshot_download"):
        fn = getattr(huggingface_hub, fname, None)
        if fn is None:
            continue
        if not getattr(fn, "_ma_compat", False):
            try:
                if "use_auth_token" in inspect.signature(fn).parameters:
                    continue                           # this hub version still supports use_auth_token natively
            except (TypeError, ValueError):
                pass
        w = _wrap_hub_fn(fn)
        setattr(huggingface_hub, fname, w)
        for modname in ("huggingface_hub.file_download", "huggingface_hub._snapshot_download"):
            m = sys.modules.get(modname)
            if m is not None and hasattr(m, fname):
                setattr(m, fname, w)
        patched.append(fname)
    patch_loaded_modules()
    return patched


def patch_loaded_modules():
    """pyannote modules bind hf_hub_download at import time; rebind them to the wrapper if they are loaded."""
    try:
        import huggingface_hub
    except Exception:
        return
    w = getattr(huggingface_hub, "hf_hub_download", None)
    if w is None or not getattr(w, "_ma_compat", False):
        return
    for name, mod in list(sys.modules.items()):
        if name.startswith("pyannote.") and mod is not None and getattr(mod, "hf_hub_download", None) is not None:
            if getattr(mod.hf_hub_download, "_ma_compat", False) is False:
                try:
                    mod.hf_hub_download = w
                except Exception:
                    pass


@contextlib.contextmanager
def trusted_torch_load():
    """Within this block torch.load uses weights_only=False (pyannote checkpoints from the Hugging Face Hub only)."""
    try:
        import torch
    except Exception:
        yield
        return
    original = torch.load
    if getattr(original, "_ma_compat", False):
        original = original._ma_original

    @functools.wraps(original)
    def _load(*args, **kwargs):
        kwargs["weights_only"] = False
        return original(*args, **kwargs)

    _load._ma_compat = True
    _load._ma_original = original
    torch.load = _load
    try:
        yield
    finally:
        torch.load = original


def apply_all() -> dict:
    """Apply every shim; returns what was changed (for the setup log)."""
    patch_numpy()
    return {"torchaudio": patch_torchaudio(), "huggingface_hub": patch_huggingface_hub()}


def import_pyannote():
    """Import pyannote.audio.Pipeline with all shims in place. Returns the class, or raises ImportError."""
    apply_all()
    from pyannote.audio import Pipeline          # noqa: E402
    patch_loaded_modules()
    return Pipeline


def dtype_kwarg(dtype) -> dict:
    """transformers >= 4.56 calls it `dtype`, older versions `torch_dtype`."""
    try:
        import transformers
        major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
        return {"dtype": dtype} if (major, minor) >= (4, 56) else {"torch_dtype": dtype}
    except Exception:
        return {"torch_dtype": dtype}


In [ ]:
%%writefile stages/textutil.py
import re

try:
    from num2words import num2words as _n2w
except Exception:       # works without num2words, just with weaker number normalisation
    _n2w = None

# ---- shared token definitions (identical in Stage 1 textutil.py and Stage 2 Cell 5) ----
_re = re

NUMBER_WORDS = set("""zero one two three four five six seven eight nine ten eleven twelve thirteen
fourteen fifteen sixteen seventeen eighteen nineteen twenty thirty forty fifty sixty seventy eighty
ninety hundred thousand million billion first second third fourth fifth sixth seventh eighth ninth
tenth eleventh twelfth twentieth thirtieth hundredth half quarter dozen once twice thrice""".split())
NEGATIONS = set("""not no never none nothing neither nor cannot without nobody don't doesn't didn't won't
wouldn't can't couldn't shouldn't isn't aren't wasn't weren't hasn't haven't hadn't mustn't""".split())
COMMIT = set("will shall should must may might can could would maybe agreed agree decided decide commit".split())
WEEKDAYS = set("monday tuesday wednesday thursday friday saturday sunday mon tue tues wed thu thurs fri sat sun".split())
MONTHS = set("""january february march april may june july august september october november december
jan feb mar apr jun jul aug sep sept oct nov dec""".split())
TIME_WORDS = set("am pm noon midnight today tomorrow yesterday tonight o'clock eod eow eom".split())
PROTECTED_WORDS = NUMBER_WORDS | NEGATIONS | COMMIT | WEEKDAYS | MONTHS | TIME_WORDS

_TOKEN_RE = _re.compile(r"[A-Za-z0-9'$€£₹%]+(?:[./:-][A-Za-z0-9']+)*")


def tokens(text: str) -> list:
    """Word tokens. Keeps 0.08, 10:30, CI/CD, $500, 23rd, don't as single tokens."""
    return _TOKEN_RE.findall(text or "")


def is_critical(word: str) -> bool:
    """Numbers, dates, times, negations and commitment words: the tokens the rubric says must never change."""
    w = _re.sub(r"[^a-z0-9'$€£₹%./:-]", "", str(word).lower())
    if not w:
        return False
    return (w in PROTECTED_WORDS or w.endswith("n't") or any(ch.isdigit() for ch in w)
            or any(ch in w for ch in "$€£₹%"))


def protected_tokens(text: str) -> list:
    """Lower-cased critical tokens in order of appearance (compared before/after every edit)."""
    return [t.lower() for t in tokens(text) if is_critical(t)]
# ---- end shared token definitions ----


def _digits_to_words(s: str) -> str:
    s = re.sub(r"(?<=\d),(?=\d{3})", "", s)            # 1,000 -> 1000
    s = re.sub(r"(?<=\d)\.(?=\d)", " point ", s)        # 3.5 -> 3 point 5
    if _n2w is None:
        return s

    def ordinal(m):
        try:
            return " " + _n2w(int(m.group(1)), to="ordinal") + " "
        except Exception:
            return m.group(0)

    def cardinal(m):
        try:
            return " " + _n2w(int(m.group())) + " "
        except Exception:
            return m.group(0)

    s = re.sub(r"(\d+)(st|nd|rd|th)\b", ordinal, s)
    s = re.sub(r"\d+", cardinal, s)
    return s


def norm_text(s: str) -> str:
    """Lowercase, numbers->words, no punctuation. Used for WER, term checks and alignment."""
    s = s.lower().replace("\u2019", "'")
    s = s.replace("%", " percent ").replace("&", " and ").replace("/", " ")
    s = _digits_to_words(s)
    s = s.replace("-", " ")
    s = re.sub(r"[^a-z0-9']", " ", s)
    return " ".join(s.split())


In [ ]:
%%writefile stages/ingest.py
import inspect
import json
import os
import subprocess
import tempfile
import wave

import numpy as np

from config import CFG, SR
from errors import (DecodeError, EmptyAudioError, NoSpeechError,
                    TooLargeError, UnsupportedFormatError)

ALLOWED = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".opus", ".webm", ".mov", ".m4v", ".aif", ".aiff", ".3gp",
           ".mp4", ".aac", ".wma", ".mkv"}


def _run(cmd):
    return subprocess.run(cmd, capture_output=True, text=True)


def probe(path: str) -> dict:
    r = _run(["ffprobe", "-v", "error", "-show_entries",
              "format=duration:format_tags=creation_time,date:stream=codec_type:stream_tags=creation_time",
              "-of", "json", path])
    if r.returncode != 0:
        raise DecodeError("This file could not be read as audio. It may be corrupted or not an audio file.")
    try:
        info = json.loads(r.stdout or "{}")
    except Exception:
        raise DecodeError("This file could not be read as audio.")
    if not any(s.get("codec_type") == "audio" for s in info.get("streams", [])):
        raise DecodeError("No audio track was found in this file.")
    try:
        dur = float(info.get("format", {}).get("duration") or 0)
    except (TypeError, ValueError):
        dur = 0.0
    tags = dict(info.get("format", {}).get("tags", {}) or {})
    for st in info.get("streams", []):
        for k, v in (st.get("tags") or {}).items():
            tags.setdefault(k, v)
    created = tags.get("creation_time") or tags.get("date") or None
    return {"duration_s": dur, "creation_time": created}


def convert_to_wav(path: str, ffmpeg_filter: str = "") -> str:
    """16 kHz mono 16-bit PCM. No loudness normalisation by default (it can amplify noise)."""
    out = os.path.join(tempfile.mkdtemp(), "audio_16k_mono.wav")
    cmd = ["ffmpeg", "-y", "-i", path, "-vn", "-ac", "1", "-ar", str(SR)]
    if ffmpeg_filter:
        cmd += ["-af", ffmpeg_filter]
    cmd += ["-c:a", "pcm_s16le", out]
    r = _run(cmd)
    if r.returncode != 0 or not os.path.exists(out) or os.path.getsize(out) < 1000:
        raise DecodeError("Audio conversion failed. The file may be damaged.")
    return out


def load_wav(wav_path: str) -> np.ndarray:
    """Read 16-bit PCM WAV into float32 [-1, 1] (no PyAV needed)."""
    try:
        with wave.open(wav_path, "rb") as wf:
            if wf.getsampwidth() != 2:
                raise DecodeError("Unexpected audio format after conversion.")
            frames = wf.readframes(wf.getnframes())
            audio = np.frombuffer(frames, dtype=np.int16).astype(np.float32) / 32768.0
            if wf.getnchannels() > 1:
                audio = audio.reshape(-1, wf.getnchannels()).mean(axis=1)
            return audio
    except DecodeError:
        raise
    except Exception:
        raise DecodeError("Could not read the converted audio.")


def _energy_segments(audio: np.ndarray, cfg=CFG):
    """Adaptive energy fallback if Silero VAD is unavailable. Threshold follows the noise floor,
    so a constant tone or steady noise is NOT mistaken for speech."""
    frame = int(0.03 * SR)
    n = len(audio) // frame
    if n == 0:
        return []
    rms = np.sqrt((audio[: n * frame].reshape(n, frame) ** 2).mean(axis=1))
    thr = max(0.002, 3.0 * float(np.percentile(rms, 10)))
    mask = rms > thr
    segs, start = [], None
    for i, m in enumerate(mask):
        if m and start is None:
            start = i
        elif not m and start is not None:
            segs.append([start * 0.03, i * 0.03])
            start = None
    if start is not None:
        segs.append([start * 0.03, n * 0.03])
    merged = []
    gap = cfg.vad_min_silence_ms / 1000.0
    for s, e in segs:
        if merged and s - merged[-1][1] <= gap:
            merged[-1][1] = e
        else:
            merged.append([s, e])
    pad = cfg.vad_speech_pad_ms / 1000.0
    total = len(audio) / SR
    return [(max(0.0, s - pad), min(total, e + pad)) for s, e in merged if e - s >= 0.2]


def vad_segments(audio: np.ndarray, cfg=CFG):
    """Returns ([(start_s, end_s), ...], method). Silero VAD (bundled with faster-whisper) with fallback."""
    total_s = len(audio) / SR
    if total_s == 0:
        return [], "none"
    try:
        from faster_whisper.vad import VadOptions, get_speech_timestamps
        params = dict(threshold=cfg.vad_threshold,
                      min_silence_duration_ms=cfg.vad_min_silence_ms,
                      speech_pad_ms=cfg.vad_speech_pad_ms)
        allowed = set(inspect.signature(VadOptions).parameters)
        opts = VadOptions(**{k: v for k, v in params.items() if k in allowed})
        try:
            ts = get_speech_timestamps(audio, opts, sampling_rate=SR)
        except TypeError:
            ts = get_speech_timestamps(audio, opts)
        if not ts:
            return [], "silero"
        scale = 1.0 if ts[-1]["end"] <= total_s + 1.0 else 1.0 / SR   # seconds or samples?
        return [(t["start"] * scale, t["end"] * scale) for t in ts], "silero"
    except Exception:
        return _energy_segments(audio, cfg), "energy"


def validate_and_convert(path: str, cfg=CFG) -> dict:
    if not path or not os.path.exists(path):
        raise EmptyAudioError("No file was provided.")
    size = os.path.getsize(path)
    if size == 0:
        raise EmptyAudioError("The uploaded file is empty (0 bytes).")
    if size > cfg.max_file_mb * 1024 * 1024:
        raise TooLargeError(f"File is larger than {cfg.max_file_mb} MB.")
    ext = os.path.splitext(path)[1].lower()
    if ext not in ALLOWED:
        raise UnsupportedFormatError(
            f"Unsupported file type '{ext}'. Supported: {', '.join(sorted(ALLOWED))}.")
    meta = probe(path)
    if meta["duration_s"] < 0.5:
        raise EmptyAudioError("The recording is empty or too short to process.")
    if meta["duration_s"] > cfg.max_duration_s:
        raise TooLargeError(f"Recording is longer than {cfg.max_duration_s // 3600} hours.")
    wav = convert_to_wav(path, cfg.ffmpeg_filter)
    audio = load_wav(wav)
    segs, method = vad_segments(audio, cfg)
    speech_s = sum(e - s for s, e in segs)
    if speech_s < cfg.min_speech_s:
        raise NoSpeechError("No speech was detected in this recording. "
                            "It may be silent or contain only noise or music.")
    return {"wav_path": wav, "audio": audio, "duration_s": meta["duration_s"],
            "speech_s": round(speech_s, 1), "speech_segments": segs, "vad_method": method,
            "creation_time": meta.get("creation_time")}


In [ ]:
%%writefile stages/stt.py
import difflib
import gc
import inspect
from dataclasses import asdict
from functools import lru_cache

import numpy as np

from config import CFG, SR
from errors import ModelError
from stages.textutil import norm_text, is_critical

PHANTOMS = [
    "thank you for watching", "thanks for watching", "please subscribe",
    "like and subscribe", "subtitles by", "see you in the next video",
    "transcribed by", "amara.org",
]


# ----------------------------------------------------------------- model handling
@lru_cache(maxsize=1)
def _load_model(model_size, device, compute_type):
    from faster_whisper import WhisperModel
    return WhisperModel(model_size, device=device, compute_type=compute_type)


def get_model(cfg=CFG):
    try:
        return _load_model(cfg.model_size, cfg.device, cfg.compute_type)
    except Exception as e:
        raise _model_error(e)


def free_gpu():
    """Release the Whisper model (call before loading an LLM so both fit on a 16 GB T4)."""
    _load_model.cache_clear()
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass


def _model_error(e) -> ModelError:
    msg = str(e)
    low = msg.lower()
    if any(k in low for k in ("cudnn", "cublas", "libcuda", "cuda")):
        return ModelError(
            "GPU libraries could not be loaded (" + msg[:140] + "). Re-run the 'GPU setup' cell, "
            "restart the runtime if you just installed packages, or use WHISPER_DEVICE=cpu WHISPER_MODEL=small.")
    return ModelError("Speech model failed: " + msg[:200])


def _filter_kwargs(fn, kwargs):
    """Drop arguments this faster-whisper version does not know (protects against version drift)."""
    try:
        params = inspect.signature(fn).parameters
        if any(p.kind == inspect.Parameter.VAR_KEYWORD for p in params.values()):
            return dict(kwargs), []
        ok = {k: v for k, v in kwargs.items() if k in params}
        return ok, sorted(set(kwargs) - set(ok))
    except (TypeError, ValueError):
        return dict(kwargs), []


# ----------------------------------------------------------------- firewall helpers
def has_loop(text: str) -> bool:
    toks = norm_text(text).split()
    for n, reps in ((1, 6), (2, 4), (3, 4), (4, 4)):
        for i in range(0, len(toks) - n * reps + 1):
            chunk = toks[i:i + n]
            if all(toks[i + k * n: i + (k + 1) * n] == chunk for k in range(reps)):
                return True
    return False


def drop_reason(seg: dict, prev_kept_text: str, dup_count: int, cfg=CFG):
    t = norm_text(seg["text"])
    if not t:
        return "empty"
    if seg["no_speech_prob"] > cfg.no_speech_drop and seg["avg_logprob"] < cfg.logprob_drop:
        return "no_speech_low_confidence"
    if seg["compression_ratio"] > cfg.compression_drop:
        return "high_compression_ratio(loop)"
    if has_loop(seg["text"]):
        return "repetition_loop"
    if len(t.split()) <= 12 and any(p in t for p in PHANTOMS) and \
            (seg["no_speech_prob"] > 0.2 or seg["avg_logprob"] < -0.7):
        return "phantom_phrase"
    if t == prev_kept_text and dup_count >= 2:
        return "duplicate_segment"
    return None


def detect_language_safe(model, audio, speech_segments=None):
    try:
        start = int((speech_segments[0][0] if speech_segments else 0.0) * SR)
        clip = audio[start:start + SR * 30]
        if len(clip) < SR:
            clip = audio[: SR * 30]
        res = model.detect_language(clip)
        return res[0], float(res[1])
    except Exception:
        return None, None


def build_prompt(glossary, cfg=CFG):
    if not glossary or cfg.prompt_style == "none":
        return None
    kept, used = [], 0
    for term in glossary:
        if used + len(term) + 2 > cfg.prompt_max_chars:
            break
        kept.append(term)
        used += len(term) + 2
    if not kept:
        return None
    terms = ", ".join(kept)
    if cfg.prompt_style == "list":
        return terms
    return f"This is a meeting transcript. Vocabulary: {terms}."


def build_kwargs(cfg, glossary):
    kw = dict(language=cfg.language, beam_size=cfg.beam_size, word_timestamps=True,
              initial_prompt=build_prompt(glossary, cfg))
    if cfg.use_hotwords and glossary:
        kw["hotwords"] = ", ".join(glossary)[: cfg.prompt_max_chars]
    if cfg.firewall:
        kw.update(
            vad_filter=True,
            vad_parameters=dict(threshold=cfg.vad_threshold,
                                min_silence_duration_ms=cfg.vad_min_silence_ms,
                                speech_pad_ms=cfg.vad_speech_pad_ms),
            condition_on_previous_text=False,
            temperature=list(cfg.temperature))
        if cfg.repetition_penalty and cfg.repetition_penalty != 1.0:
            kw["repetition_penalty"] = cfg.repetition_penalty
        if cfg.no_repeat_ngram_size:
            kw["no_repeat_ngram_size"] = cfg.no_repeat_ngram_size
        if cfg.hallucination_silence_threshold and cfg.hallucination_silence_threshold > 0:
            kw["hallucination_silence_threshold"] = cfg.hallucination_silence_threshold
    else:   # "bare Whisper": what most people run by default
        kw.update(vad_filter=False, condition_on_previous_text=True)
    return kw


# ----------------------------------------------------------------- short-segment merge
def merge_short_segments(segs: list, min_words: int, max_gap_s: float = 2.0) -> list:
    """Whisper sometimes emits fragments like '3.' / 'The launch.' as separate segments. A fragment shorter
    than min_words is merged FORWARD into the next segment when the previous one ended a sentence (it is
    the start of a new thought, e.g. a list marker), otherwise BACKWARD into the previous one. Word
    timestamps, confidences and flags are carried over; original ids are kept in 'merged_from'."""
    if min_words <= 0 or len(segs) < 2:
        return segs

    def _absorb(dst, src, prepend):
        dst["text"] = (src["text"] + " " + dst["text"]).strip() if prepend else (dst["text"] + " " + src["text"]).strip()
        dst["start"] = min(dst["start"], src["start"]); dst["end"] = max(dst["end"], src["end"])
        a, b = (src, dst) if prepend else (dst, src)
        dst["words"] = a.get("words", []) + b.get("words", [])
        dst["low_conf_words"] = a.get("low_conf_words", []) + b.get("low_conf_words", [])
        dst["critical_low_conf"] = a.get("critical_low_conf", []) + b.get("critical_low_conf", [])
        dst["avg_logprob"] = min(dst["avg_logprob"], src["avg_logprob"])
        dst["no_speech_prob"] = max(dst["no_speech_prob"], src["no_speech_prob"])
        dst["compression_ratio"] = max(dst["compression_ratio"], src["compression_ratio"])
        dst["merged_from"] = a.get("merged_from", [a["id"]]) + b.get("merged_from", [b["id"]])

    out, i = [], 0
    while i < len(segs):
        seg = segs[i]
        if len(norm_text(seg["text"]).split()) < min_words:
            prev_ended = bool(out) and out[-1]["text"].rstrip()[-1:] in ".!?"
            nxt = segs[i + 1] if i + 1 < len(segs) else None
            if nxt is not None and (not out or prev_ended) and nxt["start"] - seg["end"] <= max_gap_s:
                _absorb(nxt, seg, prepend=True); i += 1; continue
            if out and seg["start"] - out[-1]["end"] <= max_gap_s:
                _absorb(out[-1], seg, prepend=False); i += 1; continue
        out.append(seg); i += 1
    for n, seg in enumerate(out, 1):
        seg.setdefault("merged_from", [seg["id"]])
        seg["id"] = n
    return out


# ----------------------------------------------------------------- L3 self-consistency
def _annotate_stability(model, audio, kept, kwargs, cfg):
    kw = dict(kwargs)
    kw.update(beam_size=1, temperature=[0.0], initial_prompt=None, condition_on_previous_text=False)
    for k in ("hotwords", "hallucination_silence_threshold", "repetition_penalty", "no_repeat_ngram_size"):
        kw.pop(k, None)
    kw["word_timestamps"] = False
    kw, _ = _filter_kwargs(model.transcribe, kw)
    it, _info = model.transcribe(audio, **kw)
    second = [(s.start, s.end, norm_text(s.text)) for s in it]
    for seg in kept:
        a = norm_text(seg["text"]).split()
        overlap = []
        for s0, s1, txt in second:
            ov = min(seg["end"], s1) - max(seg["start"], s0)
            if ov > 0.3 * max(0.1, min(seg["end"] - seg["start"], s1 - s0)):
                overlap.append(txt)
        b = " ".join(overlap).split()
        ratio = difflib.SequenceMatcher(None, a, b, autojunk=False).ratio() if (a and b) else 0.0
        seg["stability"] = round(ratio, 2)
        seg["unstable"] = ratio < cfg.consistency_min_similarity


# ----------------------------------------------------------------- main entry
def transcribe(audio, glossary=None, cfg=None, on_progress=None, speech_segments=None) -> dict:
    """audio: float32 mono 16 kHz numpy array. Returns the raw-transcript dict."""
    cfg = cfg or CFG
    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim != 1 or audio.size == 0:
        raise ModelError("Internal error: audio must be a non-empty mono array.")
    total_s = len(audio) / SR
    model = get_model(cfg)

    warnings = []
    lang, lang_p = detect_language_safe(model, audio, speech_segments)
    if lang and lang != cfg.language and lang_p and lang_p > 0.7:
        warnings.append(f"Audio may not be '{cfg.language}' (detected '{lang}', p={lang_p:.2f}). "
                        "Results may be inaccurate.")

    kwargs = build_kwargs(cfg, glossary)
    kwargs, ignored = _filter_kwargs(model.transcribe, kwargs)
    if ignored:
        warnings.append("Ignored options not supported by this faster-whisper version: " + ", ".join(ignored))

    kept, dropped = [], []
    prev_text, dup = "", 0
    try:
        seg_iter, info = model.transcribe(audio, **kwargs)
        for s in seg_iter:
            words = [{"w": w.word, "start": round(w.start, 2), "end": round(w.end, 2),
                      "p": round(w.probability, 3)} for w in (s.words or [])]
            seg = {"id": len(kept) + len(dropped) + 1,
                   "start": round(s.start, 2), "end": round(s.end, 2), "text": s.text.strip(),
                   "avg_logprob": round(s.avg_logprob, 3), "no_speech_prob": round(s.no_speech_prob, 3),
                   "compression_ratio": round(s.compression_ratio, 2), "words": words}
            if on_progress and total_s > 0:
                on_progress(min(1.0, s.end / total_s))
            reason = drop_reason(seg, prev_text, dup, cfg) if cfg.firewall else None
            if reason:
                seg["drop_reason"] = reason
                dropped.append(seg)
                continue
            norm_seg = norm_text(seg["text"])
            dup = dup + 1 if norm_seg == prev_text else 0
            prev_text = norm_seg
            seg["low_conf_words"] = [w["w"].strip() for w in words if w["p"] < cfg.low_conf_word_prob]
            seg["critical_low_conf"] = [w["w"].strip() for w in words
                                        if is_critical(w["w"]) and w["p"] < cfg.critical_prob]
            kept.append(seg)
        if getattr(cfg, "merge_short_words", 0) and kept:
            kept = merge_short_segments(kept, cfg.merge_short_words)
        if cfg.consistency and kept:
            _annotate_stability(model, audio, kept, kwargs, cfg)
    except RuntimeError as e:
        raise _model_error(e)

    if not kept:
        warnings.append("No transcript text was produced after filtering.")
    return {
        "text": " ".join(s["text"] for s in kept).strip(),
        "segments": kept,
        "dropped": dropped,
        "flags": {"low_conf_words": sum(len(s["low_conf_words"]) for s in kept),
                  "critical_low_conf": sum(len(s["critical_low_conf"]) for s in kept),
                  "dropped_segments": len(dropped),
                  "unstable_segments": sum(1 for s in kept if s.get("unstable"))},
        "info": {"model": f"faster-whisper/{cfg.model_size}", "device": cfg.device,
                 "compute_type": cfg.compute_type, "language": info.language,
                 "language_probability": round(float(info.language_probability), 3),
                 "duration": round(total_s, 1),
                 "duration_after_vad": round(float(getattr(info, "duration_after_vad", total_s)), 1),
                 "firewall": cfg.firewall, "glossary_used": bool(glossary),
                 "prompt": kwargs.get("initial_prompt"), "config": asdict(cfg)},
        "warnings": warnings,
    }


In [ ]:
%%writefile stages/diarize.py
"""Speaker diarization with pyannote/speaker-diarization-3.1 (Stage 1b).

The neural part (pyannote) only answers "who spoke when". Everything after that is deterministic:
  * each Whisper WORD gets the speaker whose turn overlaps it most (nearest turn within max_gap_to_turn_s otherwise),
  * a Whisper segment that contains a speaker change is split at the change (runs shorter than min_words_per_turn
    are merged into the neighbouring run, so one stray word does not create a new line),
  * speaker labels are renumbered by first appearance (SPEAKER_00 speaks first),
  * segment ids are renumbered 1..n (Stages 2 and 3 cite lines by these ids).
If pyannote cannot run (no token, gated model not accepted, out of memory, any error) every segment gets the
`unknown_label` and the reason is recorded; the pipeline continues.
"""
import gc
import time
from collections import defaultdict

import numpy as np

from config import DIAR_CFG, SR, get_hf_token

_PIPELINE = None
_PIPELINE_KEY = None


# ----------------------------------------------------------------------------------------------- model
def _device(cfg):
    import torch
    if cfg.device == "cpu":
        return torch.device("cpu")
    if cfg.device == "cuda" or (cfg.device == "auto" and torch.cuda.is_available()):
        return torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    return torch.device("cpu")


def load_pipeline(cfg=None, hf_token=None):
    """Returns (pipeline or None, reason). Never raises."""
    global _PIPELINE, _PIPELINE_KEY
    cfg = cfg or DIAR_CFG
    key = (cfg.model_id, cfg.device)
    if _PIPELINE is not None and _PIPELINE_KEY == key:
        return _PIPELINE, "cached"
    token = hf_token or get_hf_token()
    if not token:
        return None, ("no Hugging Face token: add HF_TOKEN as a Colab secret (or environment variable) and accept the "
                      f"conditions at https://hf.co/{cfg.model_id} and https://hf.co/pyannote/segmentation-3.0")
    try:
        import compat
        Pipeline = compat.import_pyannote()
        with compat.trusted_torch_load():
            try:
                pipe = Pipeline.from_pretrained(cfg.model_id, use_auth_token=token)
            except TypeError:                                    # a pyannote version that only knows `token`
                pipe = Pipeline.from_pretrained(cfg.model_id, token=token)
    except Exception as e:
        msg = f"{type(e).__name__}: {str(e)[:240]}"
        low = msg.lower()
        if any(k in low for k in ("gated", "401", "403", "access", "restricted", "authorized")):
            msg += (f" | accept the user conditions at https://hf.co/{cfg.model_id} and "
                    "https://hf.co/pyannote/segmentation-3.0 with the account that owns the token")
        return None, "pyannote could not be loaded: " + msg
    if pipe is None:
        return None, (f"pyannote returned no pipeline (gated model?). Accept the conditions at https://hf.co/{cfg.model_id} "
                      "and https://hf.co/pyannote/segmentation-3.0, then re-run.")
    try:
        pipe.to(_device(cfg))
    except Exception as e:
        print("diarization: staying on CPU ->", type(e).__name__, str(e)[:120])
    _PIPELINE, _PIPELINE_KEY = pipe, key
    return pipe, "loaded"


def free_diarizer():
    global _PIPELINE, _PIPELINE_KEY
    _PIPELINE = _PIPELINE_KEY = None
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


# ----------------------------------------------------------------------------------------------- inference
def run_pyannote(pipe, audio: np.ndarray, cfg=None) -> list:
    """[(start_s, end_s, raw_label), ...] sorted by start. Audio is passed in memory (no torchaudio file I/O)."""
    import torch
    cfg = cfg or DIAR_CFG
    wav = torch.from_numpy(np.ascontiguousarray(audio, dtype=np.float32)).unsqueeze(0)
    kw = {}
    if cfg.num_speakers and cfg.num_speakers > 0:
        kw["num_speakers"] = int(cfg.num_speakers)
    else:
        if cfg.min_speakers and cfg.min_speakers > 0:
            kw["min_speakers"] = int(cfg.min_speakers)
        if cfg.max_speakers and cfg.max_speakers > 0:
            kw["max_speakers"] = int(cfg.max_speakers)
    out = pipe({"waveform": wav, "sample_rate": SR}, **kw)
    ann = getattr(out, "speaker_diarization", out)       # pyannote 4 returns an object holding the annotation
    turns = [(float(t.start), float(t.end), str(lbl)) for t, _, lbl in ann.itertracks(yield_label=True)]
    return sorted(turns)


def relabel_by_first_appearance(turns: list) -> list:
    order = {}
    for s, e, lbl in sorted(turns):
        order.setdefault(lbl, f"SPEAKER_{len(order):02d}")
    return [(s, e, order[lbl]) for s, e, lbl in turns]


# ----------------------------------------------------------------------------------------------- assignment
def _speaker_for_span(s: float, e: float, turns: list, max_gap: float):
    ov_by = defaultdict(float)
    for ts, te, lbl in turns:                     # turns are sorted by start time
        if ts >= e:
            break
        ov = min(e, te) - max(s, ts)
        if ov > 0:
            ov_by[lbl] += ov
    if ov_by:
        return max(ov_by.items(), key=lambda x: x[1])[0]
    mid, near, dist = (s + e) / 2, None, float("inf")
    for ts, te, lbl in turns:
        d = 0.0 if ts <= mid <= te else min(abs(mid - ts), abs(mid - te))
        if d < dist:
            near, dist = lbl, d
    return near if dist <= max_gap else None


def _recompute_flags(seg: dict, stt_cfg):
    from stages.textutil import is_critical
    words = seg.get("words", [])
    seg["low_conf_words"] = [w["w"].strip() for w in words if w.get("p") is not None and w["p"] < stt_cfg.low_conf_word_prob]
    seg["critical_low_conf"] = [w["w"].strip() for w in words
                                if w.get("p") is not None and is_critical(w["w"]) and w["p"] < stt_cfg.critical_prob]


def assign_and_split(segments: list, turns: list, cfg=None, stt_cfg=None) -> list:
    """Attach `speaker` to every segment (and every word); split segments at speaker changes. Returns new list."""
    from config import CFG as _STT
    cfg = cfg or DIAR_CFG
    stt_cfg = stt_cfg or _STT
    turns = sorted(turns)
    out = []
    for seg in segments:
        words = seg.get("words") or []
        if not words or not turns:
            spk = _speaker_for_span(seg["start"], seg["end"], turns, cfg.max_gap_to_turn_s) if turns else None
            seg = dict(seg, speaker=spk or cfg.unknown_label)
            out.append(seg)
            continue
        labels = [_speaker_for_span(w["start"], w["end"], turns, cfg.max_gap_to_turn_s) for w in words]
        # unlabeled words inherit the previous (else next) labeled word
        last = None
        for i, l in enumerate(labels):
            if l is None:
                labels[i] = last
            else:
                last = l
        nxt = None
        for i in range(len(labels) - 1, -1, -1):
            if labels[i] is None:
                labels[i] = nxt
            else:
                nxt = labels[i]
        if all(l is None for l in labels):
            labels = [cfg.unknown_label] * len(words)
        for w, l in zip(words, labels):
            w["speaker"] = l
        # runs of identical speakers
        runs = []
        for w, l in zip(words, labels):
            if runs and runs[-1][0] == l:
                runs[-1][1].append(w)
            else:
                runs.append([l, [w]])
        # merge short runs into the longer neighbour
        changed = True
        while changed and len(runs) > 1:
            changed = False
            for i, (l, ws) in enumerate(runs):
                if len(ws) < max(1, cfg.min_words_per_turn):
                    left = runs[i - 1] if i > 0 else None
                    right = runs[i + 1] if i + 1 < len(runs) else None
                    tgt = left if (left and (not right or len(left[1]) >= len(right[1]))) else right
                    if tgt is left:
                        left[1].extend(ws)
                    else:
                        right[1][:0] = ws
                    for w in ws:
                        w["speaker"] = tgt[0]
                    runs.pop(i)
                    changed = True
                    break
            # coalesce neighbours that now share a speaker
            merged = []
            for r in runs:
                if merged and merged[-1][0] == r[0]:
                    merged[-1][1].extend(r[1])
                else:
                    merged.append(r)
            runs = merged
        if len(runs) == 1 or not cfg.split_segments:
            counts = defaultdict(float)
            for w in words:
                counts[w["speaker"]] += max(0.01, w["end"] - w["start"])
            out.append(dict(seg, speaker=max(counts.items(), key=lambda x: x[1])[0]))
            continue
        for l, ws in runs:
            child = {k: v for k, v in seg.items() if k not in ("words", "text", "start", "end", "id")}
            child.update(id=seg["id"], start=round(ws[0]["start"], 2), end=round(ws[-1]["end"], 2),
                         text="".join(w["w"] for w in ws).strip(), words=ws, speaker=l,
                         split_from=seg["id"], merged_from=seg.get("merged_from", [seg["id"]]))
            _recompute_flags(child, stt_cfg)
            out.append(child)
    for n, s in enumerate(out, 1):
        s["id"] = n
    return out


# ----------------------------------------------------------------------------------------------- stage entry
def diarize(audio: np.ndarray, segments: list, cfg=None, hf_token=None, stt_cfg=None) -> tuple:
    """Returns (segments_with_speakers, info). Never raises."""
    cfg = cfg or DIAR_CFG
    t0 = time.time()
    info = {"model": cfg.model_id, "status": "skipped", "reason": "", "n_speakers": 0, "turns": [], "seconds": 0.0}
    if not cfg.enabled:
        info["reason"] = "disabled in settings"
    elif not segments:
        info["reason"] = "no transcript segments"
    else:
        pipe, why = load_pipeline(cfg, hf_token)
        if pipe is None:
            info["reason"] = why
        else:
            try:
                turns = relabel_by_first_appearance(run_pyannote(pipe, audio, cfg))
                if not turns:
                    info["reason"] = "pyannote found no speech turns"
                else:
                    segments = assign_and_split(segments, turns, cfg, stt_cfg)
                    info.update(status="ok", reason="", n_speakers=len({l for _, _, l in turns}),
                                turns=[{"start": round(s, 2), "end": round(e, 2), "speaker": l} for s, e, l in turns])
            except Exception as e:
                info["reason"] = f"diarization inference failed: {type(e).__name__}: {str(e)[:200]}"
            finally:
                free_diarizer()
    if info["status"] != "ok":
        for s in segments:
            s["speaker"] = cfg.unknown_label
        print("diarization skipped ->", info["reason"])
    info["seconds"] = round(time.time() - t0, 1)
    return segments, info


def self_test() -> int:
    """Speaker assignment / splitting logic with synthetic turns (no model). Returns number of failures."""
    def W(w, a, b, p=0.9):
        return {"w": w, "start": a, "end": b, "p": p}
    segs = [{"id": 1, "start": 0.0, "end": 4.0, "text": "Hello all. Thanks Priya, I will do it.",
             "words": [W(" Hello", 0.0, 0.4), W(" all.", 0.4, 0.8), W(" Thanks", 1.2, 1.6), W(" Priya,", 1.6, 2.0),
                       W(" I", 2.1, 2.2), W(" will", 2.2, 2.5), W(" do", 2.5, 2.8), W(" it.", 2.8, 3.1)]},
            {"id": 2, "start": 4.0, "end": 6.0, "text": "Okay great.", "words": [W(" Okay", 4.0, 4.5), W(" great.", 4.5, 5.0)]},
            {"id": 3, "start": 6.0, "end": 8.0, "text": "Sure um yes",
             "words": [W(" Sure", 6.0, 6.4), W(" um", 6.45, 6.5, 0.3), W(" yes", 6.6, 7.0)]}]
    turns = relabel_by_first_appearance([(0.0, 0.9, "B"), (1.1, 3.3, "A"), (3.9, 5.2, "B"), (5.9, 6.47, "A"), (6.48, 6.55, "B"),
                                          (6.56, 7.1, "A")])
    out = assign_and_split([dict(s, words=[dict(w) for w in s["words"]]) for s in segs], turns)
    checks = [
        ("first speaker is SPEAKER_00", turns[0][2] == "SPEAKER_00"),
        ("segment split at the speaker change", [s["speaker"] for s in out[:2]] == ["SPEAKER_00", "SPEAKER_01"]
         and out[0]["text"] == "Hello all." and out[1]["text"].startswith("Thanks Priya")),
        ("ids renumbered 1..n", [s["id"] for s in out] == list(range(1, len(out) + 1))),
        ("one-word blip is not a new line", out[-1]["text"] == "Sure um yes" and out[-1]["speaker"] == "SPEAKER_01"),
        ("split children remember their parent", out[0].get("split_from") == 1 and out[1].get("split_from") == 1),
        ("no turns -> unknown label", assign_and_split([dict(segs[1])], [], None)[0]["speaker"] == DIAR_CFG.unknown_label),
    ]
    bad = 0
    for name, ok in checks:
        print(("PASS " if ok else "FAIL ") + name); bad += not ok
    print("\nALL GOOD" if not bad else f"\n{bad} diarization test(s) failed")
    return bad


In [ ]:
%%writefile stages/asr2.py
"""Optional L4 cross-check: a second, architecturally different ASR (NVIDIA Parakeet TDT via onnx-asr).
It never replaces the Whisper transcript. It only marks WHERE the two systems disagree, so Phase 2
can focus on (and be cautious with) those spans.
Install (CPU, no NeMo / PyTorch needed):  pip install "onnx-asr[cpu,hub]"
"""
import difflib
import time
from collections import Counter

import numpy as np

from config import CFG, SR
from stages.textutil import norm_text, is_critical

_MODELS = {}


def load(cfg=CFG):
    """Load the Parakeet ONNX model once per (model, quantization, providers). Raises if onnx-asr is missing."""
    import inspect
    import onnx_asr
    providers = [p.strip() for p in str(cfg.asr2_providers or "").split(",") if p.strip()] or None
    key = (cfg.asr2_model, cfg.asr2_quantization, tuple(providers or ()))
    if key not in _MODELS:
        params = set(inspect.signature(onnx_asr.load_model).parameters)
        kw = {}
        if providers and "providers" in params:
            kw["providers"] = providers
        try:
            if cfg.asr2_quantization and "quantization" in params:
                _MODELS[key] = onnx_asr.load_model(cfg.asr2_model, quantization=cfg.asr2_quantization, **kw)
            else:
                _MODELS[key] = onnx_asr.load_model(cfg.asr2_model, **kw)
        except Exception:
            _MODELS[key] = onnx_asr.load_model(cfg.asr2_model, **kw)     # retry without quantisation
    return _MODELS[key]


def free():
    _MODELS.clear()


def chunk_plan(speech_segments, total_s, max_chunk_s=25.0, max_gap_s=1.0, pad=0.15):
    segs = list(speech_segments) if speech_segments else [(0.0, total_s)]
    chunks, cur = [], None
    for s, e in segs:
        pieces, t = [], s
        while e - t > max_chunk_s:
            pieces.append((t, t + max_chunk_s))
            t += max_chunk_s
        pieces.append((t, e))
        for ps, pe in pieces:
            if cur and ps - cur[1] <= max_gap_s and pe - cur[0] <= max_chunk_s:
                cur = (cur[0], pe)
            else:
                if cur:
                    chunks.append(cur)
                cur = (ps, pe)
    if cur:
        chunks.append(cur)
    return [(max(0.0, s - pad), min(total_s, e + pad)) for s, e in chunks if e - s >= 0.3]


def _as_text(res) -> str:
    if isinstance(res, (list, tuple)):
        res = res[0] if res else ""
    return (res if isinstance(res, str) else getattr(res, "text", str(res))).strip()


def transcribe_chunks(audio, speech_segments, cfg=CFG, batch: int = 8):
    model = load(cfg)
    total_s = len(audio) / SR
    plan = chunk_plan(speech_segments, total_s, cfg.asr2_max_chunk_s)
    out = []
    for b in range(0, len(plan), batch):
        part = plan[b:b + batch]
        clips = [np.ascontiguousarray(audio[int(s * SR): int(e * SR)], dtype=np.float32) for s, e in part]
        try:
            res = model.recognize(clips, sample_rate=SR)
            texts = [_as_text(r) for r in res] if isinstance(res, (list, tuple)) and len(res) == len(clips) else None
        except Exception:
            texts = None
        if texts is None:                                   # batch failed: one clip at a time
            texts = []
            for clip in clips:
                try:
                    texts.append(_as_text(model.recognize(clip, sample_rate=SR)))
                except Exception as ex:
                    texts.append(None)
                    out.append({"error": str(ex)[:120]})
        for (s, e), t in zip(part, texts):
            if t is None:
                continue
            out.append({"start": round(s, 2), "end": round(e, 2), "text": t})
    errors = [o for o in out if "error" in o]
    return [o for o in out if "error" not in o], errors


def _whisper_tokens(raw):
    toks = []
    for seg in raw["segments"]:
        added = False
        for w in seg.get("words", []):
            for t in norm_text(w["w"]).split():
                toks.append((t, seg["id"], w["start"], w["end"]))
                added = True
        if not added:
            for t in norm_text(seg["text"]).split():
                toks.append((t, seg["id"], seg["start"], seg["end"]))
    return toks


def compare(raw, chunks, model_name=None):
    wt = _whisper_tokens(raw)
    pt = []
    for c in chunks:
        for t in norm_text(c["text"]).split():
            pt.append((t, c["start"], c["end"]))
    a = [x[0] for x in wt]
    b = [x[0] for x in pt]
    sm = difflib.SequenceMatcher(None, a, b, autojunk=False)
    flagged, dis = set(), []
    seg_bad = Counter()
    seg_n = Counter(x[1] for x in wt)
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            continue
        if tag == "insert":
            idx = [k for k in (i1 - 1, i1) if 0 <= k < len(a)]
        else:
            idx = list(range(i1, i2))
        flagged.update(idx)
        for k in range(i1, i2):
            seg_bad[wt[k][1]] += 1
        if wt and i2 > i1:
            start, end = wt[i1][2], wt[i2 - 1][3]
            seg_ids = sorted({wt[k][1] for k in range(i1, i2)})
        elif wt:
            ref_tok = wt[min(max(i1 - 1, 0), len(wt) - 1)]
            start, end, seg_ids = ref_tok[2], ref_tok[3], [ref_tok[1]]
        else:
            start, end, seg_ids = (pt[j1][1], pt[j1][2], []) if pt else (0.0, 0.0, [])
        wtxt, ptxt = " ".join(a[i1:i2]), " ".join(b[j1:j2])
        dis.append({"id": len(dis) + 1, "type": tag, "whisper": wtxt, "parakeet": ptxt,
                    "start": round(start, 2), "end": round(end, 2), "seg_ids": seg_ids,
                    "critical": any(is_critical(t) for t in (a[i1:i2] + b[j1:j2])),
                    "context": " ".join(a[max(0, i1 - 6): i2 + 6])})
    for seg in raw["segments"]:
        n, bad = seg_n.get(seg["id"], 0), seg_bad.get(seg["id"], 0)
        seg["asr2_diff_frac"] = round(bad / n, 2) if n else 0.0
        seg["asr2_unsupported"] = bool(n >= 3 and bad / n >= 0.8)
    return {"model": model_name or CFG.asr2_model, "text": " ".join(c["text"] for c in chunks).strip(),
            "chunks": chunks, "disagreements": dis,
            "disagreement_rate": round(len(flagged) / max(len(a), 1), 3),
            "critical_disagreements": sum(1 for d in dis if d["critical"]),
            "unsupported_segments": sum(1 for s in raw["segments"] if s.get("asr2_unsupported")),
            "flagged_token_idx": sorted(flagged), "whisper_tokens": a}


def cross_check(raw, audio, speech_segments, cfg=CFG):
    """Run Parakeet on the same speech regions and align it to the Whisper words. Raises on model problems
    (the caller turns that into a warning). Never modifies the Whisper text."""
    t0 = time.time()
    chunks, errors = transcribe_chunks(audio, speech_segments, cfg)
    if errors and not chunks:
        raise RuntimeError(f"Parakeet failed on every chunk ({errors[0].get('error')})")
    res = compare(raw, chunks, cfg.asr2_model)
    res["seconds"] = round(time.time() - t0, 1)
    res["chunk_errors"] = len(errors)
    return res


In [ ]:
%%writefile pipeline_stt.py
"""Stage 1: audio -> validated 16 kHz mono -> faster-whisper (hallucination firewall) -> pyannote 3.1 speakers
-> optional Parakeet (ONNX) cross-check -> numbered lines + stage1_record.json.
Model problems in diarization or the cross-check degrade to warnings; only unusable INPUT raises PipelineError."""
import json
import os

from config import CFG, DIAR_CFG, SCHEMA_VERSION
from errors import PipelineError
from stages import ingest, stt
from stages import diarize


def _fmt(sec: float) -> str:
    sec = int(sec or 0)
    h, rem = divmod(sec, 3600)
    m, s = divmod(rem, 60)
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"


def to_numbered_lines(raw: dict) -> list:
    """['[L001 00:03] [SPEAKER_00]: text', ...] — the line ids every later citation refers to."""
    lines = []
    for s in raw["segments"]:
        spk = s.get("speaker") or DIAR_CFG.unknown_label
        lines.append(f"[L{s['id']:03d} {_fmt(s['start'])}] [{spk}]: {s['text']}")
    return lines


def process(audio_path: str, glossary=None, cfg=None, on_progress=None, hf_token=None, diar_cfg=None) -> dict:
    cfg = cfg or CFG
    diar_cfg = diar_cfg or DIAR_CFG
    report = on_progress or (lambda stage, p: None)

    report("validating", 0.02)
    meta = ingest.validate_and_convert(audio_path, cfg)          # raises PipelineError for unusable input

    report("transcribing", 0.05)
    try:
        raw = stt.transcribe(meta["audio"], glossary=glossary, cfg=cfg,
                             on_progress=lambda f: report("transcribing", 0.05 + 0.65 * f),
                             speech_segments=meta["speech_segments"])
    finally:
        stt.free_gpu()                                           # Whisper out of VRAM before pyannote / LLMs

    report("diarizing", 0.72)
    raw["segments"], diar_info = diarize.diarize(meta["audio"], raw["segments"], diar_cfg, hf_token=hf_token, stt_cfg=cfg)
    raw["diarization"] = diar_info
    if diar_info["status"] != "ok":
        raw["warnings"].append("Speaker diarization skipped: " + diar_info["reason"])

    raw["text"] = " ".join(s["text"] for s in raw["segments"]).strip()
    raw["flags"]["low_conf_words"] = sum(len(s.get("low_conf_words", [])) for s in raw["segments"])
    raw["flags"]["critical_low_conf"] = sum(len(s.get("critical_low_conf", [])) for s in raw["segments"])
    raw["info"].update({"input_duration_s": meta["duration_s"], "speech_s": meta["speech_s"],
                        "vad_method": meta["vad_method"], "creation_time": meta.get("creation_time")})

    if cfg.use_asr2:
        report("cross-checking", 0.85)
        try:
            from stages import asr2
            raw["asr2"] = asr2.cross_check(raw, meta["audio"], meta["speech_segments"], cfg)
        except Exception as e:
            raw["warnings"].append(f"Second-ASR (Parakeet) cross-check skipped: {type(e).__name__}: {str(e)[:200]}")
            raw["asr2"] = None

    raw["lines"] = to_numbered_lines(raw)
    try:
        os.remove(meta["wav_path"])
    except Exception:
        pass
    report("done", 1.0)
    return raw


def build_stage1_record(raw: dict, audio_path: str = None, glossary=None) -> dict:
    info = dict(raw.get("info", {}))
    cfg = info.pop("config", None)
    segs = []
    for s in raw["segments"]:
        segs.append({
            "id": s["id"], "start": s["start"], "end": s["end"],
            "speaker": s.get("speaker") or DIAR_CFG.unknown_label, "text": s["text"],
            "words": [{"w": w["w"].strip(), "start": w.get("start"), "end": w.get("end"), "p": w.get("p"),
                       **({"speaker": w["speaker"]} if w.get("speaker") else {})}
                      for w in s.get("words", [])],
            "low_conf_words": s.get("low_conf_words", []),
            "critical_low_conf": s.get("critical_low_conf", []),
            "merged_from": s.get("merged_from", [s["id"]]),
            **({"split_from": s["split_from"]} if "split_from" in s else {}),
            **({"asr2_diff_frac": s["asr2_diff_frac"]} if "asr2_diff_frac" in s else {}),
            **({"asr2_unsupported": s["asr2_unsupported"]} if "asr2_unsupported" in s else {}),
            **({"stability": s["stability"]} if "stability" in s else {}),
        })
    low = [{"segment_id": s["id"], "word": w["w"].strip(), "p": w["p"]}
           for s in raw["segments"] for w in s.get("words", []) if w["w"].strip() in s.get("low_conf_words", [])]
    crit = [{"segment_id": s["id"], "word": w} for s in raw["segments"] for w in s.get("critical_low_conf", [])]
    asr2 = raw.get("asr2")
    diar = raw.get("diarization") or {}
    return {
        "schema_version": SCHEMA_VERSION,
        "meta": {"audio_file": os.path.basename(audio_path) if audio_path else None,
                 "duration_s": info.get("input_duration_s", info.get("duration")),
                 "speech_s": info.get("speech_s"),
                 "recording_created": info.get("creation_time"),
                 "models": {"stt": info.get("model"), "diarization": diar.get("model") if diar.get("status") == "ok" else None,
                            "asr2": (asr2 or {}).get("model"), "refiner": None, "documenter": None},
                 "stt_info": info, "stt_config": cfg, "warnings": raw.get("warnings", []),
                 "flags": raw.get("flags", {})},
        "diarization": {"status": diar.get("status", "skipped"), "reason": diar.get("reason", ""),
                        "model": diar.get("model"), "n_speakers": diar.get("n_speakers", 0),
                        "turns": diar.get("turns", [])},
        "speakers": sorted({s.get("speaker") or DIAR_CFG.unknown_label for s in raw["segments"]}),
        "raw_transcript": {"text": raw["text"], "segments": segs, "lines": raw.get("lines") or to_numbered_lines(raw)},
        "low_confidence_words": low,
        "critical_low_confidence_words": crit,
        "dropped_segments": [{"id": d["id"], "start": d["start"], "end": d["end"], "text": d["text"],
                              "drop_reason": d.get("drop_reason")} for d in raw.get("dropped", [])],
        "asr2": ({"model": asr2.get("model"), "disagreement_rate": asr2.get("disagreement_rate"),
                  "critical_disagreements": asr2.get("critical_disagreements"),
                  "unsupported_segments": asr2.get("unsupported_segments"),
                  "seconds": asr2.get("seconds"), "text": asr2.get("text"),
                  "disagreements": asr2.get("disagreements", [])} if asr2 else None),
        "glossary": list(glossary or []),
    }


def check_stage1_record(rec: dict) -> list:
    problems = []
    for key in ("schema_version", "meta", "raw_transcript", "low_confidence_words", "dropped_segments"):
        if key not in rec:
            problems.append(f"missing top-level key {key!r}")
    rt = rec.get("raw_transcript", {})
    if not isinstance(rt.get("segments"), list) or not rt.get("segments"):
        problems.append("raw_transcript.segments is empty")
    else:
        ids = [s["id"] for s in rt["segments"]]
        if ids != list(range(1, len(ids) + 1)):
            problems.append("segment ids are not 1..n")
    return problems


def save_outputs(raw: dict, out_dir: str = "outputs", audio_path: str = None, glossary=None) -> dict:
    os.makedirs(out_dir, exist_ok=True)
    jp, tp = os.path.join(out_dir, "raw_transcript.json"), os.path.join(out_dir, "raw_transcript.txt")
    with open(jp, "w", encoding="utf-8") as f:
        json.dump({k: v for k, v in raw.items() if k != "audio"}, f, indent=2, ensure_ascii=False, default=str)
    with open(tp, "w", encoding="utf-8") as f:
        f.write("\n".join(raw["lines"]))
    rec = build_stage1_record(raw, audio_path, glossary)
    problems = check_stage1_record(rec)
    if problems:
        raw.setdefault("warnings", []).append("stage1_record schema problems: " + "; ".join(problems))
        rec["meta"]["warnings"] = raw["warnings"]
    rp = os.path.join(out_dir, "stage1_record.json")
    with open(rp, "w", encoding="utf-8") as f:
        json.dump(rec, f, indent=2, ensure_ascii=False, default=str)
    return {"json": jp, "txt": tp, "record": rp, "schema_problems": problems}


In [ ]:
%%writefile stages/refine.py
"""Stage 2 - domain-aware transcript refinement (LLM #1). Built from the Stage 2 notebook cells 5-16.
The LLM proposes small edits; deterministic validators decide what is applied. See the notebook for the walkthrough."""
import re, json, copy, time, gc, difflib
try:                                  # heavy imports are optional: without them the stage degrades instead of crashing
    import torch
except Exception as _e:
    torch, _TORCH_ERR = None, _e
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
except Exception as _e:
    AutoTokenizer = AutoModelForCausalLM = BitsAndBytesConfig = None
    _TF_ERR = _e
from pathlib import Path
from collections import Counter
from config import RefineConfig, DEFAULT_CASING, DIAR_CFG, models_dir
from errors import PipelineError, RefineError
import compat

CFG = RefineConfig()                 # module default; every entry point takes an explicit cfg
MODELS_DIR = Path(models_dir())      # 4-bit model cache (MA_MODELS_DIR; the notebook points it at Drive when mounted)
NAMES: set = set()                   # person-name tokens of the current transcript (set by refine())

from stages.textutil import tokens, is_critical, protected_tokens, PROTECTED_WORDS, NUMBER_WORDS, NEGATIONS, COMMIT, norm_text

def protected_in(text: str) -> list[str]:
    "Lower-cased protected tokens (digits, dates, times, number words, negations, modals) in order of appearance."
    return protected_tokens(text)

# ---- ordinary-English check (an ASR jargon error is a word that should NOT be ordinary English) ----
try:
    from wordfreq import zipf_frequency as _zipf
except Exception:                                    # tiny fallback list; install wordfreq for the real thing
    _COMMON = set("""the a an and or but if then of to in on at for with by from as is are was were be been
    being have has had do does did not no yes we you they he she it this that these those will would can could
    should may might must shall increase decrease enable disable deploy delay demo memo product project server
    service upload download start stop add remove allow deny accept reject approve before after more less higher
    lower open close include exclude cache cash time out setup set up report meeting team""".split())
    def _zipf(w, lang="en"):
        return 5.0 if w.lower() in _COMMON else 0.0

def is_common_word(word: str, cfg=None) -> bool:
    thr = (cfg or CFG).common_word_zipf
    w = re.sub(r"[^a-z']", "", str(word).lower())
    return bool(w) and _zipf(w, "en") >= thr

def looks_technical(term: str) -> bool:
    "CamelCase, acronym, digit, slash or hyphen inside: PyTorch, CI/CD, K8s, AdamW, gpt-4."
    t = term.strip()
    return bool(re.search(r"[a-z][A-Z]|[A-Z]{2,}|\d|[/_-]", t)) or (t.isupper() and len(t) >= 2)

# ---- person names: spaCy PERSON entities (fallback: capitalised, not ordinary English) ----
SENTENCE_STARTERS = set("the a an and but so or if then first second third next okay ok yes no we you i it this that "
                        "these those there here someone everyone nobody anyone also finally lastly today now please "
                        "thanks thank right well".split()) | {"let's", "that's", "it's", "we're", "they're"}
_NLP = None
_NLP_KEY = None
def _load_nlp(cfg=None):
    global _NLP, _NLP_KEY
    cfg = cfg or CFG
    key = (cfg.use_ner, cfg.spacy_model)
    if _NLP is None or _NLP_KEY != key:
        _NLP, _NLP_KEY = False, key
        if cfg.use_ner:
            try:
                import spacy
                _NLP = spacy.load(cfg.spacy_model, disable=["parser", "lemmatizer"])
            except Exception as e:
                print("spaCy unavailable ->", type(e).__name__, "| using heuristic name detection")
    return _NLP or None

def person_names(segments: list, glossary: list[str], cfg=None) -> set[str]:
    """Tokens treated as people's names (never respelled unless the glossary vouches for a spelling, or the
    acoustic evidence says Whisper itself was unsure of the span). NER replaces the old 'any capitalised word'
    rule, which blocked corrections like 'Cooper Netties' -> 'Kubernetes'."""
    gl = {g.lower() for g in glossary}; names = set()
    nlp = _load_nlp(cfg)
    if nlp is not None:
        for s in segments:
            for ent in nlp(s["text"]).ents:
                if ent.label_ == "PERSON":
                    for t in tokens(ent.text):
                        if t[:1].isupper() and len(t) > 1 and t.lower() not in gl and not is_critical(t):
                            names.add(t)
    else:
        for s in segments:
            for i, t in enumerate(tokens(s["text"])):
                if (i > 0 and t[0].isupper() and not t.isupper() and len(t) > 1 and t.lower() not in gl
                        and t.lower() not in SENTENCE_STARTERS and not is_critical(t) and not is_common_word(t)):
                    names.add(t)
    return names

def capitalised_names(text: str, glossary: list[str], names: set | None = None) -> list[str]:
    "Name tokens present in this text (kept for the damage check and the per-segment index)."
    pool = names if names is not None else NAMES
    gl = {g.lower() for g in glossary}
    return [t for t in tokens(text) if t in pool and t.lower() not in gl]

def build_protected_index(segments: list, glossary: list[str], names: set | None = None) -> dict:
    return {s["id"]: {"protected": protected_in(s["text"]), "names": capitalised_names(s["text"], glossary, names)}
            for s in segments}

def make_chunks(segments: list, size: int, overlap: int) -> list[list[dict]]:
    chunks, i = [], 0
    while i < len(segments):
        chunks.append(segments[i:i + size])
        if i + size >= len(segments):
            break
        i += size - overlap
    return chunks

def _min_word_prob(sid: int, phrase: str, word_probs: dict) -> float | None:
    "Minimum Whisper word probability over the words of `phrase` inside segment `sid` (None if unknown)."
    probs = word_probs.get(sid) or []
    if not probs:
        return None
    want = [w.lower() for w in tokens(phrase)]
    have = [(re.sub(r"[^a-z0-9']", "", w.lower()), p) for w, p in probs]
    ps = [p for w, p in have if w and w in want and p is not None]
    return min(ps) if ps else None

def auto_glossary(segments: list, word_probs: dict, min_count: int = 2, cfg=None) -> list[str]:
    """Capitalised tokens / acronyms repeated >= min_count times with consistent spelling. Only tokens Whisper was
    SURE about (word prob >= high_conf_p) and that are not ordinary English qualify: a consistently misheard
    'Cooper Netties' must not become a 'confirmed spelling'. Sent to the model as OBSERVED spellings."""
    cfg = cfg or CFG
    c, low = Counter(), set()
    for s in segments:
        for i, t in enumerate(tokens(s["text"])):
            if (t.isupper() and 2 <= len(t) <= 8) or (i > 0 and t[0].isupper() and len(t) > 2):
                if is_critical(t) or t.lower() in SENTENCE_STARTERS or is_common_word(t) or t in NAMES:
                    continue
                c[t] += 1
                p = _min_word_prob(s["id"], t, word_probs)
                if p is not None and p < cfg.high_conf_p:
                    low.add(t)
    return sorted(t for t, n in c.items() if n >= min_count and t not in low)

SYSTEM_PROMPT = '''You are an ASR error corrector, not an editor. You receive part of a meeting transcript produced by speech-to-text, one segment per line as [ID] text.
Your ONLY job: find words or phrases that are clearly misrecognised domain-specific terms, acronyms, product/tool names, or technical vocabulary, and propose the intended term.

Rules:
- Propose an edit only if the replacement sounds similar to the original AND fits the meeting context.
- Use the canonical spelling and casing of the term (PyTorch, PostgreSQL, CI/CD, Kubernetes, AdamW).
- A misrecognised term is often Capitalised by the recogniser ("Cooper Netties", "Pie Torch"). That is not a person's name; correct it.
- NEVER change numbers, dates, times, quantities, names of people, negations (not, n't, never, no, cannot, without), or words of commitment (will, won't, should, must, might, can, agreed, decided).
- NEVER replace an ordinary English word with another ordinary English word (increase/decrease, enable/disable, demo/memo). Only domain terms are corrected.
- NEVER fix grammar, remove filler words (um, uh, like, I mean), paraphrase, summarise, reorder, or merge segments.
- Do not edit a person's name unless the glossary lists the correct spelling.
- Where a second recogniser heard the span differently, it is shown as [alt: ...]. Prefer the reading that is a real term; if neither is, leave it.
- If unsure, do not edit. Returning an empty list is correct when nothing is clearly wrong.
- "original" must be copied exactly (character for character) from the given segment, and must be at most 4 words.
- Return ONLY JSON, no commentary: {"edits":[{"segment_id":int,"original":str,"replacement":str,"reason":str,"confidence":float}]}

Examples:
[12] the post grass database crashed twice      -> {"segment_id":12,"original":"post grass","replacement":"PostgreSQL","reason":"phonetic match, database name","confidence":0.9}
[17] deploy it on the Cooper Netties cluster    -> {"segment_id":17,"original":"Cooper Netties","replacement":"Kubernetes","reason":"capitalised misrecognition of the orchestrator name","confidence":0.9}
[20] we need to push it to rod by friday        -> {"segment_id":20,"original":"rod","replacement":"prod","reason":"deployment target","confidence":0.85}
[31] the see eye see dee pipeline is red        -> {"segment_id":31,"original":"see eye see dee","replacement":"CI/CD","reason":"spelled-out acronym","confidence":0.9}
[44] we will not migrate to aws this quarter    -> no edit (negation and commitment; "aws" is already right)
[50] fifteen minutes, down from fifty           -> no edit (numbers are never changed)
[52] we should increase the retry count         -> no edit (ordinary English; nothing is misrecognised)'''

def build_user_message(chunk: list[dict], glossary: list[str], suspect: dict, context: str, disagree: dict | None = None) -> str:
    lines = [f"Meeting context: {context or 'business meeting (domain not yet known)'}"]
    if glossary:
        lines.append("Terms known to appear in this meeting (use these spellings): " + ", ".join(glossary[:60]))
    sus = [f"[{s['id']}] {', '.join(suspect[s['id']])}" for s in chunk if s["id"] in suspect]
    if sus:
        lines.append("Low-confidence words from the speech recogniser (check these first): " + "; ".join(sus))
    alts = [f"[{s['id']}] '{d['whisper']}' [alt: '{d['alt']}']" for s in chunk for d in (disagree or {}).get(s["id"], [])]
    if alts:
        lines.append("Spans where a second recogniser heard something else: " + "; ".join(alts[:20]))
    lines.append("\nTranscript segments:")
    lines += [f"[{s['id']}] {s['text']}" for s in chunk]
    lines.append('\nReturn ONLY the JSON object {"edits":[...]}.')
    return "\n".join(lines)

from rapidfuzz import fuzz
import jellyfish

# spelled-out letters -> letter, so "see eye see dee" ~ "CI/CD", "A P I" ~ "API", "ay double you ess" ~ "AWS"
LETTER_WORDS = {"a":"a","ay":"a","bee":"b","be":"b","see":"c","sea":"c","cee":"c","dee":"d","e":"e","ee":"e","ef":"f",
    "gee":"g","aitch":"h","eye":"i","i":"i","jay":"j","kay":"k","el":"l","em":"m","en":"n","oh":"o","o":"o","pee":"p",
    "queue":"q","cue":"q","are":"r","es":"s","ess":"s","tee":"t","tea":"t","you":"u","u":"u","vee":"v","ex":"x",
    "why":"y","zee":"z","zed":"z"}

# ordinary-word pairs that flip meaning while sounding alike; never an ASR jargon fix
POLARITY_PAIRS = {frozenset(p) for p in [("increase","decrease"),("enable","disable"),("enabled","disabled"),
    ("add","remove"),("allow","deny"),("allow","block"),("start","stop"),("upload","download"),("before","after"),
    ("more","less"),("higher","lower"),("on","off"),("include","exclude"),("accept","reject"),("approve","reject"),
    ("open","close"),("up","down"),("in","out"),("pass","fail"),("passed","failed"),("true","false"),("yes","no"),
    ("can","can't"),("will","won't"),("do","don't"),("is","isn't"),("agree","disagree"),("buy","sell"),
    ("push","pull"),("encrypt","decrypt"),("import","export"),("input","output"),("lock","unlock"),("connect","disconnect")]}

def _alnum(s: str) -> str:
    return re.sub(r"[^a-z0-9]", "", s.lower())

def _spelled(s: str) -> str | None:
    ws = [w.lower() for w in tokens(s)]
    out, i = [], 0
    while i < len(ws):
        if ws[i] == "double" and i + 1 < len(ws) and ws[i + 1] in ("you", "u"):
            out.append("w"); i += 2; continue
        if ws[i] in LETTER_WORDS or len(ws[i]) == 1:
            out.append(LETTER_WORDS.get(ws[i], ws[i])); i += 1; continue
        return None
    return "".join(out) if out else None

# ---- phoneme-level similarity (g2p_en, optional) ----
_G2P = None
def _phonemes(s: str) -> list[str] | None:
    global _G2P
    if _G2P is None:
        try:
            from g2p_en import G2p
            _G2P = G2p()
        except Exception:
            _G2P = False
    if not _G2P:
        return None
    try:
        return [re.sub(r"\d", "", p) for p in _G2P(s) if p.strip() and p != " "]
    except Exception:
        return None

def phoneme_distance(a: str, b: str) -> float | None:
    "Normalised phoneme edit distance in [0,1]; None if g2p is unavailable."
    pa, pb = _phonemes(a), _phonemes(b)
    if not pa or not pb:
        return None
    return jellyfish.levenshtein_distance(" ".join(pa), " ".join(pb)) / max(len(" ".join(pa)), len(" ".join(pb)), 1)

def find_spans(original: str, text: str, ignore_case: bool = False) -> list[tuple[int, int]]:
    "Whole-word occurrences of `original` in `text` ('rod' never matches inside 'production')."
    pat = r"(?<![A-Za-z0-9])" + re.escape(original.strip()) + r"(?![A-Za-z0-9])"
    return [(m.start(), m.end()) for m in re.finditer(pat, text, flags=re.I if ignore_case else 0)]

def replace_spans(text: str, original: str, replacement: str) -> str:
    pat = r"(?<![A-Za-z0-9])" + re.escape(original.strip()) + r"(?![A-Za-z0-9])"
    return re.sub(pat, lambda m: replacement, text)

def evidence_tier(sid: int, original: str, word_probs: dict | None, cfg: RefineConfig) -> tuple[str, float | None]:
    "low / medium / high acoustic confidence of the original span (medium when probabilities are unknown)."
    p = _min_word_prob(sid, original, word_probs or {})
    if p is None:
        return "medium", None
    return ("low" if p < cfg.low_conf_p else "high" if p >= cfg.high_conf_p else "medium"), p

LM_SCORER = None      # set in Cell 10 to a function text -> total log-probability under the refiner LLM

def lm_prefers(text: str, original: str, replacement: str, cfg: RefineConfig) -> tuple[bool | None, float | None]:
    "Does the LLM find the edited sentence likelier by cfg.lm_margin nats? None when no scorer is available."
    if LM_SCORER is None or not cfg.use_lm_rescoring:
        return None, None
    try:
        before = LM_SCORER(text); after = LM_SCORER(replace_spans(text, original, replacement))
    except Exception:
        return None, None
    return (after - before) >= cfg.lm_margin, round(after - before, 2)

def phonetically_plausible(original: str, replacement: str, glossary: list[str], cfg: RefineConfig,
                           tier: str = "medium") -> tuple[bool, str]:
    o, r = _alnum(original), _alnum(replacement)
    if replacement.strip().lower() in {g.lower() for g in glossary}:
        return True, "glossary"
    if _spelled(original) == r and r:
        return True, "spelled-letters"
    if not o or not r:
        return False, "empty"
    ow, rw = tokens(original), tokens(replacement)
    # per-word comparison when the word counts match (catches 'increase'->'decrease' style swaps word by word)
    if len(ow) == len(rw) and len(ow) > 1:
        ratios = [fuzz.ratio(_alnum(a), _alnum(b)) for a, b in zip(ow, rw) if _alnum(a) != _alnum(b)]
        ratio = min(ratios) if ratios else 100
    else:
        ratio = fuzz.ratio(o, r)
    meta_eq = jellyfish.metaphone(o) == jellyfish.metaphone(r)
    pd = phoneme_distance(original, replacement)
    thr = {"low": cfg.fuzzy_min_ratio - 10, "medium": cfg.fuzzy_min_ratio, "high": cfg.fuzzy_min_ratio + 10}[tier]
    pthr = {"low": cfg.phoneme_max_dist + 0.1, "medium": cfg.phoneme_max_dist, "high": cfg.phoneme_max_dist - 0.1}[tier]
    if pd is not None and pd <= pthr:
        return True, f"phonemes={pd:.2f}"
    if meta_eq and tier != "high":
        return True, "metaphone"
    if ratio >= thr:
        return True, f"fuzzy={ratio:.0f}"
    return False, f"not similar (fuzzy={ratio:.0f}, metaphone {jellyfish.metaphone(o)}!={jellyfish.metaphone(r)}" + \
                  (f", phonemes={pd:.2f}" if pd is not None else "") + f", tier={tier})"

def validate_edit(e: dict, seg_by_id: dict, glossary: list[str], cfg: RefineConfig, names: set | None = None,
                  word_probs: dict | None = None) -> tuple[bool, str]:
    "Return (ok, reason). reason is 'ok (...)' or why it was rejected. Every check is deterministic."
    try:
        sid = int(e["segment_id"]); orig = str(e["original"]).strip(); rep = str(e["replacement"]).strip()
        conf = float(e.get("confidence", 1.0))
    except (KeyError, TypeError, ValueError):
        return False, "malformed"
    if sid not in seg_by_id:
        return False, "unknown segment"
    text = seg_by_id[sid]["text"]
    if not orig or not rep:
        return False, "empty"
    if orig == rep:
        return False, "no-op"
    spans = find_spans(orig, text)
    if not spans:
        return False, "original not found as whole words"
    if conf < cfg.min_confidence:
        return False, f"confidence {conf:.2f} < {cfg.min_confidence}"
    ow, rw = tokens(orig), tokens(rep)
    if len(ow) > cfg.max_original_words:
        return False, f"too long ({len(ow)} words)"
    if protected_in(orig) != protected_in(rep):
        return False, f"touches protected tokens {protected_in(orig)} -> {protected_in(rep)}"
    new_text = replace_spans(text, orig, rep)
    if protected_in(text) != protected_in(new_text):
        return False, "would alter protected tokens in segment"
    gl = {g.lower() for g in glossary}
    in_glossary = rep.lower() in gl
    changed_o = [a for a in ow if a.lower() not in {b.lower() for b in rw}]
    changed_r = [b for b in rw if b.lower() not in {a.lower() for a in ow}]
    for a in changed_o:
        for b in changed_r:
            if frozenset((a.lower(), b.lower())) in POLARITY_PAIRS:
                return False, f"polarity flip {a!r} -> {b!r}"
    tier, p = evidence_tier(sid, orig, word_probs, cfg)
    # names: NER says person -> untouchable, unless the glossary vouches for the replacement or Whisper itself
    # was unsure of the span (a capitalised misrecognition that NER mistook for a name)
    name_pool = names if names is not None else set()
    touched = [t for t in ow if t in name_pool and t.lower() not in gl and t not in rw]
    if touched and not in_glossary and tier != "low":
        return False, f"touches name(s) {touched}"
    # ordinary-word guard: replacing ordinary English with ordinary English is never a jargon fix,
    # unless the glossary vouches for it or the LLM finds the edit clearly likelier
    spelled = bool(_alnum(rep)) and _spelled(orig) == _alnum(rep)      # 'A P I' -> 'API' is deterministic evidence
    if (changed_o and changed_r and not in_glossary and not spelled
            and all(is_common_word(a, cfg) for a in changed_o) and all(is_common_word(b, cfg) for b in changed_r)):
        pref, gain = lm_prefers(text, orig, rep, cfg)
        if not pref:
            return False, "ordinary word -> ordinary word" + ("" if pref is None else f" (LM gain {gain:+.2f} < {cfg.lm_margin})")
    ok, why = phonetically_plausible(orig, rep, glossary, cfg, tier)
    if not ok:
        return False, why
    if why != "spelled-letters" and abs(len(ow) - len(rw)) > cfg.max_word_delta:
        return False, f"length change {len(ow)}->{len(rw)} words"
    # replacement must be a KNOWN term: glossary, spelled letters, technical-looking, or the LLM prefers it
    known = in_glossary or why == "spelled-letters" or looks_technical(rep)
    pref, gain = lm_prefers(text, orig, rep, cfg)
    if pref is False:
        return False, f"LM prefers the original (gain {gain:+.2f} < {cfg.lm_margin})"
    if tier == "high" and not (in_glossary or why == "spelled-letters" or pref):
        return False, f"Whisper was confident (p={p:.2f}); needs glossary, spelled letters or an LM win"
    if not known and not pref and not is_common_word(rep, cfg):
        return False, f"replacement {rep!r} is not a known term (no glossary, LM or dictionary evidence)"
    extra = f", p={p:.2f}" if p is not None else ""
    extra += f", LM {gain:+.2f}" if gain is not None else ""
    return True, f"ok ({why}, tier={tier}{extra})"

def validate_chunk_edits(edits: list, seg_by_id: dict, glossary: list[str], cfg: RefineConfig, seen: set,
                         names: set | None = None, chunk: list | None = None, word_probs: dict | None = None):
    accepted, rejected = [], []
    n_words = sum(len(s["text"].split()) for s in (chunk or seg_by_id.values()))
    cap = max(2, round(cfg.max_edits_per_100_words * n_words / 100))
    for e in edits[: cap * 3]:
        key = (e.get("segment_id"), e.get("original"))
        if key in seen:
            continue                         # already accepted from an overlapping chunk
        ok, why = validate_edit(e, seg_by_id, glossary, cfg, names, word_probs)
        (accepted if ok else rejected).append({**e, "status": "accepted" if ok else "rejected", "check": why})
        if ok:
            seen.add(key)                    # a rejected proposal may be retried by the overlapping chunk
        if len(accepted) >= cap:
            break
    return accepted, rejected

SAVE_4BIT_TO_DRIVE = None          # legacy switch; None = use cfg.save_4bit_copy

def load_llm(cfg):
    if torch is None or AutoModelForCausalLM is None:
        raise ImportError("torch/transformers are not importable in this runtime "
                          f"({type(globals().get('_TORCH_ERR') or globals().get('_TF_ERR')).__name__})")
    t0 = time.time()
    local = MODELS_DIR / (cfg.model_id.replace("/", "__") + ("-4bit" if cfg.load_in_4bit else ""))
    source = "hub"
    if (local / "config.json").exists():
        try:
            print("Loading cached copy:", local)
            tok = AutoTokenizer.from_pretrained(local)
            model = AutoModelForCausalLM.from_pretrained(local, device_map="auto")
            source = "cache"
        except Exception as e:                       # damaged / half-written cache: fall back to the Hub
            print("Cached copy unusable ->", type(e).__name__, str(e)[:120], "| downloading again")
    if source == "hub":
        print("Downloading from Hugging Face:", cfg.model_id, "(one-time, ~16 GB; a few minutes)")
        tok = AutoTokenizer.from_pretrained(cfg.model_id)
        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                   bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True) if cfg.load_in_4bit else None
        model = AutoModelForCausalLM.from_pretrained(cfg.model_id, quantization_config=quant,
                                                     device_map="auto", **compat.dtype_kwarg(torch.float16))
    model.eval()
    save = cfg.save_4bit_copy if SAVE_4BIT_TO_DRIVE is None else SAVE_4BIT_TO_DRIVE
    if source == "hub" and save and cfg.load_in_4bit:
        try:
            print("Saving 4-bit copy (one-time, ~5.5 GB, a few minutes) ->", local)
            tmp = local.with_name(local.name + ".partial")
            model.save_pretrained(tmp, safe_serialization=True); tok.save_pretrained(tmp)
            if local.exists():
                import shutil; shutil.rmtree(local, ignore_errors=True)
            tmp.rename(local)
            print("Saved ->", local)
        except Exception as e:
            print("Could not save the 4-bit copy (continuing):", type(e).__name__, str(e)[:120])
    mem = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
    print(f"Loaded {cfg.model_id} from {source} in {time.time()-t0:.0f}s | GPU mem {mem:.1f} GB")
    return tok, model

def lm_total_logprob(text: str) -> float:
    """Total log-probability of `text` under the refiner (one forward pass, no generation). Used by the validators:
    an edit is kept only if the edited sentence is likelier than the original by CFG.lm_margin nats."""
    ids = tokenizer(text, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        logits = llm(**ids).logits[0, :-1].float()
    lp = torch.log_softmax(logits, dim=-1).gather(1, ids["input_ids"][0, 1:, None]).squeeze(1)
    return float(lp.sum().item())

tokenizer = llm = None

def ensure_loaded(cfg: RefineConfig, models_dir: Path | None = None) -> bool:
    "Load the refiner once (4-bit). Returns False (and leaves llm=None) if it cannot be loaded; the stage then falls back to raw."
    global tokenizer, llm, LM_SCORER, MODELS_DIR
    if models_dir: MODELS_DIR = Path(models_dir)
    if llm is not None:
        return True
    try:
        tokenizer, llm = load_llm(cfg)
    except Exception as e:
        print("WARNING:", RefineError(f"Refiner model could not be loaded ({type(e).__name__}: {str(e)[:160]}). "
                                      "Refinement will be skipped and the raw transcript used as refined.").user_message)
        tokenizer = llm = None
    LM_SCORER = lm_total_logprob if (llm is not None and cfg.use_lm_rescoring) else None
    return llm is not None

def free_llm():
    "Release the refiner before Stage 3 loads its model (both must not be resident on a 16 GB card)."
    global tokenizer, llm, LM_SCORER
    tokenizer = llm = LM_SCORER = None
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass
def llm_generate(system: str, user: str, cfg, max_new_tokens: int = None) -> str:
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    kw = {"enable_thinking": False} if "qwen3" in cfg.model_id.lower() else {}
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **kw)
    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens or cfg.max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def _loads_lenient(text: str):
    "json.loads with a repair fallback (truncated output, trailing commas, single quotes)."
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        try:
            from json_repair import repair_json
            return json.loads(repair_json(text))
        except Exception:
            return None

def parse_edits(text: str) -> list | None:
    "Extract {'edits': [...]} from model output; tolerate code fences / prose / truncation. None if unparseable."
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        return [] if text.strip() in ("", "[]") else None
    obj = _loads_lenient(m.group(0))
    if obj is None:                                  # last resort: pick out the complete edit objects one by one
        objs = [_loads_lenient(x) for x in re.findall(r"\{[^{}]*\"segment_id\"[^{}]*\}", text)]
        objs = [o for o in objs if isinstance(o, dict)]
        return objs or None
    edits = obj.get("edits", []) if isinstance(obj, dict) else obj
    return edits if isinstance(edits, list) else None

DISCOVERY_PROMPT = '''You read the opening of a meeting transcript produced by speech recognition. Identify the domain so that a
corrector can fix misrecognised terminology. Return ONLY JSON:
{"domain": "one line, e.g. 'software engineering stand-up' or 'hospital operations review'",
 "terms": ["up to 15 domain terms, product or tool names likely to be spoken in this meeting, canonical spelling"],
 "acronyms": ["up to 10 acronyms likely to appear, canonical form"],
 "people": ["names of people exactly as they appear in the text"]}
Do not invent people. Terms may include correct spellings of words that look misrecognised in the text.'''

def discover_domain(segments: list[dict], cfg: RefineConfig, max_words: int = 1500) -> dict | None:
    "One LLM call on the opening of the meeting -> domain line + seed glossary. None on any failure."
    if llm is None or not cfg.use_domain_discovery:
        return None
    words, head = 0, []
    for s in segments:
        head.append(s["text"]); words += len(s["text"].split())
        if words >= max_words:
            break
    try:
        raw = llm_generate(DISCOVERY_PROMPT, "Transcript opening:\n" + "\n".join(head) + "\n\nReturn ONLY the JSON object.", cfg, max_new_tokens=400)
        m = re.search(r"\{.*\}", re.sub(r"<think>.*?</think>", "", raw, flags=re.S), flags=re.S)
        obj = _loads_lenient(m.group(0)) if m else None
    except Exception as e:
        print("domain discovery skipped:", type(e).__name__, str(e)[:120]); return None
    if not isinstance(obj, dict):
        return None
    clean = lambda xs: [str(x).strip() for x in (xs or []) if isinstance(x, (str, int, float)) and str(x).strip()][:20]
    return {"domain": str(obj.get("domain", "")).strip()[:200], "terms": clean(obj.get("terms")),
            "acronyms": clean(obj.get("acronyms")), "people": clean(obj.get("people"))}

def refine_chunk(chunk: list[dict], glossary: list[str], suspect: dict, cfg: RefineConfig, disagree: dict | None = None) -> tuple[list, str, dict]:
    user = build_user_message(chunk, glossary, suspect, cfg.meeting_context, disagree)
    try:
        raw = llm_generate(SYSTEM_PROMPT, user, cfg)
    except Exception as e:                           # OOM, timeout, missing model: this chunk stays raw
        return [], "", {"attempts": 0, "warning": f"LLM call failed ({type(e).__name__}: {str(e)[:100]}); chunk left unedited"}
    edits = parse_edits(raw); attempts = 1
    if edits is None:
        try:
            raw = llm_generate(SYSTEM_PROMPT, user + '\n\nYour previous reply was not valid JSON. Reply with ONLY {"edits":[...]}.', cfg)
            edits = parse_edits(raw)
        except Exception:
            edits = None
        attempts = 2
    warn = None
    if edits is None:
        edits, warn = [], "LLM returned invalid JSON twice; chunk left unedited"
    return edits, raw, {"attempts": attempts, "warning": warn}

# canonical casing for terms the pipeline knows (glossary + accepted replacements + config.DEFAULT_CASING)
BUILTIN_CASING = DEFAULT_CASING

def casing_map(glossary: list[str], applied: list[dict], cfg=None) -> dict:
    cfg = cfg or CFG
    m = {t.lower(): t for t in (BUILTIN_CASING if cfg.use_builtin_casing else [])}
    m.update({t.lower(): t for t in (cfg.casing_terms or []) if len(t) > 1})
    m.update({g.lower(): g for g in glossary if len(g) > 1})
    m.update({e["replacement"].lower(): e["replacement"] for e in applied if len(e["replacement"]) > 1})
    return m

def apply_casing(segments: list[dict], cmap: dict, names: set) -> list[dict]:
    "Fix casing of known terms word-by-word (never touches protected tokens or names). Returns the edits made."
    fixes = []
    for s in segments:
        def _fix(m):
            w = m.group(0); canon = cmap.get(w.lower())
            if canon and canon != w and not is_critical(w) and w not in names and not is_common_word(w):
                fixes.append({"segment_id": s["id"], "original": w, "replacement": canon, "reason": "canonical casing",
                              "confidence": 1.0, "status": "accepted", "check": "casing", "applied": True})
                return canon
            return w
        s["text"] = re.sub(r"[A-Za-z][A-Za-z0-9./+#-]*", _fix, s["text"])
    return fixes

def apply_edits(segments: list[dict], accepted: list[dict]) -> list[dict]:
    "Whole-word replacement of every occurrence. Raw segments are never modified (deep copy)."
    refined = copy.deepcopy(segments)
    by_id = {s["id"]: s for s in refined}
    for e in accepted:
        s = by_id[int(e["segment_id"])]
        if find_spans(e["original"], s["text"]):
            s["text"] = replace_spans(s["text"], e["original"], e["replacement"]); e["applied"] = True
        else:
            e["applied"] = False; e["check"] = "superseded by an earlier edit in the same segment"
    return refined

def propagate_edits(segments: list[dict], accepted: list[dict], seg_by_id: dict, glossary: list[str], cfg: RefineConfig,
                    names: set, word_probs: dict, seen: set) -> tuple[list, list]:
    "Whisper mishears a term the same way every time; apply each accepted fix transcript-wide (validated per segment)."
    extra_acc, extra_rej = [], []
    pairs = {(e["original"], e["replacement"]) for e in accepted}
    for orig, rep in sorted(pairs):
        for s in segments:
            key = (s["id"], orig)
            if key in seen or not find_spans(orig, s["text"]):
                continue
            e = {"segment_id": s["id"], "original": orig, "replacement": rep, "reason": "propagated from an accepted edit", "confidence": 1.0}
            ok, why = validate_edit(e, seg_by_id, glossary, cfg, names, word_probs)
            (extra_acc if ok else extra_rej).append({**e, "status": "accepted" if ok else "rejected", "check": why, "propagated": True})
            if ok:
                seen.add(key)
    return extra_acc, extra_rej

def edit_ratio(raw_segs: list[dict], ref_segs: list[dict]) -> tuple[int, int]:
    a = " ".join(s["text"] for s in raw_segs).split(); b = " ".join(s["text"] for s in ref_segs).split()
    sm = difflib.SequenceMatcher(None, a, b, autojunk=False)
    changed = sum(max(i2 - i1, j2 - j1) for tag, i1, i2, j1, j2 in sm.get_opcodes() if tag != "equal")
    return changed, len(a)

def run_stage2(segments: list[dict], glossary: list[str], suspect: dict, cfg: RefineConfig, on_progress=None,
               disagree: dict | None = None, names: set | None = None, word_probs: dict | None = None) -> dict:
    t0 = time.time(); seg_by_id = {s["id"]: s for s in segments}
    names = set(names if names is not None else NAMES); word_probs = word_probs or {}
    glossary = list(glossary); warnings, status = [], "ok"
    # ---- domain discovery: context + seed glossary (replaces the hardcoded meeting context) ----
    domain = None
    if cfg.use_domain_discovery and not cfg.meeting_context:
        if on_progress: on_progress("discovering domain", 0.0)
        domain = discover_domain(segments, cfg, max_words=cfg.discovery_max_words)
        if domain:
            cfg.meeting_context = domain["domain"] or cfg.meeting_context
            glossary = sorted(set(glossary) | set(domain["terms"]) | set(domain["acronyms"]))
            names |= {t for p in domain["people"] for t in tokens(p) if t[:1].isupper()}
        else:
            warnings.append("domain discovery unavailable; generic context used")
    chunks = make_chunks(segments, cfg.chunk_segments, cfg.chunk_overlap)
    accepted, rejected, seen = [], [], set()
    if llm is None:
        warnings.append("refiner model not loaded; refined transcript = raw transcript"); status = "fallback_raw"
        chunks = []
    for ci, chunk in enumerate(chunks):
        edits, raw, info = refine_chunk(chunk, glossary, suspect, cfg, disagree)
        if info["warning"]:
            warnings.append(f"chunk {ci+1}: {info['warning']}")
        a, r = validate_chunk_edits(edits, seg_by_id, glossary, cfg, seen, names, chunk, word_probs)
        accepted += a; rejected += r
        if on_progress:
            on_progress("refining", (ci + 1) / len(chunks))
    pa, pr = propagate_edits(segments, accepted, seg_by_id, glossary, cfg, names, word_probs, seen)
    accepted += pa; rejected += pr
    refined = apply_edits(segments, accepted)
    applied = [e for e in accepted if e.get("applied")]
    casing = apply_casing(refined, casing_map(glossary, applied, cfg), names)
    # ---- circuit breaker: too many changes or any protected-token damage -> raw transcript ----
    changed, total = edit_ratio(segments, refined)
    damage = sum(1 for a, b in zip(segments, refined) if protected_in(a["text"]) != protected_in(b["text"]))
    if total and (changed / total > cfg.max_edit_ratio or damage):
        warnings.append(f"circuit breaker: {changed}/{total} words changed ({100*changed/total:.1f}%) or protected-token damage={damage}; "
                        "refined transcript reverted to raw")
        status = "fallback_raw"; refined = copy.deepcopy(segments); applied, casing = [], []
        for e in accepted: e["applied"] = False; e["check"] = "reverted by circuit breaker"
        changed = 0
    return {
        "refined_segments": refined,
        "refined_text": " ".join(s["text"] for s in refined),
        "edits_applied": applied + casing,
        "edits_rejected": rejected + [e for e in accepted if not e.get("applied")],
        "warnings": warnings, "status": status, "domain": domain, "glossary": glossary,
        "edit_ratio": round(changed / max(1, total), 4), "names": sorted(names),
        "meta": {"refiner": f"{cfg.model_id} (4-bit NF4)" if cfg.load_in_4bit else cfg.model_id,
                 "n_chunks": len(chunks), "glossary": glossary, "n_suspect_segments": len(suspect),
                 "meeting_context": cfg.meeting_context, "lm_rescoring": LM_SCORER is not None,
                 "settings": {k: v for k, v in vars(cfg).items() if k != "glossary"},
                 "elapsed_s": round(time.time() - t0, 1)},
    }

def damage_report(raw_segs: list, ref_segs: list, glossary: list[str], names: set | None = None) -> dict:
    """Independent of the validators: compares the protected-token sequence (numbers, dates, times, negations,
    commitments) and the set of person names of raw vs refined, segment by segment. Target is always 0."""
    issues = []
    for a, b in zip(raw_segs, ref_segs):
        if protected_in(a["text"]) != protected_in(b["text"]):
            issues.append({"segment_id": a["id"], "type": "protected_tokens",
                           "raw": protected_in(a["text"]), "refined": protected_in(b["text"])})
        na = set(capitalised_names(a["text"], glossary, names)); nb = set(capitalised_names(b["text"], glossary, names))
        if na - nb:
            issues.append({"segment_id": a["id"], "type": "name_removed", "raw": sorted(na - nb)})
    if len(raw_segs) != len(ref_segs) or any(a["id"] != b["id"] for a, b in zip(raw_segs, ref_segs)):
        issues.append({"type": "segment_ids_changed"})
    return {"damage_count": len(issues), "issues": issues}

from datetime import datetime

def fmt_ts(sec: float) -> str:
    sec = int(sec or 0); h, rem = divmod(sec, 3600); m, s_ = divmod(rem, 60)
    return f"{h}:{m:02d}:{s_:02d}" if h else f"{m:02d}:{s_:02d}"

def numbered_lines(segments: list) -> str:
    return "\n".join(f"[L{s['id']:03d} {fmt_ts(s.get('start', 0))}] [{s.get('speaker') or DIAR_CFG.unknown_label}]: {s['text']}"
                      for s in segments)

def build_stage2_record(s1: dict, r2: dict, dmg: dict) -> dict:
    "stage1_record.json + refined_transcript, edits, domain, glossary, status. This is what Stage 3 reads."
    record = copy.deepcopy(s1["record"])
    record["schema_version"] = record.get("schema_version", "1.0")
    record.setdefault("meta", {}).setdefault("models", {})["refiner"] = r2["meta"]["refiner"]
    record["meta"]["refiner_settings"] = r2["meta"]["settings"]
    record["meta"]["refiner_warnings"] = r2["warnings"]
    record["meta"]["meeting_context"] = r2["meta"]["meeting_context"]
    record["refined_transcript"] = {
        "status": r2["status"],
        "text": r2["refined_text"],
        "segments": [{k: s.get(k) for k in ("id", "start", "end", "speaker", "text")} for s in r2["refined_segments"]],
        "lines": numbered_lines(r2["refined_segments"]).split("\n"),
        "edits": [{k: e.get(k) for k in ("segment_id", "original", "replacement", "reason", "confidence", "check", "propagated")}
                  for e in r2["edits_applied"]],
        "rejected_edits": [{k: e.get(k) for k in ("segment_id", "original", "replacement", "reason", "confidence", "check")}
                           for e in r2["edits_rejected"]],
        "edit_ratio": r2["edit_ratio"],
        "damage_count": dmg["damage_count"],
    }
    record["domain"] = r2["domain"]
    record["glossary"] = r2["glossary"]
    record["people"] = r2["names"]
    return record

def save_stage2(s1: dict, r2: dict, dmg: dict, out_dir: Path, run_dir: Path | None = None) -> Path:
    run_dir = Path(run_dir) if run_dir else out_dir / f"stage2_{datetime.now():%Y%m%d_%H%M%S}"; run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "refined_transcript.txt").write_text(r2["refined_text"], encoding="utf-8")
    (run_dir / "refined_transcript_numbered.txt").write_text(numbered_lines(r2["refined_segments"]), encoding="utf-8")
    edit_log = {"applied": r2["edits_applied"], "rejected": r2["edits_rejected"], "warnings": r2["warnings"],
                "status": r2["status"], "domain": r2["domain"], "glossary": r2["glossary"],
                "damage_check": dmg, "meta": r2["meta"]}
    (run_dir / "edit_log.json").write_text(json.dumps(edit_log, indent=2, ensure_ascii=False), encoding="utf-8")
    record = build_stage2_record(s1, r2, dmg)
    (run_dir / "stage2_record.json").write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved to", run_dir)
    for f in sorted(run_dir.iterdir()):
        print(f"  {f.name:<34} {f.stat().st_size/1024:7.1f} KB")
    return run_dir

# Three measurements for the tech note:
#   1. WER raw -> WER refined against a reference transcript (REFERENCE_TXT, e.g. clipa.txt from Stage 1)
#   2. planted-term fix rate (PLANTED: what Whisper produced -> what it should be)
#   3. damage count, edit ratio, rejected-edit histogram, and (optional) a negative control
REFERENCE_TXT = None          # path to a human transcript of the same recording (set in the notebook Inputs cell)
PLANTED = {                   # "what Whisper produced" : "what it should be"   (leave empty to skip)
    # "post grass": "PostgreSQL",
    # "see eye see dee": "CI/CD",
    # "pie torch": "PyTorch",
}

def _norm_eval(s: str) -> str:
    try:
        from num2words import num2words
        s = re.sub(r"\d+", lambda m: " " + num2words(int(m.group())) + " ", s)
    except Exception:
        pass
    s = s.lower().replace("/", " ").replace("-", " ")
    return " ".join(re.sub(r"[^a-z0-9']", " ", s).split())

def wer(ref: str, hyp: str) -> float:
    r, h = _norm_eval(ref).split(), _norm_eval(hyp).split()
    if not r:
        return float(bool(h))
    try:
        import jiwer
        return float(jiwer.wer(" ".join(r), " ".join(h) if h else ""))
    except Exception:
        prev = list(range(len(h) + 1))
        for i in range(1, len(r) + 1):
            cur = [i] + [0] * len(h)
            for j in range(1, len(h) + 1):
                cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (r[i - 1] != h[j - 1]))
            prev = cur
        return prev[-1] / len(r)


# ----------------------------------------------------------------------------- entry points
def load_stage1(path: str) -> dict:
    """Reads stage1_record.json (schema 1.0 from pipeline_stt.save_outputs). Also accepts the flat
    raw_transcript.json shape. Returns segments plus ALL the evidence Stage 1 produced."""
    d = json.loads(Path(path).read_text(encoding="utf-8"))
    if "raw_transcript" in d and isinstance(d["raw_transcript"], dict):          # stage1_record.json
        segs_in = d["raw_transcript"]["segments"]
        low  = d.get("low_confidence_words", [])
        meta = d.get("meta", {})
        asr2 = d.get("asr2")
        glossary = d.get("glossary", [])
    elif "segments" in d:                                                        # raw_transcript.json
        segs_in = d["segments"]
        low  = [{"segment_id": s["id"], "word": w, "p": None} for s in d["segments"] for w in s.get("low_conf_words", [])]
        meta = {"models": {"stt": d.get("info", {}).get("model")}, "duration_s": d.get("info", {}).get("duration")}
        asr2 = d.get("asr2"); glossary = []
    else:
        raise RefineError("Unrecognised Stage 1 file: expected 'raw_transcript' or 'segments'.")
    segs, word_probs, critical_low = [], {}, {}
    for s in segs_in:
        text = str(s["text"]).strip()
        if not text:
            continue
        segs.append({"id": int(s["id"]), "start": s.get("start"), "end": s.get("end"), "text": text,
                     "speaker": s.get("speaker") or DIAR_CFG.unknown_label})
        word_probs[int(s["id"])] = [(w["w"].strip(), w.get("p")) for w in s.get("words", []) if w.get("p") is not None]
        if s.get("critical_low_conf"):
            critical_low[int(s["id"])] = list(s["critical_low_conf"])
    ids = [s["id"] for s in segs]
    if ids != list(range(1, len(ids) + 1)):
        raise RefineError(f"Segment ids must be 1..n (got {ids[:5]}...). Re-run Stage 1's export cell.")
    disagree = {}
    for dis in (asr2 or {}).get("disagreements", []) or []:
        if dis.get("critical"):                       # protected tokens in a disagreement are never auto-resolved
            continue
        if dis.get("type") == "replace" and dis.get("whisper") and dis.get("parakeet"):
            for sid in dis.get("seg_ids", []):
                disagree.setdefault(int(sid), []).append({"whisper": dis["whisper"], "alt": dis["parakeet"]})
    return {"segments": segs, "low_confidence_words": low, "critical_low": critical_low, "word_probs": word_probs,
            "disagreements": disagree, "asr2": asr2, "glossary": list(glossary), "meta": meta, "record": d}

def prepare(s1: dict, cfg: RefineConfig) -> dict:
    "Names, glossary, suspect spans and disagreements for a loaded Stage 1 record (the notebook's cells 5-6)."
    global NAMES
    segments = s1["segments"]; word_probs = s1["word_probs"]
    glossary_user = list(cfg.glossary) or list(s1.get("glossary", []))
    NAMES = person_names(segments, glossary_user, cfg)
    auto = auto_glossary(segments, word_probs, cfg=cfg)
    glossary = sorted(set(glossary_user) | set(auto))
    seg_tokens = {s["id"]: {t.lower().strip(" .,?!") for t in tokens(s["text"])} for s in segments}
    suspect, stale = {}, 0
    for w in s1["low_confidence_words"]:
        sid, word = int(w["segment_id"]), str(w["word"]).strip(" .,?!")
        if sid in seg_tokens and word.lower() in seg_tokens[sid] and not is_critical(word):
            suspect.setdefault(sid, []).append(word)
        else:
            stale += 1
    return {"glossary": glossary, "auto_glossary": auto, "suspect": suspect, "stale_suspects": stale,
            "disagree": s1["disagreements"], "names": NAMES, "word_probs": word_probs}

def refine(stage1_record, cfg: RefineConfig | None = None, models_dir: Path | None = None, on_progress=None,
           user_glossary=None) -> tuple[dict, dict, dict]:
    """Stage 2 end to end: record (path or dict) -> (R2, S1, damage). Never raises for model problems:
    if the refiner cannot run, R2['status'] == 'fallback_raw' and the refined transcript equals the raw one."""
    cfg = cfg or RefineConfig()
    if user_glossary:
        cfg.glossary = sorted(set(cfg.glossary) | set(user_glossary))
    if isinstance(stage1_record, (str, Path)):
        s1 = load_stage1(str(stage1_record))
    else:
        import tempfile
        tmp = Path(tempfile.mkdtemp()) / "stage1_record.json"; tmp.write_text(json.dumps(stage1_record)); s1 = load_stage1(str(tmp))
    prep = prepare(s1, cfg)
    ensure_loaded(cfg, models_dir)
    r2 = run_stage2(s1["segments"], prep["glossary"], prep["suspect"], cfg, on_progress=on_progress,
                    disagree=prep["disagree"], names=prep["names"], word_probs=prep["word_probs"])
    dmg = damage_report(s1["segments"], r2["refined_segments"], prep["glossary"], set(r2["names"]))
    return r2, s1, dmg

def negative_control(clean_text: str, cfg: RefineConfig | None = None, models_dir: Path | None = None) -> dict:
    "Feed a clean human transcript through Stage 2; a good refiner changes ~0 words."
    global NAMES
    cfg = copy.copy(cfg or RefineConfig()); cfg.meeting_context = ""
    sents = [t.strip() for t in re.split(r"(?<=[.!?])\s+", clean_text.strip()) if t.strip()]
    segs = [{"id": i + 1, "start": None, "end": None, "text": t} for i, t in enumerate(sents)]
    ensure_loaded(cfg, models_dir); NAMES = person_names(segs, cfg.glossary, cfg)
    r = run_stage2(segs, list(cfg.glossary), {}, cfg, disagree={}, names=NAMES, word_probs={})
    changed, total = edit_ratio(segs, r["refined_segments"])
    print(f"negative control: {changed}/{total} words changed ({100*changed/max(1,total):.2f}%), applied {len(r['edits_applied'])}, rejected {len(r['edits_rejected'])}")
    return r

def self_test() -> int:
    "Validator self-test (no model). Returns the number of failed checks."
    global LM_SCORER
    # Two groups: (A) deterministic checks only (LM_SCORER off) and (B) with a stand-in LM scorer that prefers
    # the known-correct sentence. Every guard added to Cell 5 / Cell 8 has a must-pass and a must-fail case here.
    _test_segs = {1: {"id": 1, "text": "The post grass database crashed twice, Priya will fix it by Friday."},
                  2: {"id": 2, "text": "We will not ship on the fifteenth, build time is fifty minutes."},
                  3: {"id": 3, "text": "The see eye see dee pipeline pushes to rod via the A P I."},
                  4: {"id": 4, "text": "Priya will send the terror form scripts."},
                  5: {"id": 5, "text": "We saw it in production. Push the fix to rod tonight."},
                  6: {"id": 6, "text": "We should increase the retry count and enable the cache before the demo."},
                  7: {"id": 7, "text": "Deploy it on the Cooper Netties cluster running on ay double you ess."},
                  8: {"id": 8, "text": "The launch is on the 23rd of October at 5pm, budget is $500."},
                  9: {"id": 9, "text": "We evaluated the pie torch model, then the pie torch export."}}
    _probs = {7: [("Deploy", 0.98), ("it", 0.99), ("on", 0.99), ("the", 0.99), ("Cooper", 0.31), ("Netties", 0.22),
                  ("cluster", 0.97), ("running", 0.9), ("on", 0.99), ("ay", 0.4), ("double", 0.5), ("you", 0.6), ("ess", 0.3)],
              1: [("Priya", 0.96)], 4: [("Priya", 0.97), ("terror", 0.35), ("form", 0.6)]}
    _names = {"Priya", "Cooper", "Netties"}          # 'Cooper Netties' is what NER would call a person -> must still be fixable
    _glossary = ["Terraform"]

    def _good_lm(text):                 # stand-in: rewards sentences containing the correct terms (whole words)
        bonus = sum(3.0 for t in ("PostgreSQL", "prod", "PyTorch", "Kubernetes", "Adam optimizer", "Terraform")
                    if re.search(r"(?<![A-Za-z0-9])" + re.escape(t) + r"(?![A-Za-z0-9])", text))
        return -len(text.split()) + bonus

    _cases_A = [   # (edit, expected) with LM_SCORER = None
        ({"segment_id": 1, "original": "post grass", "replacement": "PostgreSQL", "confidence": 0.9}, True),
        ({"segment_id": 3, "original": "see eye see dee", "replacement": "CI/CD", "confidence": 0.9}, True),
        ({"segment_id": 3, "original": "A P I", "replacement": "API", "confidence": 0.9}, True),
        ({"segment_id": 7, "original": "ay double you ess", "replacement": "AWS", "confidence": 0.9}, True),     # spelled W
        ({"segment_id": 4, "original": "terror form", "replacement": "Terraform", "confidence": 0.9}, True),     # glossary
        ({"segment_id": 9, "original": "pie torch", "replacement": "PyTorch", "confidence": 0.9}, True),         # canonical casing
        ({"segment_id": 2, "original": "will not", "replacement": "will", "confidence": 0.9}, False),           # negation
        ({"segment_id": 2, "original": "fifty", "replacement": "fifteen", "confidence": 0.9}, False),           # number
        ({"segment_id": 1, "original": "Priya", "replacement": "Prya", "confidence": 0.9}, False),              # name (confident)
        ({"segment_id": 4, "original": "Priya", "replacement": "Pria", "confidence": 0.9}, False),              # name (confident)
        ({"segment_id": 1, "original": "crashed twice", "replacement": "failed", "confidence": 0.9}, False),    # rewrite
        ({"segment_id": 1, "original": "postgres", "replacement": "PostgreSQL", "confidence": 0.9}, False),     # not verbatim
        ({"segment_id": 1, "original": "post grass", "replacement": "PostgreSQL", "confidence": 0.3}, False),   # low conf
        ({"segment_id": 5, "original": "rod", "replacement": "prod", "confidence": 0.9}, False),                # 'prod' needs LM evidence
        ({"segment_id": 6, "original": "increase", "replacement": "decrease", "confidence": 0.9}, False),       # polarity
        ({"segment_id": 6, "original": "enable", "replacement": "disable", "confidence": 0.9}, False),          # polarity
        ({"segment_id": 6, "original": "demo", "replacement": "memo", "confidence": 0.9}, False),               # ordinary words
        ({"segment_id": 8, "original": "23rd", "replacement": "22nd", "confidence": 0.9}, False),               # ordinal
        ({"segment_id": 8, "original": "5pm", "replacement": "5am", "confidence": 0.9}, False),                 # time
        ({"segment_id": 8, "original": "$500", "replacement": "$600", "confidence": 0.9}, False),               # currency
        ({"segment_id": 1, "original": "Friday", "replacement": "Monday", "confidence": 0.9}, False),           # weekday
    ]
    _cases_B = [   # with the stand-in LM scorer
        ({"segment_id": 5, "original": "rod", "replacement": "prod", "confidence": 0.9}, True),                 # LM evidence
        ({"segment_id": 7, "original": "Cooper Netties", "replacement": "Kubernetes", "confidence": 0.9}, True), # capitalised jargon, low p
        ({"segment_id": 6, "original": "increase", "replacement": "decrease", "confidence": 0.9}, False),       # polarity still blocked
        ({"segment_id": 6, "original": "demo", "replacement": "memo", "confidence": 0.9}, False),               # LM does not prefer it
        ({"segment_id": 1, "original": "Priya", "replacement": "Prya", "confidence": 0.9}, False),              # name still blocked
    ]

    def _run_cases(cases, label):
        fails = 0
        for e, expect in cases:
            ok, why = validate_edit(e, _test_segs, _glossary, CFG, _names, _probs)
            flag = "PASS" if ok == expect else "FAIL"; fails += ok != expect
            print(f"{flag}  {'accept' if ok else 'reject':<6} {e['original']!r:>20} -> {e['replacement']!r:<12} {why}")
        print(f"-- {label}: {len(cases) - fails}/{len(cases)} as expected\n")
        return fails

    _saved = LM_SCORER
    LM_SCORER = None;     fails = _run_cases(_cases_A, "A: deterministic checks only")
    LM_SCORER = _good_lm; fails += _run_cases(_cases_B, "B: with stand-in LM scorer")
    LM_SCORER = _saved

    # whole-word replacement and multiple occurrences
    _t = _test_segs[5]["text"]; _r = replace_spans(_t, "rod", "prod")
    _ok = "pproduction" not in _r and _r.endswith("to prod tonight.")
    print(("PASS" if _ok else "FAIL"), " whole-word replace:", _r); fails += not _ok
    _r2 = replace_spans(_test_segs[9]["text"], "pie torch", "PyTorch")
    _ok = _r2.count("PyTorch") == 2
    print(("PASS" if _ok else "FAIL"), " replaces every occurrence:", _r2); fails += not _ok
    print("\nALL GOOD" if not fails else f"\n{fails} validator test(s) failed - fix before running the model")
    return fails

def print_report(R2: dict, RAW_SEGMENTS: list, CFG: RefineConfig):
    """Prints applied and rejected edits, edit ratio, and a side-by-side diff."""
    print("=== APPLIED EDITS ===")
    applied = R2.get("edits_applied", [])
    if not applied:
        print("  (None)")
    for e in applied:
        print(f"  seg {int(e.get('segment_id', 0)):03d}  {e.get('original')!r} -> {e.get('replacement')!r}   [{e.get('check', 'ok')}]  conf={e.get('confidence')}")

    print("\n=== REJECTED EDITS (FIREWALL PREVENTED DAMAGE) ===")
    rejected = R2.get("edits_rejected", [])
    if not rejected:
        print("  (None)")
    for e in rejected:
        print(f"  seg {str(e.get('segment_id', '')):>3}  {e.get('original')!r} -> {e.get('replacement')!r}   REASON: {e.get('check', 'unknown')}")

    for w in R2.get("warnings", []):
        print("WARNING:", w)

    ratio = R2.get("edit_ratio", 0.0)
    print(f"\nEdit Ratio: {100 * ratio:.2f}% of words changed")

    print("\n=== CHANGED SEGMENTS (RAW vs REFINED) ===")
    changed_count = 0
    ref_segs = R2.get("refined_segments", [])
    for a, b in zip(RAW_SEGMENTS, ref_segs):
        if a.get("text", "").strip() != b.get("text", "").strip():
            changed_count += 1
            print(f"[{a['id']:03d}] RAW : {a['text']}\n      REF : {b['text']}\n")
    if changed_count == 0:
        print("  (No segments were modified)")


In [ ]:
%%writefile stages/document.py
"""Stage 3 - minutes, decisions, action items (LLM #2). Built from the Stage 3 notebook cells 4-14.
The LLM proposes items with evidence quotes; the verifier decides what enters the record; Markdown is rendered by code."""
import re, json, copy, time, gc
try:                                  # heavy imports are optional: without them the stage degrades instead of crashing
    import torch
except Exception as _e:
    torch, _TORCH_ERR = None, _e
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
except Exception as _e:
    AutoTokenizer = AutoModelForCausalLM = BitsAndBytesConfig = None
    _TF_ERR = _e
from pathlib import Path
from datetime import datetime
from dataclasses import replace as _dc_replace
from config import DocConfig, DIAR_CFG, models_dir
from errors import PipelineError, DocumentError
import compat

CFG = DocConfig()
MODELS_DIR = Path(models_dir())


def _segments_from_text(text: str) -> list[dict]:
    numbered = re.findall(r"\[L(\d+)\s+([\d:\-]+)\]\s*(?:\[([^\]]+)\]:\s*)?(.*)", text)
    if numbered:                                                       # refined_transcript_numbered.txt
        return [{"id": int(i), "start": None, "end": None, "speaker": spk or DIAR_CFG.unknown_label, "text": t.strip()}
                for i, _, spk, t in numbered]
    sents = [t.strip() for t in re.split(r"(?<=[.!?])\s+", text.strip()) if t.strip()]
    return [{"id": i + 1, "start": None, "end": None, "text": t} for i, t in enumerate(sents)]

def load_stage2(path: str) -> dict:
    """Reads stage2_record.json. Stage 3 reads ONLY the refined transcript (never the raw one), plus the domain,
    glossary and people list Stage 2 established. Plain .txt transcripts are accepted for testing."""
    p = Path(path); raw = p.read_text(encoding="utf-8")
    if p.suffix.lower() == ".txt":
        return {"segments": _segments_from_text(raw), "record": {"refined_transcript": {"text": raw}}, "meta": {},
                "domain": None, "glossary": [], "people": []}
    d = json.loads(raw)
    rt = d.get("refined_transcript")
    if isinstance(rt, dict) and rt.get("segments"):
        segs = rt["segments"]
    elif isinstance(rt, dict) and rt.get("text"):
        segs = _segments_from_text(rt["text"])
    elif isinstance(rt, str):
        segs = _segments_from_text(rt)
    else:
        raise DocumentError(f"No refined_transcript found. Keys: {list(d.keys())}. Run Stage 2 first.")
    segs = [{"id": int(s.get("id", i + 1)), "start": s.get("start"), "end": s.get("end"),
             "speaker": s.get("speaker") or DIAR_CFG.unknown_label, "text": str(s["text"]).strip()}
            for i, s in enumerate(segs) if str(s.get("text", "")).strip()]
    if isinstance(rt, dict) and rt.get("status") == "fallback_raw":
        print("NOTE: Stage 2 fell back to the raw transcript (see meta.refiner_warnings); documenting the raw text.")
    return {"segments": segs, "record": d, "meta": d.get("meta", {}), "domain": d.get("domain"),
            "glossary": d.get("glossary", []) or [], "people": d.get("people", []) or []}

def fmt_ts(sec) -> str:
    if sec is None: return "--:--"
    sec = int(sec); h, rem = divmod(sec, 3600); m, s_ = divmod(rem, 60)
    return f"{h}:{m:02d}:{s_:02d}" if h else f"{m:02d}:{s_:02d}"

def numbered(segments: list[dict]) -> str:
    return "\n".join(f"[L{s['id']:03d} {fmt_ts(s['start'])}] {s['text']}" for s in segments)

def make_chunks(segments: list[dict], max_words: int, overlap: int) -> list[list[dict]]:
    chunks, cur, words = [], [], 0
    for s in segments:
        w = len(s["text"].split())
        if cur and words + w > max_words:
            chunks.append(cur); cur = cur[-overlap:] if overlap else []; words = sum(len(x["text"].split()) for x in cur)
        cur.append(s); words += w
    if cur: chunks.append(cur)
    return chunks


SYSTEM_PROMPT = '''You write meeting minutes from a refined meeting transcript. Each line is numbered like [L012 03:41].
Use ONLY what was said. Never infer, never guess, never add recommendations of your own.

Return ONLY a JSON object with exactly these keys:
{
 "summary": "3-5 sentence overview of the meeting",
 "discussion_points": [{"topic": "short topic name", "points": ["concise point", ...], "evidence_lines": [3, 4]}],
 "candidates": [{"statement": "what was said, concisely",
                 "type": "agreed_decision" | "proposal" | "open_question" | "tentative",
                 "evidence_quote": "verbatim words from the transcript, the WHOLE clause including any 'not'",
                 "evidence_lines": [12, 13]}],
 "action_items": [{"task": "the work to be done",
                   "owner": "name exactly as spoken, or unspecified",
                   "owner_stated": true | false,
                   "deadline": "time exactly as spoken (e.g. by Friday), or unspecified",
                   "deadline_stated": true | false,
                   "evidence_quote": "verbatim words from the transcript containing the task, the owner and the deadline",
                   "evidence_lines": [31]}]
}

Rules for "type":
- agreed_decision: the group clearly agreed or finalised it. "Okay, we're going with Postgres." / "We decided the release date is the 15th." / "Agreed."
- proposal: someone suggested it and it was NOT clearly accepted. "I think we could use Redis." / "Rahul suggested X, but nobody agreed."
- open_question: raised but not answered. "Should we move to AWS?"
- tentative: hedged. "Let's maybe revisit that." / "We should probably..."
A decision that was later reversed in the meeting must appear with its FINAL state only.

Rules for action items:
- Only commitments or assignments that were accepted or stated as a plan. "I'll send the report." / "Priya, can you handle it?" + "Sure".
- A suggestion with no acceptance is NOT a task. "Someone needs to write the runbook" IS a task, with owner unspecified.
- owner: copy the name exactly as spoken, from the SAME sentence as the task. If no name is spoken for that task, write "unspecified" and owner_stated=false.
- A first-person commitment ("I'll do it", "I will send it", "let me check") has NO owner unless the speaker says their own name: owner "unspecified". A speaker label is not a name.
- deadline: copy the time words exactly as spoken ("by Friday", "end of next week"), from the same sentence as the task. Never convert to a calendar date. A date mentioned for another reason is not a deadline. If none spoken, "unspecified" and deadline_stated=false.
- Every candidate and action item MUST include evidence_quote (exact words from the lines, at most 25 words, one clause) and evidence_lines (at most 3 consecutive lines; 1-2 for action items).
- Spell people's names as they are spelled in the transcript / glossary.
- If there are no decisions or no action items, return empty lists. That is a correct answer.
- Be concise. No filler. No commentary outside the JSON.'''

def build_user_message(chunk: list[dict], context: str, part: str = "", glossary: list[str] | None = None) -> str:
    gl = f"Names and terms used in this meeting: {', '.join(glossary[:40])}\n" if glossary else ""
    return (f"Meeting context: {context}\n{gl}{part}"
            f"Transcript:\n{numbered(chunk)}\n\nReturn ONLY the JSON object.")

from rapidfuzz import fuzz

NEG_ACCEPT = ("nobody agreed", "no one agreed", "not agreed", "wasn't agreed", "was rejected", "we rejected",
              "decided against", "didn't agree", "did not agree", "not adopted", "shot down", "parked", "table that",
              "tabled", "hold off", "let's revisit", "we'll revisit", "revisit that", "no consensus", "pushed back",
              "left open", "not now", "maybe later", "park that", "not decided", "undecided", "still open")
_NEG_ACCEPT_RE = re.compile(r"(?<![a-z0-9'])(?:" + "|".join(re.escape(p) for p in NEG_ACCEPT) + r")(?![a-z0-9'])")
DECISION_MARKERS = ("decided", "decide", "agreed", "agree", "we're going with", "we are going with", "go with", "going with",
                    "approved", "final", "finalised", "finalized", "settled", "confirmed", "sign off", "signed off",
                    "let's do", "we will", "we'll", "okay so", "so we are", "so we're", "it is settled", "locked in")
_MARKER_RE = re.compile(r"(?<![a-z0-9'])(?:" + "|".join(re.escape(p) for p in DECISION_MARKERS) + r")(?![a-z0-9'])")
_FIRST_PERSON_RE = re.compile(r"(?<![a-z0-9'])(?:i'll|i will|i can|i could|i'm going to|i am going to|let me|i'd|i shall|i'm gonna)(?![a-z0-9'])")
_COMMIT_RE = re.compile(r"(?<![a-z0-9'])(?:will|'ll|shall|going to|gonna|can you|could you|would you|please|needs? to|has to|have to|"
                        r"should|must|take care|handle|own|owns|owner|responsible|assigned|action item|to do|todo|by|before|until|till|due)(?![a-z0-9'])")
_DEADLINE_CUE = {"by", "before", "until", "till", "on", "within", "due", "this", "next", "end", "eod", "eow", "eom", "today",
                 "tomorrow", "tonight", "later", "after", "in", "at"}
_STOP = set("""the a an and or but so if then of to in on at for with by from as is are was were be been being have has had
do does did not no we you they he she it this that these those will would can could should may might must shall i me my our
your their its it's that's let's okay ok yes well just also very really there here about into over under up down out off
than too all any some each every one two three four five six seven eight nine ten""".split())
_NEGATION_WORDS = {"not", "no", "never", "none", "nothing", "neither", "nor", "cannot", "without", "nobody"}
_NEGATION_EQUIV = {"postpone", "postponed", "cancel", "cancelled", "canceled", "reject", "rejected", "skip", "skipped", "drop",
                   "dropped", "stop", "stopped", "delay", "delayed", "defer", "deferred", "decline", "declined", "avoid",
                   "exclude", "excluded", "remove", "removed", "hold", "halt", "halted", "against", "won't", "don't", "unchanged"}
_AUX = {"even", "really", "just", "yet", "ever", "also", "to", "be", "been", "being", "very", "quite", "fully", "actually", "going"}

def _norm(s: str) -> str:
    return " ".join(re.sub(r"[^a-z0-9' ]", " ", str(s).lower().replace("’", "'")).split())

def _words(s: str) -> list[str]:
    return _norm(s).split()

def evidence_text(lines_ids, lines: dict) -> str:
    ids = [int(i) for i in (lines_ids or []) if str(i).lstrip("L").isdigit() or isinstance(i, int)]
    return " ".join(lines[i] for i in ids if i in lines)

def _negs(s: str) -> set[str]:
    return {w for w in _words(s) if w in _NEGATION_WORDS or w.endswith("n't")}

def _clean_ids(ids) -> list[int]:
    out = []
    for i in (ids or []):
        s = str(i).strip().lstrip("L")
        if s.isdigit(): out.append(int(s))
    return sorted(set(out))

def quote_found(quote: str, ev_text: str, cfg: DocConfig) -> tuple[bool, int, tuple | None]:
    """The quote must occur in the evidence text (exact after normalisation, or fuzzy-aligned), AND the words
    immediately around it must not carry a negation the quote left out: 'migrate production this quarter' cut out
    of 'we will not migrate production this quarter' is rejected even though it is a verbatim substring."""
    q, e = _norm(quote), _norm(ev_text)
    if not q or not e or len(q.split()) > cfg.max_quote_words:
        return False, 0, None
    if q in e:
        start = e.index(q); end = start + len(q); score = 100
    else:
        if len(e) < 0.9 * len(q):                    # the evidence is shorter than the quote: the quote cannot be in it
            return False, int(fuzz.ratio(q, e)), None  # (partial_ratio would score how much of the EVIDENCE is in the quote)
        al = fuzz.partial_ratio_alignment(q, e)
        score = int(al.score) if al else 0
        if score < cfg.quote_min_ratio:
            return False, score, None
        start, end = al.dest_start, al.dest_end
        if end - start < 0.8 * len(q):               # aligned span must cover (almost) the whole quote
            return False, score, None
    span = e[start:end]
    if _negs(q) != _negs(span):                      # flipped, added or removed negation inside the aligned words
        return False, -score, None
    # a negation cut off right before the quote ('will not | migrate production') or right after it
    before = e[:start].split(); after = e[end:].split()[:1]
    pre = before[-1:] if before else []
    if pre and pre[0] in _AUX and len(before) > 1:   # 'not even migrate', 'not really ready'
        pre.append(before[-2])
    if any(w in _NEGATION_WORDS or w.endswith("n't") for w in pre + after):
        return False, -score, None
    return True, score, (start, end)

def _content(s: str) -> list[str]:
    return [w for w in _words(s) if w not in _STOP and len(w) > 1]

def _word_match(a: str, b: str) -> bool:
    return a == b or (len(a) >= 4 and len(b) >= 4 and (a.startswith(b[:5]) or b.startswith(a[:5]))) or fuzz.ratio(a, b) >= 85

_NLI = None
def _nli_entails(premise: str, hypothesis: str, cfg: DocConfig | None = None) -> float | None:
    "Optional cross-encoder entailment probability (quote -> statement). None if disabled/unavailable."
    global _NLI
    cfg = cfg or CFG
    if not cfg.use_nli:
        return None
    if _NLI is None:
        try:
            from sentence_transformers import CrossEncoder
            _NLI = CrossEncoder(cfg.nli_model)
        except Exception as e:
            print("NLI unavailable ->", type(e).__name__); _NLI = False
    if not _NLI:
        return None
    try:
        import numpy as np
        scores = _NLI.predict([(premise, hypothesis)])[0]      # [contradiction, entailment, neutral]
        ex = np.exp(scores - scores.max()); return float((ex / ex.sum())[1])
    except Exception:
        return None

def statement_grounded(statement: str, quote: str, cfg: DocConfig) -> tuple[bool, str]:
    """The statement (free text written by the model) must be supported by its quote: enough shared content words,
    no polarity inversion ('Migrate production' from 'we will not migrate production'), optional NLI entailment."""
    sw, qw = _content(statement), _content(quote)
    if not sw:
        return False, "empty statement"
    hits = sum(any(_word_match(a, b) for b in qw) for a in sw)
    overlap = hits / len(sw)
    if overlap < cfg.statement_overlap_min:
        return False, f"statement not supported by its quote (overlap {overlap:.2f})"
    q_words = _words(quote)
    negated = set()
    for i, w in enumerate(q_words):
        if w in _NEGATION_WORDS or w.endswith("n't"):
            negated.update(x for x in q_words[i + 1:i + 4] if x not in _STOP)
    s_words = set(_words(statement))
    s_negated = bool(_negs(statement)) or bool(s_words & _NEGATION_EQUIV)
    if negated and not s_negated and any(_word_match(a, b) for a in s_words for b in negated):
        return False, "statement drops the negation in its quote"
    ent = _nli_entails(quote, statement, cfg)
    if ent is not None and ent < 0.5:
        return False, f"quote does not entail statement (NLI {ent:.2f})"
    return True, f"overlap {overlap:.2f}" + (f", NLI {ent:.2f}" if ent is not None else "")

def _locate_quote_lines(quote: str, ids: list[int], lines: dict, cfg: DocConfig, max_lines: int) -> tuple[list[int], int, str]:
    """Find the quote in 1..max_lines consecutive lines. First among the cited ids; then, because a model often cites
    only one of two lines a sentence was split across, among the cited ids plus `evidence_neighbour_lines` lines on
    either side. The quote must still be found verbatim (same negation rules), so this never weakens the check.
    Returns (line ids, score, evidence text)."""
    cited = sorted(i for i in set(ids) if i in lines)
    pools = [cited]
    n = max(0, int(getattr(cfg, "evidence_neighbour_lines", 0) or 0))
    if n and cited:
        wide = sorted({j for i in cited for j in range(i - n, i + n + 1) if j in lines})
        if wide != cited:
            pools.append(wide)
    for pool in pools:
        for width in range(1, max_lines + 1):
            for k in range(len(pool) - width + 1):
                sub = pool[k:k + width]
                if sub != list(range(sub[0], sub[0] + width)):
                    continue
                if pool is not cited and not (set(sub) & set(cited)):
                    continue                              # a neighbour window must still contain a cited line
                ev = evidence_text(sub, lines); ok, score, _ = quote_found(quote, ev, cfg)
                if ok:
                    return sub, score, ev
    return [], 0, ""

def _near_quote(term: str, quote: str, ev_text: str, cfg: DocConfig, window: int) -> bool:
    "term occurs inside the quote, or within `window` words of the quote inside the evidence text."
    t = _norm(term)
    if not t or t == "unspecified":
        return False
    def _in(text):
        toks = _words(text); n = len(t.split())
        return any(fuzz.ratio(t, " ".join(toks[i:i + n])) >= cfg.name_min_ratio for i in range(max(1, len(toks) - n + 1)))
    if _in(quote):
        return True
    same = _same_sentence_windows(quote, ev_text, window)
    if same is not None:                             # the quote was located word-for-word: stay inside its sentence
        return _in(same[0]) or _in(same[1])
    q, e = _norm(quote), _norm(ev_text)
    pos = e.find(q)
    if pos < 0:
        al = fuzz.partial_ratio_alignment(q, e); pos = al.dest_start if al else 0; end = al.dest_end if al else len(e)
    else:
        end = pos + len(q)
    before = " ".join(e[:pos].split()[-window:]); after = " ".join(e[end:].split()[:window])
    return _in(before) or _in(after)

def _same_sentence_windows(quote: str, ev_text: str, window: int):
    """(words before, words after) the quote, at most `window` each, never crossing a sentence end ('. ! ?').
    'Someone needs to write the runbook. Rahul will send ...' -> Rahul is NOT next to the runbook quote.
    Returns None when the quote is not found word-for-word (the caller then uses the fuzzy alignment)."""
    raw = str(ev_text).split()
    nw = [_norm(w) for w in raw]
    qw = _words(quote)
    if not qw:
        return None
    idx = [i for i, w in enumerate(nw) if w]
    flat = [nw[i] for i in idx]
    for k in range(len(flat) - len(qw) + 1):
        if flat[k:k + len(qw)] == qw:
            a, b = idx[k], idx[k + len(qw) - 1]
            after = []
            if not re.search(r"[.!?][\"')\]]*$", raw[b]):
                for w in raw[b + 1:]:
                    after.append(w)
                    if len(after) >= window or re.search(r"[.!?][\"')\]]*$", w):
                        break
            before = []
            for w in reversed(raw[:a]):
                if re.search(r"[.!?][\"')\]]*$", w) or len(before) >= window:
                    break
                before.insert(0, w)
            return " ".join(before), " ".join(after)
    return None

def _deadline_cued(deadline: str, ev_text: str) -> bool:
    "A deadline must be introduced as one ('by Friday', 'this quarter', 'before the demo'), not just be a date in the text."
    d, e = _words(deadline), _words(ev_text)
    if not d:
        return False
    if d[0] in _DEADLINE_CUE:
        return True
    for i in range(len(e) - len(d) + 1):
        if e[i:i + len(d)] == d and any(w in _DEADLINE_CUE for w in e[max(0, i - 2):i]):
            return True
    return False

def verify_candidate(c: dict, lines: dict, cfg: DocConfig) -> tuple[dict | None, str]:
    ids = _clean_ids(c.get("evidence_lines"))
    if not ids or not any(i in lines for i in ids):
        return None, "evidence lines missing/unknown"
    quote = str(c.get("evidence_quote", "")).strip()
    found, score, ev = _locate_quote_lines(quote, ids, lines, cfg, max_lines=3)
    if not found:
        ok, s, _ = quote_found(quote, evidence_text(ids, lines), cfg)
        return None, ("quote changes a negation vs the evidence" if s < 0 else f"quote not found in evidence (ratio {abs(s)})")
    statement = str(c.get("statement", "")).strip()
    ok, why = statement_grounded(statement, quote, cfg)
    if not ok:
        return None, why
    t = str(c.get("type", "")).strip().lower()
    if t not in ("agreed_decision", "proposal", "open_question", "tentative"):
        return None, f"unknown type {t!r}"
    # safety net: a "decision" whose own evidence says it was not agreed is a proposal
    nq = _norm(quote); m = None
    for m in _NEG_ACCEPT_RE.finditer(nq): pass
    if t == "agreed_decision" and m and not _MARKER_RE.search(nq[m.end():]):
        t = "proposal"                                # 'but nobody agreed' with nothing decided after it
    marker = bool(_MARKER_RE.search(_norm(ev)))
    return {"statement": statement, "type": t, "evidence_quote": quote, "evidence_lines": found,
            "quote_match": score, "marker_found": marker, "grounding": why}, "ok"

def verify_action(a: dict, lines: dict, cfg: DocConfig) -> tuple[dict | None, str, list[str]]:
    notes = []
    ids = _clean_ids(a.get("evidence_lines"))
    if not ids or not any(i in lines for i in ids):
        return None, "evidence lines missing/unknown", notes
    quote = str(a.get("evidence_quote", "")).strip()
    found, score, ev = _locate_quote_lines(quote, ids, lines, cfg, max_lines=cfg.max_action_evidence_lines)
    if not found:
        ok, s, _ = quote_found(quote, evidence_text(ids, lines), cfg)
        why = ("quote changes a negation vs the evidence" if s < 0 else
               f"quote not found within {cfg.max_action_evidence_lines} consecutive evidence lines (ratio {abs(s)})")
        return None, why, notes
    task = str(a.get("task", "")).strip()
    if not task:
        return None, "empty task", notes
    ok, why = statement_grounded(task, quote, cfg)
    if not ok:
        return None, "task " + why, notes
    nq = _norm(quote)
    owner = str(a.get("owner", "unspecified")).strip() or "unspecified"
    if owner.lower() != "unspecified":
        if not _near_quote(owner, quote, ev, cfg, cfg.owner_window_words):
            notes.append(f"owner '{owner}' not in or next to the evidence quote -> unspecified"); owner = "unspecified"
        elif _FIRST_PERSON_RE.search(nq) and not _near_quote(owner, quote, "", cfg, 0):
            notes.append(f"first-person commitment; '{owner}' is not named in the quote -> unspecified"); owner = "unspecified"
        elif not _COMMIT_RE.search(nq):
            notes.append(f"no commitment wording near '{owner}' -> unspecified"); owner = "unspecified"
    deadline = str(a.get("deadline", "unspecified")).strip() or "unspecified"
    if deadline.lower() != "unspecified":
        if not _near_quote(deadline, quote, ev, _dc_replace(cfg, name_min_ratio=cfg.deadline_min_ratio), cfg.owner_window_words):
            notes.append(f"deadline '{deadline}' not in or next to the evidence quote -> unspecified"); deadline = "unspecified"
        elif not _deadline_cued(deadline, ev):
            notes.append(f"'{deadline}' is a time mention, not a stated deadline -> unspecified"); deadline = "unspecified"
    if owner.lower() == "unspecified": owner = "unspecified"
    if deadline.lower() == "unspecified": deadline = "unspecified"
    return {"task": task, "owner": owner, "owner_stated": owner != "unspecified",
            "deadline": deadline, "deadline_stated": deadline != "unspecified",
            "evidence_quote": quote, "evidence_lines": found, "quote_match": score,
            "commitment_marker": bool(_COMMIT_RE.search(nq))}, "ok", notes

# ---- entity screen for free text (summary, discussion points): names, numbers, dates must exist in the transcript ----
_ONES = "zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen sixteen seventeen eighteen nineteen".split()
_TENS = {"twenty": 20, "thirty": 30, "forty": 40, "fifty": 50, "sixty": 60, "seventy": 70, "eighty": 80, "ninety": 90}
_ORD = {"first": 1, "second": 2, "third": 3, "fourth": 4, "fifth": 5, "sixth": 6, "seventh": 7, "eighth": 8, "ninth": 9, "tenth": 10,
        "eleventh": 11, "twelfth": 12, "thirteenth": 13, "fourteenth": 14, "fifteenth": 15, "sixteenth": 16, "seventeenth": 17,
        "eighteenth": 18, "nineteenth": 19, "twentieth": 20, "thirtieth": 30}
_MONTHS_DAYS = set("january february march april may june july august september october november december monday tuesday "
                   "wednesday thursday friday saturday sunday".split())

def _numbers_in(text: str) -> set[int]:
    "Integers mentioned as digits (23, 23rd, 1,000) or as words (twenty third, fifteen, fifty)."
    out = set()
    for m in re.finditer(r"\d[\d,]*", text):
        try: out.add(int(m.group().replace(",", "")))
        except ValueError: pass
    ws = _words(text); i = 0
    while i < len(ws):
        w = ws[i]; val = None
        if w in _TENS:
            val = _TENS[w]
            if i + 1 < len(ws) and (ws[i + 1] in _ONES[1:10] or ws[i + 1] in _ORD and _ORD[ws[i + 1]] < 10):
                val += _ONES.index(ws[i + 1]) if ws[i + 1] in _ONES else _ORD[ws[i + 1]]; i += 1
        elif w in _ONES: val = _ONES.index(w)
        elif w in _ORD: val = _ORD[w]
        elif w == "hundred": val = 100
        elif w == "thousand": val = 1000
        if val is not None: out.add(val)
        i += 1
    return out

try:
    from wordfreq import zipf_frequency as _zipf
except Exception:
    _zipf = None

def _rare(word: str, thr: float = 3.5) -> bool:
    "Not ordinary English (wordfreq zipf < thr). Without wordfreq nothing counts as rare (old behaviour)."
    return bool(_zipf) and _zipf(word, "en") < thr

def _entities_in(text: str) -> set[str]:
    """Capitalised tokens (names, products) plus month/weekday names. A sentence-initial capitalised word counts only
    if it is not ordinary English ('Sanjay will ...' is screened, 'The team ...' is not)."""
    ents = set()
    for sent in re.split(r"(?<=[.!?])\s+", text):
        toks = re.findall(r"[A-Za-z][A-Za-z0-9'/.-]*", sent)
        for k, t in enumerate(toks):
            tl = t.lower().strip(".")
            if tl in _MONTHS_DAYS or (t[0].isupper() and len(t) > 1 and tl not in _STOP and (k > 0 or _rare(tl))):
                ents.add(tl)
    return ents

def entity_screen(text: str, transcript_norm: str, transcript_numbers: set[int], transcript_ents: set[str]) -> tuple[str, list[str]]:
    "Drop any sentence whose names/terms/numbers do not occur in the transcript. Returns (kept text, reasons)."
    kept, dropped = [], []
    for sent in re.split(r"(?<=[.!?])\s+", str(text).strip()):
        if not sent.strip():
            continue
        bad = [str(n) for n in _numbers_in(sent) if n not in transcript_numbers]
        bad += [e for e in _entities_in(sent) if e not in transcript_ents and e not in transcript_norm.split()]
        (dropped if bad else kept).append(f"{sent.strip()}  [not in transcript: {', '.join(bad)}]" if bad else sent.strip())
    return " ".join(kept), dropped

def verify_output(obj: dict, lines: dict, cfg: DocConfig) -> dict:
    log = {"dropped": [], "adjusted": [], "screened": []}
    transcript = " ".join(lines[i] for i in sorted(lines))
    t_norm, t_nums, t_ents = _norm(transcript), _numbers_in(transcript), _entities_in(transcript)
    cands, actions = [], []
    for c in obj.get("candidates", []) or []:
        if not isinstance(c, dict): continue
        v, why = verify_candidate(c, lines, cfg)
        (cands.append(v) if v else log["dropped"].append({"kind": "candidate", "item": c, "reason": why}))
    for a in obj.get("action_items", []) or []:
        if not isinstance(a, dict): continue
        v, why, notes = verify_action(a, lines, cfg)
        if v:
            actions.append(v); log["adjusted"] += [{"task": v["task"], "note": n} for n in notes]
        else:
            log["dropped"].append({"kind": "action_item", "item": a, "reason": why})
    summary, s_dropped = entity_screen(str(obj.get("summary", "")), t_norm, t_nums, t_ents)
    log["screened"] += [{"kind": "summary", "text": d} for d in s_dropped]
    points = []
    for d in (obj.get("discussion_points") or []):
        if not isinstance(d, dict): continue
        ids = [i for i in _clean_ids(d.get("evidence_lines")) if i in lines]
        ev = evidence_text(ids, lines) if ids else transcript
        kept = []
        for p in d.get("points", []) or []:
            p = str(p).strip()
            if not p: continue
            txt, bad = entity_screen(p, t_norm, t_nums, t_ents)
            if bad:
                log["screened"].append({"kind": "point", "topic": d.get("topic"), "text": bad[0]}); continue
            cw = _content(p); ew = _content(ev)
            if cw and sum(any(_word_match(a, b) for b in ew) for a in cw) / len(cw) < cfg.statement_overlap_min:
                log["screened"].append({"kind": "point", "topic": d.get("topic"), "text": f"{p}  [not supported by lines {ids or 'transcript'}]"}); continue
            kept.append(p)
        if kept:
            points.append({"topic": str(d.get("topic", "")).strip(), "points": kept, "evidence_lines": ids})
    return {"summary": summary,
            "discussion_points": points,
            "decisions": [c for c in cands if c["type"] == "agreed_decision"],
            "proposals_not_adopted": [c for c in cands if c["type"] == "proposal"],
            "tentative": [c for c in cands if c["type"] == "tentative"],
            "open_questions": [c for c in cands if c["type"] == "open_question"],
            "action_items": actions, "verification": log}

SAVE_4BIT_TO_DRIVE = None          # legacy switch; None = use cfg.save_4bit_copy

def load_llm(cfg):
    if torch is None or AutoModelForCausalLM is None:
        raise ImportError("torch/transformers are not importable in this runtime "
                          f"({type(globals().get('_TORCH_ERR') or globals().get('_TF_ERR')).__name__})")
    t0 = time.time()
    local = MODELS_DIR / (cfg.model_id.replace("/", "__") + ("-4bit" if cfg.load_in_4bit else ""))
    source = "hub"
    if (local / "config.json").exists():
        try:
            print("Loading cached copy:", local)
            tok = AutoTokenizer.from_pretrained(local)
            model = AutoModelForCausalLM.from_pretrained(local, device_map="auto")
            source = "cache"
        except Exception as e:
            print("Cached copy unusable ->", type(e).__name__, str(e)[:120], "| downloading again")
    if source == "hub":
        print("Downloading from Hugging Face:", cfg.model_id, "(one-time, ~15 GB)")
        tok = AutoTokenizer.from_pretrained(cfg.model_id)
        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                   bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True) if cfg.load_in_4bit else None
        model = AutoModelForCausalLM.from_pretrained(cfg.model_id, quantization_config=quant,
                                                     device_map="auto", **compat.dtype_kwarg(torch.float16))
    model.eval()
    save = cfg.save_4bit_copy if SAVE_4BIT_TO_DRIVE is None else SAVE_4BIT_TO_DRIVE
    if source == "hub" and save and cfg.load_in_4bit:
        try:
            print("Saving 4-bit copy (one-time, ~5.5 GB, a few minutes) ->", local)
            tmp = local.with_name(local.name + ".partial")
            model.save_pretrained(tmp, safe_serialization=True); tok.save_pretrained(tmp)
            if local.exists():
                import shutil; shutil.rmtree(local, ignore_errors=True)
            tmp.rename(local)
            print("Saved ->", local)
        except Exception as e:
            print("Could not save the 4-bit copy (continuing):", type(e).__name__, str(e)[:120])
    mem = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
    print(f"Loaded {cfg.model_id} from {source} in {time.time()-t0:.0f}s | GPU mem {mem:.1f} GB")
    return tok, model

tokenizer = llm = None

def ensure_loaded(cfg: DocConfig, models_dir: Path | None = None) -> bool:
    global tokenizer, llm, MODELS_DIR
    if models_dir: MODELS_DIR = Path(models_dir)
    if llm is not None:
        return True
    try:
        tokenizer, llm = load_llm(cfg)
    except Exception as e:
        print("WARNING:", DocumentError(f"Documenter model could not be loaded ({type(e).__name__}: {str(e)[:160]}). "
                                        "Minutes, decisions and action items cannot be generated for this run.").user_message)
        tokenizer = llm = None
    return llm is not None

def free_llm():
    global tokenizer, llm
    tokenizer = llm = None
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass

def llm_generate(system: str, user: str, cfg, max_new_tokens: int = None) -> str:
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    kw = {"enable_thinking": False} if "qwen3" in cfg.model_id.lower() else {}
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **kw)
    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens or cfg.max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

EMPTY = {"summary": "", "discussion_points": [], "candidates": [], "action_items": []}

def _loads_lenient(text: str):
    "json.loads with a repair fallback (truncated output, trailing commas, single quotes)."
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        try:
            from json_repair import repair_json
            return json.loads(repair_json(text))
        except Exception:
            return None

def parse_json_obj(text: str) -> dict | None:
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        m = re.search(r"\{.*", text, flags=re.S)          # truncated object: let the repair step close it
    if not m:
        return None
    obj = _loads_lenient(m.group(0))
    return obj if isinstance(obj, dict) else None

def document_chunk(chunk: list[dict], cfg: DocConfig, part: str = "", glossary: list[str] | None = None) -> tuple[dict, str, dict]:
    user = build_user_message(chunk, cfg.meeting_context, part, glossary)
    try:
        raw = llm_generate(SYSTEM_PROMPT, user, cfg)
    except Exception as e:                             # OOM, timeout, missing model: this chunk contributes nothing
        return dict(EMPTY), "", {"attempts": 0, "warning": f"LLM call failed ({type(e).__name__}: {str(e)[:100]}); chunk contributed nothing"}
    obj = parse_json_obj(raw); attempts = 1
    if obj is None:
        try:
            raw = llm_generate(SYSTEM_PROMPT, user + "\n\nYour previous reply was not valid JSON. Reply with ONLY the JSON object.", cfg)
            obj = parse_json_obj(raw)
        except Exception:
            obj = None
        attempts = 2
    warn = None
    if obj is None:
        obj, warn = dict(EMPTY), "LLM returned invalid JSON twice; chunk contributed nothing"
    return obj, raw, {"attempts": attempts, "warning": warn}

def _same_item(a: dict, b: dict, key: str, cfg: DocConfig) -> bool:
    return (set(a["evidence_lines"]) & set(b["evidence_lines"]) and fuzz.token_set_ratio(_norm(a[key]), _norm(b[key])) >= 70) \
        or fuzz.token_set_ratio(_norm(a[key]), _norm(b[key])) >= cfg.dedupe_ratio

def _dedupe(items: list[dict], key: str, cfg: DocConfig) -> list[dict]:
    "Fuzzy de-duplication across overlapping chunks (same statement cited with slightly different lines)."
    out = []
    for it in items:
        if not any(_same_item(it, o, key, cfg) for o in out):
            out.append(it)
    return out

def resolve_conflicts(cands: list[dict], cfg: DocConfig, log: dict) -> list[dict]:
    """Across chunks the same topic can surface twice with a different type or polarity ('go with X' in chunk 1,
    'scrap X' in chunk 2). The item whose evidence is LATER in the meeting wins; the loser is logged as superseded."""
    kept = []
    for c in sorted(cands, key=lambda x: max(x["evidence_lines"])):
        clash = next((k for k in kept if fuzz.token_set_ratio(_norm(k["statement"]), _norm(c["statement"])) >= cfg.dedupe_ratio
                      and (k["type"] != c["type"] or bool(_negs(k["statement"])) != bool(_negs(c["statement"])))), None)
        if clash:
            log["superseded"].append({"earlier": clash["statement"], "later": c["statement"], "lines": c["evidence_lines"]})
            kept.remove(clash)
        kept.append(c)
    return kept

def run_stage3(segments: list[dict], cfg: DocConfig, on_progress=None, glossary: list[str] | None = None) -> dict:
    t0 = time.time(); lines = {s["id"]: s["text"] for s in segments}
    chunks = make_chunks(segments, cfg.max_chunk_words, cfg.chunk_overlap_lines)
    parts, warnings, raw_outputs, status = [], [], [], "ok"
    if llm is None:
        warnings.append("documenter model not loaded; record contains empty lists"); status = "failed"; chunks = []
    for ci, chunk in enumerate(chunks):
        part = f"(Part {ci+1} of {len(chunks)} of the meeting.)\n" if len(chunks) > 1 else ""
        obj, raw, info = document_chunk(chunk, cfg, part, glossary); raw_outputs.append(raw)
        if info["warning"]:
            warnings.append(f"chunk {ci+1}: {info['warning']}"); status = "partial" if status == "ok" else status
        parts.append(verify_output(obj, lines, cfg))
        if on_progress: on_progress("documenting", (ci + 1) / len(chunks))
    if chunks and all(w.startswith("chunk") for w in warnings) and len(warnings) == len(chunks):
        status = "failed"
    # ---- reduce ----
    log = {"dropped": [x for p in parts for x in p["verification"]["dropped"]],
           "adjusted": [x for p in parts for x in p["verification"]["adjusted"]],
           "screened": [x for p in parts for x in p["verification"]["screened"]], "superseded": []}
    all_cands = [c for p in parts for k in ("decisions", "proposals_not_adopted", "tentative", "open_questions") for c in p[k]]
    all_cands = resolve_conflicts(_dedupe(all_cands, "statement", cfg), cfg, log)
    topics = {}
    for p in parts:
        for d in p["discussion_points"]:
            t = topics.setdefault(_norm(d["topic"]), {"topic": d["topic"], "points": [], "evidence_lines": []})
            t["points"] += [x for x in d["points"] if x not in t["points"]]
            t["evidence_lines"] = sorted(set(t["evidence_lines"]) | set(d["evidence_lines"]))
    merged = {
        "summary": parts[0]["summary"] if len(parts) == 1 else "",
        "discussion_points": sorted(topics.values(), key=lambda t: min(t["evidence_lines"]) if t["evidence_lines"] else 10**6),
        "decisions": [c for c in all_cands if c["type"] == "agreed_decision"],
        "proposals_not_adopted": [c for c in all_cands if c["type"] == "proposal"],
        "tentative": [c for c in all_cands if c["type"] == "tentative"],
        "open_questions": [c for c in all_cands if c["type"] == "open_question"],
        "action_items": _dedupe([a for p in parts for a in p["action_items"]], "task", cfg),
        "verification": log,
    }
    if len(parts) > 1:                       # one short call to write the overall summary from the part summaries
        joined = "\n".join(f"Part {i+1}: {p['summary']}" for i, p in enumerate(parts) if p["summary"])
        try:
            raw = llm_generate("You write a 3-5 sentence meeting summary from part summaries. Use only what is given. Return plain text.",
                               joined, cfg, max_new_tokens=400)
            transcript = " ".join(lines[i] for i in sorted(lines))
            merged["summary"], scr = entity_screen(raw.strip(), _norm(transcript), _numbers_in(transcript), _entities_in(transcript))
            log["screened"] += [{"kind": "summary", "text": d} for d in scr]
        except Exception as e:
            warnings.append(f"summary reduce call failed ({type(e).__name__}); using part summaries")
            merged["summary"] = " ".join(p["summary"] for p in parts if p["summary"])
    merged["warnings"] = warnings; merged["status"] = status
    merged["meta"] = {"documenter": f"{cfg.model_id} (4-bit NF4)" if cfg.load_in_4bit else cfg.model_id,
                      "n_chunks": len(chunks), "settings": vars(cfg), "elapsed_s": round(time.time() - t0, 1),
                      "n_dropped": len(log["dropped"]), "n_adjusted": len(log["adjusted"]),
                      "n_screened": len(log["screened"]), "n_superseded": len(log["superseded"])}
    merged["_raw_outputs"] = raw_outputs
    return merged

def _md(s) -> str:
    "Escape characters that would break a Markdown table cell."
    return str(s).replace("|", "\\|").replace("\n", " ")

def _line_info(rec: dict) -> dict:
    rt = rec.get("refined_transcript") if isinstance(rec.get("refined_transcript"), dict) else {}
    return {s["id"]: {"start": s.get("start"), "speaker": s.get("speaker")} for s in rt.get("segments", []) or []}

def speaker_names(rec: dict) -> dict:
    "User-supplied display names for diarization labels (record['speaker_names'] = {'SPEAKER_00': 'Priya', ...})."
    return {k: v for k, v in (rec.get("speaker_names") or {}).items() if v}

def _spk(label, names: dict) -> str:
    return names.get(label, label) if label else ""

def _ev(ids: list[int], info: dict, names: dict | None = None) -> str:
    "L008 (00:32, SPEAKER_01) — line ids with timestamp and speaker so a reader can seek the audio."
    names = names or {}
    out = []
    for i in ids:
        d = info.get(i, {}); bits = []
        if d.get("start") is not None: bits.append(fmt_ts(d["start"]))
        if d.get("speaker") and d["speaker"] != DIAR_CFG.unknown_label: bits.append(_spk(d["speaker"], names))
        out.append(f"L{i:03d}" + (f" ({', '.join(bits)})" if bits else ""))
    return ", ".join(out)

def render_markdown(rec: dict) -> str:
    meta, mins = rec["meta"], rec["minutes"]
    info = _line_info(rec); names = speaker_names(rec)
    models = meta.get("models", {})
    diar = rec.get("diarization") or {}
    L = [f"# Meeting record — {meta.get('audio_file', 'recording')}", "",
         f"*Duration:* {meta.get('duration_s', '?')} s · *STT:* {models.get('stt')} · *Speakers:* "
         f"{(str(diar.get('n_speakers')) + ' (' + str(models.get('diarization')) + ')') if diar.get('status') == 'ok' else 'not diarized'} · "
         f"*Cross-check:* {models.get('asr2') or 'off'} · *Refiner:* {models.get('refiner')} · *Documenter:* {models.get('documenter')}", ""]
    cal = rec.get("calendar") or {}
    if cal.get("meeting_date"):
        L += [f"*Meeting date:* {cal['meeting_date']} ({cal.get('meeting_date_source', '')})", ""]
    if rec.get("status") and rec["status"] != "ok":
        L += [f"> **Status: {rec['status']}.** " + " ".join(meta.get("documenter_warnings", [])), ""]
    L += ["## Summary", "", mins["summary"] or "_No summary produced._", "", "## Minutes", ""]
    for d in mins["discussion_points"]:
        L.append(f"**{d['topic']}**" + (f" _({_ev(d.get('evidence_lines', []), info, names)})_" if d.get("evidence_lines") else ""))
        L += [f"- {p}" for p in d["points"]]; L.append("")
    if not mins["discussion_points"]: L += ["_No discussion points recorded._", ""]
    L += ["## Key decisions", ""]
    L += [f"{i+1}. {d['decision']}" + ("" if d.get("explicit_decision_wording", True) else " _(weak evidence — no explicit decision wording nearby)_") +
          f"  \n   _Evidence ({_ev(d['evidence_lines'], info, names)}):_ \"{d['evidence_quote']}\""
          for i, d in enumerate(rec["decisions"])] or ["_No decisions were reached in this meeting._"]
    L.append("")
    if rec.get("proposals_not_adopted"):
        L += ["## Proposals discussed but not adopted", ""]
        L += [f"- {p['statement']} _({_ev(p['evidence_lines'], info, names)})_" for p in rec["proposals_not_adopted"]] + [""]
    if rec.get("tentative"):
        L += ["## Tentative — needs confirmation", ""]
        L += [f"- {p['statement']} _({_ev(p['evidence_lines'], info, names)})_" for p in rec["tentative"]] + [""]
    if rec.get("open_questions"):
        L += ["## Open questions", ""] + [f"- {q['statement']} _({_ev(q['evidence_lines'], info, names)})_" for q in rec["open_questions"]] + [""]
    L += ["## Action items", ""]
    if rec["action_items"]:
        L += ["| # | Task | Owner | Said by | Deadline (as spoken) | Due date | Evidence |", "|---|---|---|---|---|---|---|"]
        for i, a in enumerate(rec["action_items"]):
            due = (a.get("due") or {}).get("date") or "—"
            said = ", ".join(_spk(x, names) for x in a.get("speakers", []) if x != DIAR_CFG.unknown_label) or "—"
            L.append(f"| {i+1} | {_md(a['task'])} | {_md(a['owner'])} | {_md(said)} | {_md(a['deadline'])} | {_md(due)} | "
                     f"{_ev(a['evidence_lines'], info, names)} |")
    else:
        L.append("_No action items were assigned._")
    v = rec["meta"].get("documenter_verification", {})
    L += ["", "---", f"_Owners and deadlines are shown as **unspecified** when not stated in the recording. "
          f"'Said by' is the diarized speaker of the evidence line, not an inferred owner. Due dates are computed from the "
          f"spoken deadline and the meeting date. Verifier: {v.get('dropped', 0)} item(s) dropped, {v.get('adjusted', 0)} "
          f"field(s) set to unspecified, {v.get('screened', 0)} unsupported sentence(s) removed from the summary/minutes._"]
    return "\n".join(L)

def assemble_record(s2: dict, r3: dict) -> dict:
    "meeting_record.json = the Stage 2 record + minutes, decisions, proposals, tentative items, questions, action items."
    rec = copy.deepcopy(s2["record"]) if isinstance(s2["record"], dict) else {}
    rec["schema_version"] = rec.get("schema_version", "1.1")
    rec.setdefault("meta", {}).setdefault("models", {})["documenter"] = r3["meta"]["documenter"]
    rec["meta"]["documenter_settings"] = r3["meta"]["settings"]
    rec["meta"]["documenter_warnings"] = r3["warnings"]
    rec["meta"]["documenter_verification"] = {k: r3["meta"][f"n_{k}"] for k in ("dropped", "adjusted", "screened", "superseded")}
    rec["status"] = r3["status"]
    if not isinstance(rec.get("refined_transcript"), dict):
        rec["refined_transcript"] = {"status": "unknown", "text": " ".join(s["text"] for s in s2["segments"]),
                                     "segments": s2["segments"]}
    info = _line_info(rec) or {s["id"]: {"start": s.get("start"), "speaker": s.get("speaker")} for s in s2["segments"]}
    def _ev(ids): return [{"line": i, "start": info.get(i, {}).get("start"), "speaker": info.get(i, {}).get("speaker")} for i in ids]
    def _spks(ids): return sorted({info.get(i, {}).get("speaker") for i in ids if info.get(i, {}).get("speaker")})
    rec["minutes"] = {"summary": r3["summary"], "discussion_points": r3["discussion_points"]}
    rec["decisions"] = [{"decision": d["statement"], "evidence_quote": d["evidence_quote"], "evidence_lines": d["evidence_lines"],
                         "evidence": _ev(d["evidence_lines"]), "speakers": _spks(d["evidence_lines"]),
                         "explicit_decision_wording": d.get("marker_found", True)} for d in r3["decisions"]]
    rec["proposals_not_adopted"] = [{"statement": p["statement"], "type": p["type"], "evidence_quote": p["evidence_quote"],
                                     "evidence_lines": p["evidence_lines"], "evidence": _ev(p["evidence_lines"]),
                                     "speakers": _spks(p["evidence_lines"])} for p in r3["proposals_not_adopted"]]
    rec["tentative"] = [{"statement": p["statement"], "evidence_quote": p["evidence_quote"], "evidence_lines": p["evidence_lines"],
                         "evidence": _ev(p["evidence_lines"]), "speakers": _spks(p["evidence_lines"])} for p in r3["tentative"]]
    rec["open_questions"] = [{"statement": q["statement"], "evidence_quote": q.get("evidence_quote", ""),
                              "evidence_lines": q["evidence_lines"], "evidence": _ev(q["evidence_lines"]),
                              "speakers": _spks(q["evidence_lines"])} for q in r3["open_questions"]]
    rec["action_items"] = [{"task": a["task"], "owner": a["owner"], "deadline": a["deadline"],
                            "owner_stated": a["owner_stated"], "deadline_stated": a["deadline_stated"],
                            "evidence_quote": a["evidence_quote"], "evidence_lines": a["evidence_lines"], "evidence": _ev(a["evidence_lines"]),
                            "speakers": _spks(a["evidence_lines"])}
                           for a in r3["action_items"]]
    rec.setdefault("speaker_names", {})
    return rec

def save_stage3(s2: dict, r3: dict, out_dir: Path, run_dir: Path | None = None) -> tuple[Path, dict]:
    run_dir = Path(run_dir) if run_dir else out_dir / f"stage3_{datetime.now():%Y%m%d_%H%M%S}"; run_dir.mkdir(parents=True, exist_ok=True)
    rec = assemble_record(s2, r3)
    (run_dir / "meeting_record.json").write_text(json.dumps(rec, indent=2, ensure_ascii=False), encoding="utf-8")
    (run_dir / "meeting_record.md").write_text(render_markdown(rec), encoding="utf-8")
    (run_dir / "verification_log.json").write_text(json.dumps(
        {**r3["verification"], "warnings": r3["warnings"], "status": r3["status"],
         "meta": r3["meta"], "raw_model_outputs": r3["_raw_outputs"]}, indent=2, ensure_ascii=False), encoding="utf-8")
    # the PS asks for raw + refined as separate downloads too; copy them beside the record when available
    for key, name in (("raw_transcript", "raw_transcript.txt"), ("refined_transcript", "refined_transcript.txt")):
        t = rec.get(key); t = t.get("text") if isinstance(t, dict) else t
        if isinstance(t, str) and t.strip() and not (run_dir / name).exists(): (run_dir / name).write_text(t, encoding="utf-8")
    print("Saved to", run_dir)
    for f in sorted(run_dir.iterdir()): print(f"  {f.name:<26} {f.stat().st_size/1024:7.1f} KB")
    return run_dir, rec


# ----------------------------------------------------------------------------- entry points
def document(stage2_record, cfg: DocConfig | None = None, models_dir: Path | None = None, on_progress=None) -> tuple[dict, dict]:
    """Stage 3 end to end: stage2_record.json (path or dict) -> (R3, S2). Never raises for model problems:
    if the documenter cannot run, R3['status'] == 'failed' and all lists are empty."""
    cfg = cfg or DocConfig()
    if isinstance(stage2_record, (str, Path)):
        s2 = load_stage2(str(stage2_record))
    else:
        import tempfile
        tmp = Path(tempfile.mkdtemp()) / "stage2_record.json"; tmp.write_text(json.dumps(stage2_record)); s2 = load_stage2(str(tmp))
    if not cfg.meeting_context:
        cfg.meeting_context = (s2["domain"] or {}).get("domain") or s2["meta"].get("meeting_context") or "business meeting"
    ensure_loaded(cfg, models_dir)
    r3 = run_stage3(s2["segments"], cfg, on_progress=on_progress, glossary=list(s2["glossary"]) + list(s2["people"]))
    return r3, s2

def self_test() -> int:
    "Verifier self-test (no model). Returns the number of failed checks."
    _lines = {1: "Okay everyone, let's start the weekly sync.",
              2: "First, the migration from MySQL to PostgreSQL.",
              3: "Priya says the staging database is ready, but we will not migrate production this quarter,",
              4: "because the load tests failed twice.",
              7: "Rahul suggested moving the API gateway to AWS, but nobody agreed, so we are keeping the current setup.",
              8: "Third, the launch. We decided the release date is the twenty third of October.",
              9: "Priya will update the Terraform scripts by Friday.",
              10: "Someone needs to write the rollback runbook.",
              11: "Rahul will send the benchmark report. That's all, thanks everyone.",
              12: "The outage on the fifth sparked a long discussion about alerting.",
              13: "I'll send the benchmark numbers by end of the month.",
              20: "Okay, and Rahul will send",
              21: "the latency dashboard by Monday. Thanks."}
    _out = {"summary": "The team reviewed the MySQL to PostgreSQL migration and set the release date for the twenty third of October. Sanjay will present the budget of 40 lakhs next week.",
     "discussion_points": [{"topic": "Database migration", "points": ["Staging database is ready", "Production migration deferred this quarter after load tests failed twice", "Marketing approved the new logo"], "evidence_lines": [3, 4]}],
     "candidates": [
      {"statement": "Release date is 23 October", "type": "agreed_decision", "evidence_quote": "We decided the release date is the twenty third of October", "evidence_lines": [8]},
      {"statement": "Move API gateway to AWS", "type": "agreed_decision", "evidence_quote": "Rahul suggested moving the API gateway to AWS, but nobody agreed", "evidence_lines": [7]},   # -> proposal
      {"statement": "No production migration this quarter", "type": "agreed_decision", "evidence_quote": "we will not migrate production this quarter", "evidence_lines": [3]},
      {"statement": "Adopt Redis caching", "type": "agreed_decision", "evidence_quote": "we agreed to adopt Redis for caching", "evidence_lines": [4]},   # hallucinated quote
      {"statement": "Keep the current gateway setup", "type": "agreed_decision", "evidence_quote": "so we are keeping the current setup", "evidence_lines": [7]},
      {"statement": "Migrate production this quarter", "type": "agreed_decision", "evidence_quote": "we will migrate production this quarter", "evidence_lines": [3]},  # negation flipped in quote
      {"statement": "Migrate production this quarter", "type": "agreed_decision", "evidence_quote": "migrate production this quarter", "evidence_lines": [3]},          # negation cut out of quote
      {"statement": "Adopt Redis for caching", "type": "agreed_decision", "evidence_quote": "so we are keeping the current setup", "evidence_lines": [7]},             # real quote, unrelated statement
      {"statement": "Improve alerting", "type": "agreed_decision", "evidence_quote": "sparked a long discussion about alerting", "evidence_lines": [12]},            # 'sparked' must not trigger 'parked'
     ],
     "action_items": [
      {"task": "Update Terraform scripts", "owner": "Priya", "owner_stated": True, "deadline": "by Friday", "deadline_stated": True, "evidence_quote": "Priya will update the Terraform scripts by Friday", "evidence_lines": [9]},
      {"task": "Write rollback runbook", "owner": "Rahul", "owner_stated": True, "deadline": "next week", "deadline_stated": True, "evidence_quote": "Someone needs to write the rollback runbook", "evidence_lines": [10]},  # invented owner+deadline
      {"task": "Send benchmark report", "owner": "Rahul", "owner_stated": True, "deadline": "unspecified", "deadline_stated": False, "evidence_quote": "Rahul will send the benchmark report", "evidence_lines": [11]},
      {"task": "Migrate production", "owner": "Priya", "owner_stated": True, "deadline": "this quarter", "deadline_stated": True, "evidence_quote": "Priya will migrate production this quarter", "evidence_lines": [3]},  # quote not found
      {"task": "Write rollback runbook", "owner": "Priya", "owner_stated": True, "deadline": "twenty third of October", "deadline_stated": True, "evidence_quote": "Someone needs to write the rollback runbook", "evidence_lines": [3, 8, 10]},  # evidence widened
      {"task": "Send benchmark numbers", "owner": "Rahul", "owner_stated": True, "deadline": "by end of the month", "deadline_stated": True, "evidence_quote": "I'll send the benchmark numbers by end of the month", "evidence_lines": [11, 13]},  # first person
      {"task": "Send latency dashboard", "owner": "Rahul", "owner_stated": True, "deadline": "by Monday", "deadline_stated": True, "evidence_quote": "Rahul will send the latency dashboard by Monday", "evidence_lines": [21]},  # split across lines, only one cited
     ]}
    _v = verify_output(_out, _lines, CFG)
    _acts = {a["task"]: a for a in _v["action_items"]}
    checks = [
      ("decision kept",             any("twenty third" in d["evidence_quote"] for d in _v["decisions"])),
      ("not-migrate kept",          any("will not migrate" in d["evidence_quote"] for d in _v["decisions"])),
      ("AWS -> proposal",           any("AWS" in p["statement"] for p in _v["proposals_not_adopted"]) and not any("AWS" in d["statement"] for d in _v["decisions"])),
      ("hallucinated Redis dropped", not any("Redis" in d["statement"] for d in _v["decisions"])),
      ("keep-setup kept as decision", any("current setup" in d["evidence_quote"] for d in _v["decisions"])),
      ("negation-flip dropped",     not any(d["evidence_quote"] == "we will migrate production this quarter" for d in _v["decisions"])),
      ("truncated-negation dropped", not any(d["evidence_quote"] == "migrate production this quarter" for d in _v["decisions"])
                                     and any("negation" in x["reason"] for x in _v["verification"]["dropped"] if x["item"].get("evidence_quote") == "migrate production this quarter")),
      ("ungrounded statement dropped", not any(d["statement"] == "Adopt Redis for caching" for d in _v["decisions"])),
      ("'sparked' not treated as 'parked'", any(d["statement"] == "Improve alerting" for d in _v["decisions"])),
      ("Priya/Friday kept",         _acts.get("Update Terraform scripts", {}).get("owner") == "Priya" and _acts.get("Update Terraform scripts", {}).get("deadline") == "by Friday"),
      ("runbook owner+deadline unspecified", all(a["owner"] == "unspecified" and a["deadline"] == "unspecified" for a in _v["action_items"] if a["task"] == "Write rollback runbook") and any(a["task"] == "Write rollback runbook" for a in _v["action_items"])),
      ("Rahul no deadline",         _acts.get("Send benchmark report", {}).get("owner") == "Rahul" and _acts.get("Send benchmark report", {}).get("deadline") == "unspecified"),
      ("fake migrate task dropped", "Migrate production" not in _acts),
      ("first-person owner unspecified", _acts.get("Send benchmark numbers", {}).get("owner") == "unspecified" and _acts.get("Send benchmark numbers", {}).get("deadline") == "by end of the month"),
      ("summary: invented name/number removed", "Sanjay" not in _v["summary"] and "40" not in _v["summary"] and "twenty third" in _v["summary"]),
      ("quote split over a neighbouring line found", _acts.get("Send latency dashboard", {}).get("evidence_lines") == [20, 21]
                                     and _acts.get("Send latency dashboard", {}).get("owner") == "Rahul" and _acts.get("Send latency dashboard", {}).get("deadline") == "by Monday"),
      ("point: invented topic removed", not any("logo" in p for d in _v["discussion_points"] for p in d["points"]) and any("Staging" in p for d in _v["discussion_points"] for p in d["points"])),
    ]
    bad = 0
    for name, ok in checks:
        print(("PASS " if ok else "FAIL ") + name); bad += not ok
    print("\n-- dropped:", [(x["item"].get("statement") or x["item"].get("task"), x["reason"]) for x in _v["verification"]["dropped"]])
    print("-- adjusted:", [x["note"] for x in _v["verification"]["adjusted"]])
    print("-- screened:", [x["text"][:70] for x in _v["verification"]["screened"]])
    print("\nALL GOOD" if not bad else f"\n{bad} verifier test(s) failed")
    return bad

def print_report(R3: dict):
    """Prints a summary of decisions, action items, and verification drop counts."""
    v = R3.get("verification", {})
    print("=== DOCUMENTATION VERIFICATION AUDIT ===")
    print(f"Decisions Accepted     : {len(R3.get('decisions', []))}")
    print(f"Proposals (Unadopted)  : {len(R3.get('proposals_not_adopted', []))}")
    print(f"Action Items Extracted : {len(R3.get('action_items', []))}")
    print(f"Hallucinations Dropped : {len(v.get('dropped', []))}")
    print(f"Fields Set to Unspec.  : {len(v.get('adjusted', []))}")

    dropped = v.get("dropped", [])
    if dropped:
        print("\n--- DROPPED ITEMS (FAILED VERIFICATION) ---")
        for d in dropped:
            item_name = d.get("item", {}).get("statement") or d.get("item", {}).get("task") or "Unknown"
            print(f"  [X] {item_name!r} -> Reason: {d.get('reason')}")

    adjusted = v.get("adjusted", [])
    if adjusted:
        print("\n--- FIELD ADJUSTMENTS (NAME/DEADLINE UNSPECIFIED) ---")
        for a in adjusted:
            print(f"  [!] {a.get('task')!r} -> {a.get('note')}")


In [ ]:
%%writefile stages/calendar_ics.py
"""Stage 4 - calendar export (.ics, RFC 5545). Fully deterministic; no model involved.

* The meeting date comes from (in order): the user's setting, the recording's creation-time metadata, a date in the
  file name, the processing date. The source is written into the record so nobody mistakes an assumption for a fact.
* A deadline is resolved ONLY from the words Stage 3 verified as spoken ("by Friday", "end of next week",
  "23rd of October", "in two weeks"). Every resolution records the rule used. A deadline that cannot be placed on a
  date ("before the demo") gets no calendar date; it is never guessed.
* Every calendar entry carries its citation: the verbatim evidence quote, line ids, timestamps and speakers.
* UIDs are stable (same recording + date + item -> same UID), so re-importing a re-run updates instead of duplicating.
"""
import calendar as _cal
import os
import re
import uuid
from datetime import date, datetime, time as dtime, timedelta, timezone

from config import CAL_CFG, DIAR_CFG

WEEKDAYS = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
_WD_ABBR = {"mon": 0, "tue": 1, "tues": 1, "wed": 2, "thu": 3, "thur": 3, "thurs": 3, "fri": 4, "sat": 5, "sun": 6}
MONTHS = ["january", "february", "march", "april", "may", "june", "july", "august", "september", "october",
          "november", "december"]
_MON_ABBR = {m[:3]: i + 1 for i, m in enumerate(MONTHS)}
_MON_ABBR["sept"] = 9
_ONES = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten", "eleven", "twelve",
         "thirteen", "fourteen", "fifteen", "sixteen", "seventeen", "eighteen", "nineteen"]
_TENS = {"twenty": 20, "thirty": 30, "forty": 40, "fifty": 50}
_ORD = {"first": 1, "second": 2, "third": 3, "fourth": 4, "fifth": 5, "sixth": 6, "seventh": 7, "eighth": 8, "ninth": 9,
        "tenth": 10, "eleventh": 11, "twelfth": 12, "thirteenth": 13, "fourteenth": 14, "fifteenth": 15,
        "sixteenth": 16, "seventeenth": 17, "eighteenth": 18, "nineteenth": 19, "twentieth": 20, "thirtieth": 30}
_SMALL = {"a": 1, "an": 1, "one": 1, "couple": 2, "a couple": 2, "a couple of": 2, "few": 3, "a few": 3}


# ----------------------------------------------------------------------------------------------- helpers
def _suffix(n: int) -> str:
    return "th" if 10 <= n % 100 <= 20 else {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")


def words_to_digits(text: str) -> str:
    """'twenty third' -> '23rd', 'two weeks' -> '2 weeks', 'five pm' -> '5 pm'. Up to 59 / ordinals up to 31st."""
    toks = re.findall(r"[a-z0-9:'/.-]+|[^a-z0-9:'/.\s-]", text.lower())
    out, i = [], 0
    while i < len(toks):
        t = toks[i]
        nxt = toks[i + 1] if i + 1 < len(toks) else ""
        if t in _TENS:
            v = _TENS[t]
            if nxt in _ONES[1:10]:
                out.append(str(v + _ONES.index(nxt))); i += 2; continue
            if nxt in _ORD and _ORD[nxt] < 10:
                n = v + _ORD[nxt]; out.append(f"{n}{_suffix(n)}"); i += 2; continue
            out.append(str(v)); i += 1; continue
        if t in _ORD:
            n = _ORD[t]; out.append(f"{n}{_suffix(n)}"); i += 1; continue
        if t in _ONES:
            out.append(str(_ONES.index(t))); i += 1; continue
        out.append(t); i += 1
    return " ".join(out)


def _last_day(y: int, m: int) -> date:
    return date(y, m, _cal.monthrange(y, m)[1])


def _add_months(d: date, n: int) -> date:
    m = d.month - 1 + n
    y, m = d.year + m // 12, m % 12 + 1
    return date(y, m, min(d.day, _cal.monthrange(y, m)[1]))


def _week_end(ref: date, week_end_day: int, weeks_ahead: int = 0) -> date:
    monday = ref - timedelta(days=ref.weekday())
    d = monday + timedelta(days=week_end_day + 7 * weeks_ahead)
    if weeks_ahead == 0 and d < ref:
        d += timedelta(days=7)
    return d


def _quarter_end(ref: date, quarters_ahead: int = 0) -> date:
    q = (ref.month - 1) // 3 + quarters_ahead
    y = ref.year + q // 4
    m = (q % 4) * 3 + 3
    return _last_day(y, m)


def get_tz(cfg=None):
    """IANA zone from settings, else TZ env var, else the system local zone. Returns (tzinfo, name)."""
    cfg = cfg or CAL_CFG
    for name in (cfg.timezone, os.environ.get("TZ", "")):
        name = (name or "").strip()
        if name:
            try:
                from zoneinfo import ZoneInfo
                return ZoneInfo(name), name
            except Exception:
                print(f"calendar: unknown time zone {name!r}, ignoring")
    tz = datetime.now().astimezone().tzinfo
    return tz, str(tz)


# ----------------------------------------------------------------------------------------------- times
_TIME_RE = re.compile(r"\b(?:at\s+)?(\d{1,2})(?:[:.](\d{2}))?\s*(a\.?m\.?|p\.?m\.?)(?![a-z])|\b(?:at\s+)?([01]?\d|2[0-3]):([0-5]\d)\b"
                      r"|\b(noon|midday|midnight)\b")


def parse_time(text: str):
    """Returns (time, matched_text) or (None, None)."""
    m = _TIME_RE.search(text)
    if not m:
        return None, None
    if m.group(6):
        return (dtime(12, 0) if m.group(6) in ("noon", "midday") else dtime(23, 59)), m.group(0)
    if m.group(1):
        h, mi = int(m.group(1)), int(m.group(2) or 0)
        ampm = m.group(3).replace(".", "")
        if not 1 <= h <= 12 or mi > 59:
            return None, None
        h = (h % 12) + (12 if ampm == "pm" else 0)
        return dtime(h, mi), m.group(0)
    return dtime(int(m.group(4)), int(m.group(5))), m.group(0)


# ----------------------------------------------------------------------------------------------- dates
def _explicit_date(t: str, ref: date, cfg):
    """Month-name dates, ISO dates, numeric dates (date_order), 'the 15th'. Returns (date, rule) or (None, None)."""
    mon_pat = "|".join(sorted(MONTHS + list(_MON_ABBR), key=len, reverse=True))
    m = re.search(r"\b(20\d{2})-(\d{1,2})-(\d{1,2})\b", t)
    if m:
        try:
            return date(int(m.group(1)), int(m.group(2)), int(m.group(3))), "explicit ISO date"
        except ValueError:
            return None, None
    m = (re.search(rf"\b(\d{{1,2}})(?:st|nd|rd|th)?\s+(?:of\s+)?({mon_pat})\.?(?:\s*,?\s*(20\d{{2}}))?\b", t) or None)
    day = mon = yr = None
    if m:
        day, mon, yr = int(m.group(1)), m.group(2), m.group(3)
    else:
        m = re.search(rf"\b({mon_pat})\.?\s+(?:the\s+)?(\d{{1,2}})(?:st|nd|rd|th)?(?:\s*,?\s*(20\d{{2}}))?\b", t)
        if m:
            mon, day, yr = m.group(1), int(m.group(2)), m.group(3)
    if day is not None:
        month = MONTHS.index(mon) + 1 if mon in MONTHS else _MON_ABBR.get(mon[:4] if mon[:4] == "sept" else mon[:3])
        try:
            if yr:
                return date(int(yr), month, day), "explicit date"
            d = date(ref.year, month, day)
            if d < ref:
                d = date(ref.year + 1, month, day)
                return d, "explicit date (no year spoken; next occurrence after the meeting)"
            return d, "explicit date (no year spoken; this year)"
        except ValueError:
            return None, None
    m = re.search(r"\b(\d{1,2})[/.](\d{1,2})(?:[/.](\d{2,4}))?\b", t)
    if m and not re.search(r"\d[.:]\d{2}\s*(am|pm)", t):
        a, b, y = int(m.group(1)), int(m.group(2)), m.group(3)
        d_, m_ = (a, b) if str(cfg.date_order).upper().startswith("D") else (b, a)
        try:
            yy = (int(y) + 2000 if len(y) == 2 else int(y)) if y else ref.year
            d = date(yy, m_, d_)
            if not y and d < ref:
                d = date(yy + 1, m_, d_)
            return d, f"numeric date read as {cfg.date_order.upper()}"
        except ValueError:
            return None, None
    m = re.search(r"\bthe\s+(\d{1,2})(?:st|nd|rd|th)\b", t)
    if m:
        dd = int(m.group(1))
        y, mo = ref.year, ref.month
        if dd < ref.day:
            y, mo = (y + 1, 1) if mo == 12 else (y, mo + 1)
        try:
            return date(y, mo, dd), "day of month only (this month, or next month if already past)"
        except ValueError:
            return None, None
    return None, None


def resolve_deadline(phrase: str, ref: date, cfg=None) -> dict | None:
    """Spoken deadline -> {'date', 'time', 'rule', 'confidence'} relative to the meeting date, or None."""
    cfg = cfg or CAL_CFG
    if not phrase or str(phrase).strip().lower() in ("", "unspecified", "none", "n/a"):
        return None
    t = words_to_digits(str(phrase).lower().replace("’", "'"))
    t = re.sub(r"[^a-z0-9:/.' -]", " ", t)
    t = " ".join(t.split())
    tm, tm_txt = parse_time(t)
    tt = t.replace(tm_txt, " ") if tm_txt else t
    d, rule, conf = None, "", "high"

    def hit(p):
        return re.search(p, tt)

    if hit(r"\bday after tomorrow\b"):
        d, rule = ref + timedelta(days=2), "'day after tomorrow' = meeting date + 2 days"
    elif hit(r"\btomorrow\b"):
        d, rule = ref + timedelta(days=1), "'tomorrow' = meeting date + 1 day"
    elif hit(r"\b(today|tonight|eod|end of (the )?day|close of business|cob)\b"):
        d, rule = ref, "same day as the meeting"
    elif hit(r"\bend of (the )?next week\b|\bnext week\b"):
        d, rule = _week_end(ref, cfg.week_end_day, 1), f"end of next week ({WEEKDAYS[cfg.week_end_day].title()})"
        conf = "medium" if not hit(r"\bend of") else "high"
    elif hit(r"\bend of (the |this )?week\b|\beow\b|\bthis week\b|\bwithin the week\b"):
        d, rule = _week_end(ref, cfg.week_end_day, 0), f"end of this week ({WEEKDAYS[cfg.week_end_day].title()})"
    elif hit(r"\bend of (the )?next month\b|\bnext month\b"):
        n = _add_months(ref.replace(day=1), 1)
        d, rule = _last_day(n.year, n.month), "end of next month"
        conf = "medium" if not hit(r"\bend of") else "high"
    elif hit(r"\bend of (the |this )?month\b|\beom\b|\bthis month\b|\bmonth end\b|\bmonth-end\b"):
        d, rule = _last_day(ref.year, ref.month), "end of this month"
    elif hit(r"\bnext quarter\b"):
        d, rule, conf = _quarter_end(ref, 1), "end of next calendar quarter", "medium"
    elif hit(r"\bend of (the |this )?quarter\b|\beoq\b|\bthis quarter\b|\bquarter end\b"):
        d, rule = _quarter_end(ref, 0), "end of this calendar quarter"
    elif hit(r"\bnext year\b"):
        d, rule, conf = date(ref.year + 1, 12, 31), "end of next year", "medium"
    elif hit(r"\bend of (the |this )?year\b|\beoy\b|\bthis year\b|\byear end\b"):
        d, rule = date(ref.year, 12, 31), "end of this year"
    if d is None:
        m = hit(r"\b(?:in|within|after)\s+(?:the\s+next\s+)?(\d+|a couple of|a couple|couple|a few|few|a|an)\s+"
                r"(business days?|working days?|days?|weeks?|fortnights?|months?)\b")
        if m:
            n = int(m.group(1)) if m.group(1).isdigit() else _SMALL.get(m.group(1), 1)
            unit = m.group(2)
            if unit.startswith(("business", "working")):
                d, k = ref, 0
                while k < n:
                    d += timedelta(days=1)
                    if d.weekday() < 5:
                        k += 1
            elif unit.startswith("day"):
                d = ref + timedelta(days=n)
            elif unit.startswith("week"):
                d = ref + timedelta(weeks=n)
            elif unit.startswith("fortnight"):
                d = ref + timedelta(weeks=2 * n)
            else:
                d = _add_months(ref, n)
            rule = f"meeting date + {n} {unit}"
            if m.group(1) in ("a couple of", "a couple", "couple", "a few", "few"):
                conf = "medium"
    if d is None:
        d, r = _explicit_date(tt, ref, cfg)
        if d is not None:
            rule = r
    if d is None:
        wd_pat = "|".join(WEEKDAYS + sorted(_WD_ABBR, key=len, reverse=True))
        m = hit(rf"\b(next|this|coming|the coming)?\s*({wd_pat})\b")
        if m:
            name = m.group(2)
            target = WEEKDAYS.index(name) if name in WEEKDAYS else _WD_ABBR[name]
            mod = (m.group(1) or "").strip()
            if mod == "next":
                monday_next = ref - timedelta(days=ref.weekday()) + timedelta(days=7)
                d = monday_next + timedelta(days=target)
                rule = f"'next {WEEKDAYS[target].title()}' = {WEEKDAYS[target].title()} of the following week"
                conf = "medium"
            else:
                ahead = (target - ref.weekday()) % 7
                if ahead == 0 and mod not in ("this",):
                    ahead = 7
                d = ref + timedelta(days=ahead)
                rule = f"first {WEEKDAYS[target].title()} after the meeting" if ahead else f"{WEEKDAYS[target].title()} (same day)"
    if d is None and tm is not None:
        d, rule = ref, "time only -> same day as the meeting"
        conf = "medium"
    if d is None:
        return None
    return {"date": d.isoformat(), "time": tm.strftime("%H:%M") if tm else None, "rule": rule, "confidence": conf,
            "spoken": str(phrase)}


# ----------------------------------------------------------------------------------------------- meeting date
def _parse_iso_dt(s: str):
    s = str(s).strip()
    if not s:
        return None
    try:
        s2 = s.replace("Z", "+00:00")
        s2 = re.sub(r"(\.\d{6})\d+", r"\1", s2)
        dt = datetime.fromisoformat(s2)
        return dt if dt.tzinfo else dt.replace(tzinfo=timezone.utc)
    except Exception:
        return None


def meeting_datetime(record: dict, cfg=None):
    """(date, time-or-None, tzinfo, tzname, source)."""
    cfg = cfg or CAL_CFG
    tz, tzname = get_tz(cfg)
    now = datetime.now(tz)
    user_t = None
    if cfg.meeting_time:
        mt = re.match(r"^\s*(\d{1,2}):(\d{2})\s*$", str(cfg.meeting_time))
        if mt and int(mt.group(1)) < 24 and int(mt.group(2)) < 60:
            user_t = dtime(int(mt.group(1)), int(mt.group(2)))
    if cfg.meeting_date:
        s = str(cfg.meeting_date).strip()
        d = None
        try:
            d = date.fromisoformat(s)
        except ValueError:
            d0, _ = _explicit_date(s.lower(), now.date(), cfg)
            d = d0
        if d:
            return d, user_t, tz, tzname, "set by user"
        print(f"calendar: could not read meeting_date {s!r}; trying the recording metadata")
    meta = record.get("meta", {}) or {}
    created = _parse_iso_dt(meta.get("recording_created") or "")
    if created and datetime(2000, 1, 1, tzinfo=timezone.utc) < created <= datetime.now(timezone.utc) + timedelta(days=1):
        local = created.astimezone(tz)
        return local.date(), user_t or local.time().replace(second=0, microsecond=0), tz, tzname, "recording metadata (creation_time)"
    name = str(meta.get("audio_file") or "")
    m = re.search(r"(20\d{2})[-_.]?(0[1-9]|1[0-2])[-_.]?(0[1-9]|[12]\d|3[01])", name)
    if m:
        try:
            return date(int(m.group(1)), int(m.group(2)), int(m.group(3))), user_t, tz, tzname, "date in the file name"
        except ValueError:
            pass
    return now.date(), user_t, tz, tzname, "processing date (assumed; set the meeting date to correct it)"


# ----------------------------------------------------------------------------------------------- ICS writer
def _esc(text) -> str:
    return (str(text).replace("\\", "\\\\").replace(";", "\\;").replace(",", "\\,")
            .replace("\r\n", "\\n").replace("\n", "\\n").replace("\r", ""))


def _fold(line: str) -> list:
    """RFC 5545 3.1: lines longer than 75 octets are folded (CRLF + one space), never inside a UTF-8 character."""
    out, cur, size, limit = [], "", 0, 75
    for ch in line:
        n = len(ch.encode("utf-8"))
        if size + n > limit:
            out.append(cur)
            cur, size, limit = " " + ch, 1 + n, 75
        else:
            cur += ch
            size += n
    out.append(cur)
    return out


def _utc(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).strftime("%Y%m%dT%H%M%SZ")


def _uid(cfg, *parts) -> str:
    return f"{uuid.uuid5(uuid.NAMESPACE_URL, '|'.join(str(p) for p in parts))}@{cfg.uid_domain}"


def _fmt_ts(sec) -> str:
    if sec is None:
        return "--:--"
    sec = int(sec); h, rem = divmod(sec, 3600); m, s = divmod(rem, 60)
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"


def _citation(item: dict, line_info: dict, names: dict) -> str:
    refs = []
    for i in item.get("evidence_lines", []):
        d = line_info.get(i, {})
        spk = d.get("speaker")
        spk = names.get(spk, spk) if spk and spk != DIAR_CFG.unknown_label else None
        refs.append(f"L{i:03d} {_fmt_ts(d.get('start'))}" + (f" {spk}" if spk else ""))
    return f"\"{item.get('evidence_quote', '')}\" [{'; '.join(refs)}]"


def build_ics(record: dict, cfg=None, now: datetime | None = None) -> tuple:
    """Returns (ics_text, calendar_info). Also writes item['due'] into record['action_items'] (in place)."""
    cfg = cfg or CAL_CFG
    now = now or datetime.now(timezone.utc)
    mdate, mtime, tz, tzname, src = meeting_datetime(record, cfg)
    meta = record.get("meta", {}) or {}
    names = {k: v for k, v in (record.get("speaker_names") or {}).items() if v}
    rt = record.get("refined_transcript") if isinstance(record.get("refined_transcript"), dict) else {}
    line_info = {s["id"]: s for s in (rt.get("segments") or [])}
    title = ((record.get("domain") or {}).get("domain") or meta.get("meeting_context") or "Meeting").strip()
    audio = meta.get("audio_file") or "recording"
    base = (audio, mdate.isoformat())
    L = ["BEGIN:VCALENDAR", "VERSION:2.0", f"PRODID:{cfg.prodid}", "CALSCALE:GREGORIAN", "METHOD:PUBLISH",
         f"X-WR-CALNAME:{_esc('Meeting: ' + title)}"]
    stamp = _utc(now)
    items_info, n_events, n_todos = [], 0, 0
    meeting_uid = _uid(cfg, *base, "meeting")

    def add(*lines):
        for ln in lines:
            L.extend(_fold(ln))

    if cfg.include_meeting_event:
        desc = [f"Source recording: {audio}", f"Meeting date source: {src}", ""]
        mins = record.get("minutes") or {}
        if mins.get("summary"):
            desc += ["Summary:", mins["summary"], ""]
        if record.get("decisions"):
            desc.append("Decisions:")
            desc += [f"- {d['decision']} — {_citation(d, line_info, names)}" for d in record["decisions"]]
            desc.append("")
        if record.get("action_items"):
            desc.append("Action items:")
            desc += [f"- {a['task']} (owner: {a['owner']}; deadline: {a['deadline']}) — {_citation(a, line_info, names)}"
                     for a in record["action_items"]]
        add("BEGIN:VEVENT", f"UID:{meeting_uid}", f"DTSTAMP:{stamp}")
        if mtime is not None:
            start = datetime.combine(mdate, mtime).replace(tzinfo=tz or timezone.utc)
            dur = max(60, int(float(meta.get("duration_s") or 0) or 3600))
            add(f"DTSTART:{_utc(start)}", f"DTEND:{_utc(start + timedelta(seconds=dur))}")
        else:
            add(f"DTSTART;VALUE=DATE:{mdate:%Y%m%d}", f"DTEND;VALUE=DATE:{mdate + timedelta(days=1):%Y%m%d}")
        add(f"SUMMARY:{_esc('Meeting: ' + title)}", f"DESCRIPTION:{_esc(chr(10).join(desc))}",
            "CATEGORIES:MEETING", "TRANSP:TRANSPARENT", "END:VEVENT")
        n_events += 1

    for idx, a in enumerate(record.get("action_items") or []):
        due = resolve_deadline(a.get("deadline", ""), mdate, cfg) if a.get("deadline_stated", True) else None
        a["due"] = due
        speakers = [names.get(s, s) for s in a.get("speakers", []) if s and s != DIAR_CFG.unknown_label]
        owner = a.get("owner", "unspecified")
        summary = f"Action: {a['task']}" + (f" ({owner})" if owner and owner != "unspecified" else "")
        desc = [f"Task: {a['task']}", f"Owner (as spoken): {owner}",
                f"Said by (diarized speaker): {', '.join(speakers) if speakers else 'not diarized'}",
                f"Deadline (as spoken): {a.get('deadline', 'unspecified')}"]
        if due:
            desc.append(f"Due date: {due['date']}" + (f" {due['time']}" if due.get("time") else "") +
                        f" — computed: {due['rule']} (meeting date {mdate.isoformat()}, {src}; confidence {due['confidence']})")
        else:
            desc.append("Due date: none (no deadline that can be placed on a calendar was spoken)")
        desc += [f"Evidence: {_citation(a, line_info, names)}", f"Meeting: {title} — {audio}"]
        key = (*base, "action", a["task"], ",".join(map(str, a.get("evidence_lines", []))))
        info = {"task": a["task"], "owner": owner, "deadline": a.get("deadline"), "due": due, "event": False, "todo": False}
        if due:
            add("BEGIN:VEVENT", f"UID:{_uid(cfg, *key, 'event')}", f"DTSTAMP:{stamp}")
            dd = date.fromisoformat(due["date"])
            if due.get("time"):
                hh, mm = (int(x) for x in due["time"].split(":"))
                start = datetime.combine(dd, dtime(hh, mm)).replace(tzinfo=tz)
                add(f"DTSTART:{_utc(start)}", f"DTEND:{_utc(start + timedelta(minutes=max(5, cfg.default_event_minutes)))}")
            else:
                add(f"DTSTART;VALUE=DATE:{dd:%Y%m%d}", f"DTEND;VALUE=DATE:{dd + timedelta(days=1):%Y%m%d}")
            add(f"SUMMARY:{_esc(summary)}", f"DESCRIPTION:{_esc(chr(10).join(desc))}", "CATEGORIES:ACTION-ITEM",
                f"RELATED-TO:{meeting_uid}" if cfg.include_meeting_event else "X-MA-RELATED:none", "TRANSP:TRANSPARENT")
            if cfg.alarm_minutes_before and cfg.alarm_minutes_before > 0:
                add("BEGIN:VALARM", "ACTION:DISPLAY", f"DESCRIPTION:{_esc(summary)}",
                    f"TRIGGER:-PT{int(cfg.alarm_minutes_before)}M", "END:VALARM")
            add("END:VEVENT")
            info["event"] = True; n_events += 1
        if cfg.include_vtodo:
            add("BEGIN:VTODO", f"UID:{_uid(cfg, *key, 'todo')}", f"DTSTAMP:{stamp}", f"SUMMARY:{_esc(summary)}",
                f"DESCRIPTION:{_esc(chr(10).join(desc))}", "STATUS:NEEDS-ACTION", "CATEGORIES:ACTION-ITEM")
            if due:
                dd = date.fromisoformat(due["date"])
                if due.get("time"):
                    hh, mm = (int(x) for x in due["time"].split(":"))
                    add(f"DUE:{_utc(datetime.combine(dd, dtime(hh, mm)).replace(tzinfo=tz))}")
                else:
                    add(f"DUE;VALUE=DATE:{dd:%Y%m%d}")
            if cfg.include_meeting_event:
                add(f"RELATED-TO:{meeting_uid}")
            add("END:VTODO")
            info["todo"] = True; n_todos += 1
        items_info.append(info)
    L.append("END:VCALENDAR")
    text = "\r\n".join(L) + "\r\n"
    cal = {"meeting_date": mdate.isoformat(), "meeting_time": mtime.strftime("%H:%M") if mtime else None,
           "meeting_date_source": src, "timezone": tzname, "n_events": n_events, "n_todos": n_todos,
           "resolved": sum(1 for i in items_info if i["due"]), "unresolved": sum(1 for i in items_info if not i["due"]),
           "items": items_info}
    return text, cal


def validate_ics(text: str) -> list:
    """Structural checks (CRLF, folding, balanced components, required properties). [] = valid."""
    problems = []
    if not text.startswith("BEGIN:VCALENDAR\r\n") or not text.endswith("END:VCALENDAR\r\n"):
        problems.append("missing VCALENDAR wrapper / CRLF line endings")
    raw_lines = text.split("\r\n")[:-1]
    for i, ln in enumerate(raw_lines):
        if len(ln.encode("utf-8")) > 75:
            problems.append(f"line {i + 1} longer than 75 octets")
        if "\n" in ln or "\r" in ln:
            problems.append(f"line {i + 1} contains a bare line break")
    unfolded = []
    for ln in raw_lines:
        if ln.startswith(" ") and unfolded:
            unfolded[-1] += ln[1:]
        else:
            unfolded.append(ln)
    stack = []
    props = {}
    for ln in unfolded:
        if ln.startswith("BEGIN:"):
            stack.append(ln[6:]); props[len(stack)] = set()
        elif ln.startswith("END:"):
            comp = ln[4:]
            if not stack or stack[-1] != comp:
                problems.append(f"unbalanced END:{comp}")
                break
            have = props.get(len(stack), set())
            need = {"VEVENT": {"UID", "DTSTAMP", "DTSTART"}, "VTODO": {"UID", "DTSTAMP"},
                    "VCALENDAR": {"VERSION", "PRODID"}, "VALARM": {"ACTION", "TRIGGER"}}.get(comp, set())
            if need - have:
                problems.append(f"{comp} missing {sorted(need - have)}")
            stack.pop()
        else:
            name = re.split(r"[;:]", ln, 1)[0]
            if stack:
                props[len(stack)].add(name)
    if stack:
        problems.append(f"unclosed components {stack}")
    return problems


def add_calendar(record: dict, run_dir, cfg=None) -> tuple:
    """Writes meeting_calendar.ics into run_dir, stores record['calendar'] and action_items[*]['due'].
    Returns (path or None, calendar_info). Never raises."""
    from pathlib import Path
    cfg = cfg or CAL_CFG
    if not cfg.enabled:
        record["calendar"] = {"status": "disabled"}
        return None, record["calendar"]
    try:
        text, cal = build_ics(record, cfg)
        problems = validate_ics(text)
        cal["status"] = "ok" if not problems else "invalid"
        cal["problems"] = problems
        p = Path(run_dir) / "meeting_calendar.ics"
        p.write_text(text, encoding="utf-8", newline="")
        record["calendar"] = cal
        return str(p), cal
    except Exception as e:
        record["calendar"] = {"status": "failed", "error": f"{type(e).__name__}: {str(e)[:200]}"}
        for a in record.get("action_items") or []:
            a.setdefault("due", None)
        return None, record["calendar"]


def self_test() -> int:
    """Deadline resolver + ICS validity, no model. Returns number of failures."""
    class _C:  # a config copy that does not depend on the user's settings
        date_order, week_end_day = "DMY", 4
    ref = date(2026, 10, 8)            # a Thursday
    cases = [("by Friday", "2026-10-09"), ("by friday 5pm", "2026-10-09 17:00"), ("next Friday", "2026-10-16"),
             ("by Thursday", "2026-10-15"), ("end of the week", "2026-10-09"), ("end of next week", "2026-10-16"),
             ("by end of the month", "2026-10-31"), ("this quarter", "2026-12-31"), ("tomorrow", "2026-10-09"),
             ("in two weeks", "2026-10-22"), ("within 3 business days", "2026-10-13"), ("by the twenty third of October", "2026-10-23"),
             ("October 5th", "2027-10-05"), ("by the 15th", "2026-10-15"), ("the 3rd", "2026-11-03"), ("by 10/11", "2026-11-10"),
             ("EOD", "2026-10-08"), ("before the demo", None), ("unspecified", None), ("next month", "2026-11-30"),
             ("by noon tomorrow", "2026-10-09 12:00"), ("2026-12-01", "2026-12-01")]
    fails = 0
    for phrase, want in cases:
        r = resolve_deadline(phrase, ref, _C)
        got = None if r is None else r["date"] + (f" {r['time']}" if r.get("time") else "")
        ok = got == want
        fails += not ok
        print(("PASS " if ok else "FAIL ") + f"{phrase!r:32} -> {got}" + ("" if ok else f"   (expected {want})"))
    rec = {"meta": {"audio_file": "standup_2026-10-07.m4a", "duration_s": 600},
           "domain": {"domain": "software stand-up; sprint review"},
           "minutes": {"summary": "The team agreed the release date, with commas, semicolons; and a very long line " * 3},
           "refined_transcript": {"segments": [{"id": 9, "start": 29.0, "speaker": "SPEAKER_01"}, {"id": 10, "start": 37.0, "speaker": "SPEAKER_00"}]},
           "speaker_names": {"SPEAKER_01": "Priyá"},
           "decisions": [{"decision": "Release on the 23rd", "evidence_quote": "We decided the release date is the 23rd of October", "evidence_lines": [9]}],
           "action_items": [{"task": "Update Terraform scripts", "owner": "Priya", "deadline": "by Friday", "deadline_stated": True,
                             "evidence_quote": "Priya will update the Terraform scripts by Friday", "evidence_lines": [9, 10], "speakers": ["SPEAKER_01"]},
                            {"task": "Write rollback runbook", "owner": "unspecified", "deadline": "unspecified", "deadline_stated": False,
                             "evidence_quote": "Someone needs to write the rollback runbook", "evidence_lines": [10], "speakers": ["SPEAKER_00"]}]}

    class _K:
        enabled = True; meeting_date = ""; meeting_time = ""; timezone = "UTC"; date_order = "DMY"; week_end_day = 4
        include_meeting_event = True; include_vtodo = True; alarm_minutes_before = 15; default_event_minutes = 30
        prodid = "-//test//EN"; uid_domain = "test.local"
    text, cal = build_ics(rec, _K)
    probs = validate_ics(text)
    checks = [("ics structurally valid", not probs),
              ("meeting date from file name", cal["meeting_date"] == "2026-10-07" and "file name" in cal["meeting_date_source"]),
              ("Friday resolved from meeting date", rec["action_items"][0]["due"]["date"] == "2026-10-09"),
              ("unspecified deadline not placed", rec["action_items"][1]["due"] is None and cal["unresolved"] == 1),
              ("events = meeting + 1 action", cal["n_events"] == 2 and cal["n_todos"] == 2),
              ("citation in description", "L009 00:29 Priyá" in text.replace("\r\n ", "")),
              ("stable UIDs", build_ics(rec, _K)[0].split("UID:")[1][:36] == text.split("UID:")[1][:36])]
    for name, ok in checks:
        print(("PASS " if ok else "FAIL ") + name + ("" if ok else f"  {probs}"))
        fails += not ok
    print("\nALL GOOD" if not fails else f"\n{fails} calendar test(s) failed")
    return fails


In [ ]:
%%writefile report_html.py
"""Citation-grounded rendering of meeting_record.json (used by the Gradio app and saved as meeting_record.html).

Every statement shown is followed by its citations: line id, timestamp and diarized speaker. Clicking a citation
jumps to (and highlights) that line in the transcript below. The search box ranks transcript lines and returns only
verbatim lines with their citations; it never writes an answer of its own.
"""
import html
import re

from config import DIAR_CFG

_PALETTE = ["#2563eb", "#db2777", "#059669", "#d97706", "#7c3aed", "#0891b2", "#dc2626", "#4d7c0f"]

CSS = """
.ma{font-family:system-ui,-apple-system,'Segoe UI',Roboto,sans-serif;line-height:1.5;color:var(--ma-fg,#111827);max-width:100%}
.ma h2{font-size:1.15rem;margin:1.2em 0 .4em;border-bottom:1px solid #e5e7eb;padding-bottom:.2em}
.ma h3{font-size:1rem;margin:.8em 0 .3em}
.ma .meta{font-size:.82rem;color:#6b7280}
.ma .warn{background:#fef3c7;border-left:4px solid #f59e0b;padding:.4em .7em;margin:.5em 0;font-size:.88rem;color:#78350f}
.ma .item{margin:.35em 0 .6em}
.ma .quote{font-style:italic;color:#374151}
.ma a.cite{display:inline-block;font-size:.75rem;text-decoration:none;border:1px solid #cbd5e1;border-radius:999px;
  padding:0 .5em;margin:0 .15em;color:#1e3a8a;background:#eff6ff;white-space:nowrap}
.ma a.cite:hover{background:#dbeafe}
.ma .spk{display:inline-block;font-size:.72rem;font-weight:600;color:#fff;border-radius:4px;padding:0 .35em;margin-right:.3em}
.ma table{border-collapse:collapse;width:100%;font-size:.88rem}
.ma td,.ma th{border:1px solid #e5e7eb;padding:.3em .45em;vertical-align:top;text-align:left}
.ma th{background:#f9fafb}
.ma .tscroll{overflow-x:auto}
.ma .line{padding:.15em .4em;border-radius:4px;scroll-margin-top:60px}
.ma .line:target{background:#fde68a}
.ma .lid{font-family:ui-monospace,monospace;font-size:.75rem;color:#6b7280;margin-right:.4em}
.ma .flag{font-size:.72rem;color:#b45309;margin-left:.3em}
.ma .muted{color:#6b7280}
.ma .due{white-space:nowrap}
@media (prefers-color-scheme: dark){.ma{--ma-fg:#e5e7eb}.ma th{background:#1f2937}.ma td,.ma th{border-color:#374151}
 .ma .quote{color:#d1d5db}.ma a.cite{background:#1e293b;color:#bfdbfe;border-color:#334155}.ma .line:target{background:#78350f}}
"""


def _e(s) -> str:
    return html.escape(str(s if s is not None else ""))


def _fmt_ts(sec) -> str:
    if sec is None:
        return "--:--"
    sec = int(sec); h, rem = divmod(sec, 3600); m, s = divmod(rem, 60)
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"


def _segments(rec: dict) -> list:
    rt = rec.get("refined_transcript") if isinstance(rec.get("refined_transcript"), dict) else {}
    segs = rt.get("segments") or (rec.get("raw_transcript") or {}).get("segments") or []
    raw = {s["id"]: s for s in (rec.get("raw_transcript") or {}).get("segments", []) or []}
    out = []
    for s in segs:
        r = raw.get(s["id"], {})
        out.append({"id": s["id"], "start": s.get("start"), "speaker": s.get("speaker") or r.get("speaker"),
                    "text": s.get("text", ""), "asr2_diff_frac": r.get("asr2_diff_frac"),
                    "asr2_unsupported": r.get("asr2_unsupported"), "critical_low_conf": r.get("critical_low_conf") or []})
    return out


def _speaker_colors(segs: list) -> dict:
    order = []
    for s in segs:
        if s["speaker"] and s["speaker"] not in order and s["speaker"] != DIAR_CFG.unknown_label:
            order.append(s["speaker"])
    return {spk: _PALETTE[i % len(_PALETTE)] for i, spk in enumerate(order)}


class _Ctx:
    def __init__(self, rec: dict, prefix: str = "ma"):
        self.rec = rec
        self.prefix = prefix
        self.segs = _segments(rec)
        self.by_id = {s["id"]: s for s in self.segs}
        self.colors = _speaker_colors(self.segs)
        self.names = {k: v for k, v in (rec.get("speaker_names") or {}).items() if v}

    def spk_name(self, label):
        return self.names.get(label, label)

    def spk_badge(self, label):
        if not label or label == DIAR_CFG.unknown_label:
            return ""
        return f'<span class="spk" style="background:{self.colors.get(label, "#6b7280")}">{_e(self.spk_name(label))}</span>'

    def cite(self, ids) -> str:
        chips = []
        for i in ids or []:
            s = self.by_id.get(i, {})
            spk = s.get("speaker")
            spk_txt = f" · {self.spk_name(spk)}" if spk and spk != DIAR_CFG.unknown_label else ""
            chips.append(f'<a class="cite" href="#{self.prefix}-L{i:03d}" title="jump to line {i}">'
                         f'L{i:03d} · {_fmt_ts(s.get("start"))}{_e(spk_txt)}</a>')
        return "".join(chips)


def render_fragment(rec: dict, prefix: str = "ma", show_transcript: bool = True) -> str:
    c = _Ctx(rec, prefix)
    meta = rec.get("meta", {}) or {}
    models = meta.get("models", {}) or {}
    diar = rec.get("diarization") or {}
    cal = rec.get("calendar") or {}
    mins = rec.get("minutes") or {}
    H = ['<div class="ma">']
    title = ((rec.get("domain") or {}).get("domain") or meta.get("meeting_context") or "Meeting")
    H.append(f"<h2>{_e(title)} — {_e(meta.get('audio_file') or 'recording')}</h2>")
    bits = [f"duration {_e(round(float(meta.get('duration_s') or 0)))} s"]
    if cal.get("meeting_date"):
        bits.append(f"meeting date {_e(cal['meeting_date'])} ({_e(cal.get('meeting_date_source', ''))})")
    bits.append(f"speakers: {_e(diar.get('n_speakers'))} via {_e(models.get('diarization'))}" if diar.get("status") == "ok"
                else "speakers: not diarized")
    bits.append(f"STT {_e(models.get('stt'))}")
    a2 = rec.get("asr2") or {}
    if a2:
        bits.append(f"cross-check {_e(a2.get('model'))}: {round(100 * float(a2.get('disagreement_rate') or 0), 1)}% words differ")
    bits.append(f"refiner {_e(models.get('refiner'))} · documenter {_e(models.get('documenter'))}")
    H.append(f'<div class="meta">{" · ".join(bits)}</div>')
    warns = []
    if diar.get("status") not in (None, "ok"):
        warns.append("Speaker diarization did not run: " + str(diar.get("reason", "")))
    if rec.get("status") and rec.get("status") != "ok":
        warns.append(f"Documenter status: {rec['status']}. " + " ".join(meta.get("documenter_warnings", []) or []))
    rt = rec.get("refined_transcript") if isinstance(rec.get("refined_transcript"), dict) else {}
    if rt.get("status") == "fallback_raw":
        warns.append("Stage 2 fell back to the raw transcript (see the edit log).")
    if cal and "assumed" in str(cal.get("meeting_date_source", "")):
        warns.append("Meeting date was not known; due dates are computed from the processing date. Set the meeting date and click 'Update'.")
    for w in warns:
        H.append(f'<div class="warn">{_e(w)}</div>')

    H.append("<h2>Summary</h2>")
    H.append(f"<p>{_e(mins.get('summary')) or '<span class=muted>No summary produced.</span>'}</p>")

    H.append("<h2>Decisions</h2>")
    if rec.get("decisions"):
        H.append("<ol>")
        for d in rec["decisions"]:
            weak = "" if d.get("explicit_decision_wording", True) else ' <span class="flag">weak evidence</span>'
            H.append(f'<li class="item">{_e(d["decision"])}{weak}<br><span class="quote">“{_e(d["evidence_quote"])}”</span> '
                     f'{c.cite(d["evidence_lines"])}</li>')
        H.append("</ol>")
    else:
        H.append('<p class="muted">No decisions were reached in this meeting.</p>')

    H.append("<h2>Action items</h2>")
    if rec.get("action_items"):
        H.append('<div class="tscroll"><table><tr><th>#</th><th>Task</th><th>Owner (spoken)</th><th>Said by</th>'
                 '<th>Deadline (spoken)</th><th>Due date</th><th>Evidence</th></tr>')
        for i, a in enumerate(rec["action_items"], 1):
            due = a.get("due") or {}
            due_html = (f'<span class="due">{_e(due["date"])}{(" " + _e(due["time"])) if due.get("time") else ""}</span>'
                        f'<br><span class="muted" style="font-size:.75rem">{_e(due.get("rule", ""))}</span>') if due.get("date") else '<span class="muted">—</span>'
            said = "".join(c.spk_badge(s) for s in a.get("speakers", [])) or '<span class="muted">—</span>'
            H.append(f"<tr><td>{i}</td><td>{_e(a['task'])}</td><td>{_e(a['owner'])}</td><td>{said}</td>"
                     f"<td>{_e(a['deadline'])}</td><td>{due_html}</td>"
                     f"<td><span class=\"quote\">“{_e(a['evidence_quote'])}”</span><br>{c.cite(a['evidence_lines'])}</td></tr>")
        H.append("</table></div>")
    else:
        H.append('<p class="muted">No action items were assigned.</p>')

    for key, label in (("proposals_not_adopted", "Proposals discussed but not adopted"),
                       ("tentative", "Tentative — needs confirmation"), ("open_questions", "Open questions")):
        items = rec.get(key) or []
        if items:
            H.append(f"<h2>{label}</h2><ul>")
            for p in items:
                q = f'<br><span class="quote">“{_e(p.get("evidence_quote"))}”</span>' if p.get("evidence_quote") else ""
                H.append(f'<li class="item">{_e(p["statement"])}{q} {c.cite(p["evidence_lines"])}</li>')
            H.append("</ul>")

    if mins.get("discussion_points"):
        H.append("<h2>Minutes</h2>")
        for d in mins["discussion_points"]:
            H.append(f"<h3>{_e(d.get('topic'))} {c.cite(d.get('evidence_lines', []))}</h3><ul>")
            H += [f"<li>{_e(p)}</li>" for p in d.get("points", [])]
            H.append("</ul>")

    v = meta.get("documenter_verification") or {}
    H.append(f'<p class="meta">Verifier: {_e(v.get("dropped", 0))} item(s) dropped for failing evidence checks, '
             f'{_e(v.get("adjusted", 0))} owner/deadline field(s) set to unspecified, {_e(v.get("screened", 0))} unsupported '
             f'sentence(s) removed. "Said by" is the diarized speaker of the evidence line, not an inferred owner.</p>')

    if show_transcript:
        H.append("<h2>Transcript</h2>")
        for s in c.segs:
            flags = ""
            if s.get("asr2_unsupported"):
                flags += '<span class="flag" title="the second recogniser heard this line very differently">⚠ cross-check disagrees</span>'
            elif s.get("asr2_diff_frac") and float(s["asr2_diff_frac"]) >= 0.3:
                flags += f'<span class="flag" title="share of words the second recogniser heard differently">≈ {int(100 * float(s["asr2_diff_frac"]))}% differs</span>'
            if s.get("critical_low_conf"):
                flags += f'<span class="flag" title="numbers/negations Whisper was unsure about">? {_e(", ".join(s["critical_low_conf"]))}</span>'
            H.append(f'<div class="line" id="{prefix}-L{s["id"]:03d}"><span class="lid">L{s["id"]:03d} {_fmt_ts(s.get("start"))}</span>'
                     f'{c.spk_badge(s.get("speaker"))}{_e(s["text"])}{flags}</div>')
    H.append("</div>")
    return "\n".join(H)


def render_page(rec: dict) -> str:
    title = _e((rec.get("meta") or {}).get("audio_file") or "Meeting record")
    return ("<!DOCTYPE html><html lang=\"en\"><head><meta charset=\"utf-8\">"
            "<meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">"
            f"<title>Meeting record — {title}</title><style>body{{margin:1.5rem;}}{CSS}</style></head><body>"
            + render_fragment(rec) + "</body></html>")


def with_style(fragment: str) -> str:
    return f"<style>{CSS}</style>" + fragment


# ----------------------------------------------------------------------------------------------- search
_STOP = set("the a an and or but so if then of to in on at for with by from as is are was were be been it this that "
            "we you they he she i me my our your what who when where why how did do does about any there".split())


def search(rec: dict, query: str, k: int = 8) -> list:
    """Rank transcript lines for a question. Returns [{'id', 'score', 'start', 'speaker', 'text'}] (verbatim lines)."""
    from rapidfuzz import fuzz
    q = (query or "").strip()
    if not q:
        return []
    qwords = [w for w in re.findall(r"[a-z0-9']+", q.lower()) if w not in _STOP and len(w) > 1]
    segs = _segments(rec)
    names = {v.lower(): k for k, v in (rec.get("speaker_names") or {}).items() if v}
    scored = []
    for s in segs:
        t = s["text"].lower()
        words = set(re.findall(r"[a-z0-9']+", t))
        hits = sum(1 for w in qwords if w in words or any(x.startswith(w[:5]) for x in words if len(w) >= 5))
        score = 60.0 * hits / max(1, len(qwords)) + 0.4 * fuzz.token_set_ratio(q.lower(), t)
        spk = s.get("speaker")
        if spk and any(n in q.lower() and names[n] == spk for n in names):
            score += 15
        scored.append((score, s))
    scored.sort(key=lambda x: (-x[0], x[1]["id"]))
    best = scored[0][0] if scored else 0
    return [{"id": s["id"], "score": round(sc, 1), "start": s.get("start"), "speaker": s.get("speaker"), "text": s["text"]}
            for sc, s in scored[:k] if sc >= max(25.0, 0.5 * best)]


def render_search(rec: dict, query: str, k: int = 8) -> str:
    c = _Ctx(rec, "ma")
    res = search(rec, query, k)
    if not (query or "").strip():
        return '<div class="ma"><p class="muted">Type a question or keywords.</p></div>'
    if not res:
        return f'<div class="ma"><p class="muted">Nothing in the transcript matches “{_e(query)}”.</p></div>'
    H = ['<div class="ma">', f'<p class="meta">Lines from the transcript that best match “{_e(query)}” '
         f'(verbatim; nothing is generated):</p>']
    for r in res:
        H.append(f'<div class="item">{c.spk_badge(r["speaker"])}“{_e(r["text"])}” {c.cite([r["id"]])}</div>')
    H.append("</div>")
    return "\n".join(H)


In [ ]:
%%writefile pipeline.py
"""One workflow: audio -> Stage 1 (faster-whisper + pyannote 3.1 + Parakeet cross-check) -> Stage 2 (refiner LLM)
-> Stage 3 (documenter LLM, verified) -> Stage 4 (.ics calendar + citation report) -> files.
Every stage writes its outputs into the run folder before the next one starts and can be resumed from the previous
stage's record. Models are released after each stage even when a stage fails. Model failures degrade (status fields
+ warnings); only unusable INPUT raises PipelineError."""
import copy
import json
import os
import shutil
import time
import zipfile
from dataclasses import asdict
from datetime import datetime
from pathlib import Path

import config
from config import CFG as STT_CFG, DIAR_CFG, REFINE_CFG, DOC_CFG, CAL_CFG
from errors import PipelineError
from stages import stt, refine, document, diarize, calendar_ics
import pipeline_stt
import report_html

OUTPUTS = ["raw_transcript.txt", "raw_transcript.json", "stage1_record.json", "refined_transcript.txt",
           "refined_transcript_numbered.txt", "edit_log.json", "stage2_record.json", "meeting_record.json",
           "meeting_record.md", "meeting_record.html", "meeting_calendar.ics", "verification_log.json", "run_summary.json"]


def _report(cb, stage, p, msg=""):
    if cb:
        try:
            cb(stage, p, msg)
        except Exception:
            pass


def free_all():
    """Release every model this process may hold (safe to call any time)."""
    for fn in (stt.free_gpu, diarize.free_diarizer, refine.free_llm, document.free_llm):
        try:
            fn()
        except Exception:
            pass
    try:
        from stages import asr2
        asr2.free()
    except Exception:
        pass


def new_run_dir(out_dir=None) -> Path:
    out_dir = Path(out_dir or config.outputs_dir())
    base = out_dir / f"run_{datetime.now():%Y%m%d_%H%M%S}"
    run_dir, k = base, 1
    while run_dir.exists():
        k += 1
        run_dir = base.with_name(f"{base.name}_{k}")
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir


def run_stage1(audio_path, run_dir, glossary=None, cfg=None, on_progress=None, diar_cfg=None, hf_token=None):
    cfg = cfg or STT_CFG
    try:
        raw = pipeline_stt.process(audio_path, glossary=glossary, cfg=cfg, diar_cfg=diar_cfg or DIAR_CFG, hf_token=hf_token,
                                   on_progress=lambda s, p: _report(on_progress, "stage1:" + s, p))
        saved = pipeline_stt.save_outputs(raw, str(run_dir), audio_path=audio_path, glossary=glossary)
    finally:
        stt.free_gpu(); diarize.free_diarizer()
    return raw, saved


def run_stage2(stage1_record, run_dir, glossary=None, cfg=None, models_dir=None, on_progress=None):
    cfg = copy.deepcopy(cfg or REFINE_CFG)
    try:
        r2, s1, dmg = refine.refine(stage1_record, cfg, models_dir, user_glossary=glossary,
                                    on_progress=lambda s, p: _report(on_progress, "stage2:" + s, p))
        refine.save_stage2(s1, r2, dmg, Path(run_dir), run_dir=Path(run_dir))
    finally:
        refine.free_llm()
    return r2, s1, dmg


def run_stage3(stage2_record, run_dir, cfg=None, models_dir=None, on_progress=None):
    cfg = copy.deepcopy(cfg or DOC_CFG)
    try:
        r3, s2 = document.document(stage2_record, cfg, models_dir,
                                   on_progress=lambda s, p: _report(on_progress, "stage3:" + s, p))
        _, rec = document.save_stage3(s2, r3, Path(run_dir), run_dir=Path(run_dir))
    finally:
        document.free_llm()
    return r3, rec


def finalize(run_dir, cal_cfg=None, speaker_names=None, on_progress=None) -> dict:
    """Stage 4 (no model): calendar + due dates + Markdown + citation HTML, from meeting_record.json.
    Re-runnable on its own (e.g. after renaming speakers or correcting the meeting date)."""
    run_dir = Path(run_dir)
    rec_path = run_dir / "meeting_record.json"
    rec = json.loads(rec_path.read_text(encoding="utf-8"))
    if speaker_names is not None:
        rec["speaker_names"] = {k: v for k, v in dict(speaker_names).items() if v}
    ics_path, cal = calendar_ics.add_calendar(rec, run_dir, cal_cfg or CAL_CFG)
    rec_path.write_text(json.dumps(rec, indent=2, ensure_ascii=False), encoding="utf-8")
    (run_dir / "meeting_record.md").write_text(document.render_markdown(rec), encoding="utf-8")
    (run_dir / "meeting_record.html").write_text(report_html.render_page(rec), encoding="utf-8")
    _report(on_progress, "stage4:saved", 1.0, f"calendar {cal.get('status')}: {cal.get('n_events', 0)} event(s), "
                                               f"{cal.get('resolved', 0)} due date(s)")
    return {"record": rec, "ics": ics_path, "calendar": cal}


def make_zip(run_dir) -> str:
    run_dir = Path(run_dir)
    zp = run_dir.with_suffix(".zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(run_dir.iterdir()):
            if f.is_file():
                z.write(f, f.name)
    return str(zp)


def run_all(audio_path=None, glossary=None, out_dir=None, models_dir=None, on_progress=None, resume_from=None,
            stt_cfg=None, diar_cfg=None, refine_cfg=None, doc_cfg=None, cal_cfg=None, speaker_names=None,
            hf_token=None) -> dict:
    """Full workflow. resume_from = a stage1_record.json or stage2_record.json to skip earlier stages.
    Returns {'run_dir', 'files', 'status', 'timings', 'summary', 'zip'}; raises PipelineError only for bad input."""
    t0 = time.time(); timings, status = {}, {}
    models_dir = models_dir or config.models_dir()
    stt_cfg = copy.deepcopy(stt_cfg or STT_CFG); diar_cfg = copy.deepcopy(diar_cfg or DIAR_CFG)
    run_dir = new_run_dir(out_dir)
    stage1_record = stage2_record = None
    if resume_from:
        d = json.loads(Path(resume_from).read_text(encoding="utf-8"))
        if "refined_transcript" in d:
            stage2_record = str(run_dir / "stage2_record.json"); shutil.copy(resume_from, stage2_record)
        elif "raw_transcript" in d:
            stage1_record = str(run_dir / "stage1_record.json"); shutil.copy(resume_from, stage1_record)
        else:
            raise PipelineError("resume_from must be a stage1_record.json or stage2_record.json")
    try:
        if stage1_record is None and stage2_record is None:
            t = time.time()
            raw, saved = run_stage1(audio_path, run_dir, glossary, stt_cfg, on_progress, diar_cfg, hf_token)
            stage1_record = saved["record"]; timings["stage1_s"] = round(time.time() - t, 1)
            status["stage1"] = "ok" if not saved["schema_problems"] else "schema_problems"
            status["diarization"] = (raw.get("diarization") or {}).get("status", "skipped")
            status["asr2"] = "ok" if raw.get("asr2") else ("off" if not stt_cfg.use_asr2 else "skipped")
            _report(on_progress, "stage1:saved", 1.0, f"{len(raw['segments'])} segments")
        if stage2_record is None:
            t = time.time()
            r2, s1, dmg = run_stage2(stage1_record, run_dir, glossary, refine_cfg, models_dir, on_progress)
            stage2_record = str(run_dir / "stage2_record.json"); timings["stage2_s"] = round(time.time() - t, 1)
            status["stage2"] = r2["status"]
            _report(on_progress, "stage2:saved", 1.0, f"{len(r2['edits_applied'])} edits applied, {len(r2['edits_rejected'])} rejected")
        t = time.time()
        r3, rec = run_stage3(stage2_record, run_dir, doc_cfg, models_dir, on_progress)
        timings["stage3_s"] = round(time.time() - t, 1); status["stage3"] = r3["status"]
        _report(on_progress, "stage3:saved", 1.0, f"{len(rec['decisions'])} decisions, {len(rec['action_items'])} action items")
        t = time.time()
        fin = finalize(run_dir, cal_cfg, speaker_names, on_progress)
        rec = fin["record"]; status["calendar"] = fin["calendar"].get("status"); timings["stage4_s"] = round(time.time() - t, 1)
    finally:
        free_all()
    timings["total_s"] = round(time.time() - t0, 1)
    meta = rec.get("meta", {})
    summary = {"run_dir": str(run_dir), "audio_file": os.path.basename(audio_path) if audio_path else meta.get("audio_file"),
               "status": status, "timings": timings, "models": meta.get("models"),
               "diarization": {k: (rec.get("diarization") or {}).get(k) for k in ("status", "reason", "n_speakers")},
               "calendar": {k: (rec.get("calendar") or {}).get(k) for k in ("status", "meeting_date", "meeting_date_source",
                                                                           "n_events", "resolved", "unresolved")},
               "warnings": {"stage1": meta.get("warnings", []), "stage2": meta.get("refiner_warnings", []),
                            "stage3": meta.get("documenter_warnings", [])},
               "settings": {"stt": asdict(stt_cfg), "diarization": asdict(diar_cfg), "refiner": asdict(refine_cfg or REFINE_CFG),
                            "documenter": asdict(doc_cfg or DOC_CFG), "calendar": asdict(cal_cfg or CAL_CFG)}}
    (run_dir / "run_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    files = {n: str(run_dir / n) for n in OUTPUTS if (run_dir / n).exists()}
    zp = make_zip(run_dir)
    return {"run_dir": str(run_dir), "files": files, "status": status, "timings": timings, "summary": summary,
            "zip": zp, "record": rec}


if __name__ == "__main__":
    import sys
    if len(sys.argv) < 2:
        print("Usage: python pipeline.py <audio_path> [glossary terms ...]"); sys.exit(1)
    try:
        out = run_all(sys.argv[1], glossary=sys.argv[2:] or None,
                      on_progress=lambda s, p, m="": print(f"{s:28s} {p:5.0%} {m}"))
        print(json.dumps({k: v for k, v in out["summary"].items() if k != "settings"}, indent=2))
    except PipelineError as e:
        print(f"[{e.code}] {e.user_message}"); sys.exit(2)


In [ ]:
%%writefile app.py
"""Gradio interface for the meeting assistant.

Tab 1  Process   upload -> full pipeline -> citation-grounded report (every item links to its transcript lines),
                 refined transcript with speakers, downloads (.zip, .ics, .md, .json, .html).
Tab 2  Search    ranked VERBATIM transcript lines for a question, each with its citation (nothing is generated).
'Update' re-runs only Stage 4 (no models): rename speakers, fix the meeting date / time zone, regenerate the .ics.
Every handler returns an error message instead of raising, so the app never shows a stack trace.
"""
import copy
import inspect
import json
import re
import traceback
from pathlib import Path

import config
import pipeline
import report_html
from errors import PipelineError

_APP = None


def _kw(fn, **kw):
    """Keep only the keyword arguments this Gradio version accepts."""
    try:
        params = inspect.signature(fn).parameters
        if any(p.kind == inspect.Parameter.VAR_KEYWORD for p in params.values()):
            return kw
        return {k: v for k, v in kw.items() if k in params}
    except (TypeError, ValueError):
        return kw


def _mk(cls, **kw):
    return cls(**_kw(cls.__init__, **kw))


def parse_speaker_names(text: str) -> dict:
    """'SPEAKER_00=Priya, SPEAKER_01 = Rahul' (commas, semicolons or new lines) -> {'SPEAKER_00': 'Priya', ...}.
    Shorthand '0=Priya' or 'S1=Rahul' is accepted too."""
    out = {}
    for part in re.split(r"[,;\n]+", text or ""):
        if "=" not in part:
            continue
        k, v = (x.strip() for x in part.split("=", 1))
        if not k or not v:
            continue
        m = re.fullmatch(r"(?:speaker[_ ]?|s)?(\d+)", k, flags=re.I)
        out[f"SPEAKER_{int(m.group(1)):02d}" if m else k.upper()] = v
    return out


def _parse_glossary(text: str) -> list:
    return [g.strip() for g in re.split(r"[,;\n]+", text or "") if g.strip()]


def _files(run_dir: str) -> list:
    rd = Path(run_dir)
    order = ["meeting_record.html", "meeting_calendar.ics", "meeting_record.md", "meeting_record.json",
             "refined_transcript_numbered.txt", "raw_transcript.txt", "edit_log.json", "verification_log.json"]
    out = [str(rd.with_suffix(".zip"))] if rd.with_suffix(".zip").exists() else []
    out += [str(rd / n) for n in order if (rd / n).exists()]
    return out


def _status_md(summary: dict) -> str:
    st = summary.get("status", {}); cal = summary.get("calendar", {}) or {}; di = summary.get("diarization", {}) or {}
    lines = [f"**Run:** `{Path(summary.get('run_dir', '')).name}` · **time:** {summary.get('timings', {}).get('total_s', '?')} s",
             "**Stages:** " + " · ".join(f"{k}: {v}" for k, v in st.items())]
    if di.get("status") != "ok" and di.get("reason"):
        lines.append(f"**Diarization skipped:** {di.get('reason')}")
    if cal:
        lines.append(f"**Calendar:** {cal.get('n_events', 0)} event(s), {cal.get('resolved', 0)} due date(s) resolved, "
                     f"{cal.get('unresolved', 0)} without a placeable deadline · meeting date {cal.get('meeting_date')} "
                     f"({cal.get('meeting_date_source')})")
    for stage, ws in (summary.get("warnings") or {}).items():
        for w in ws[:4]:
            lines.append(f"- _{stage}:_ {w}")
    return "\n\n".join(lines)


def _cal_cfg(meeting_date, meeting_time, tz):
    c = copy.deepcopy(config.CAL_CFG)
    config.apply_overrides(c, {"meeting_date": (meeting_date or "").strip(), "meeting_time": (meeting_time or "").strip(),
                               "timezone": (tz or "").strip()})
    return c


def _load_record(run_dir):
    return json.loads((Path(run_dir) / "meeting_record.json").read_text(encoding="utf-8"))


def process(audio, glossary_text, context, meeting_date, meeting_time, tz, num_speakers, use_diar, use_asr2,
            speaker_names_text, progress=None):
    if not audio:
        return ("<p>Please upload a recording.</p>", "", [], "", "**Nothing to do:** no file uploaded.")

    def cb(stage, p, msg=""):
        try:
            if progress is not None:
                progress(min(1.0, max(0.0, float(p))), desc=f"{stage} {msg}".strip())
        except Exception:
            pass

    stt_cfg = copy.deepcopy(config.CFG); config.apply_overrides(stt_cfg, {"use_asr2": bool(use_asr2)})
    diar_cfg = copy.deepcopy(config.DIAR_CFG)
    config.apply_overrides(diar_cfg, {"enabled": bool(use_diar), "num_speakers": int(num_speakers or 0)})
    ref_cfg = copy.deepcopy(config.REFINE_CFG); doc_cfg = copy.deepcopy(config.DOC_CFG)
    if (context or "").strip():
        ref_cfg.meeting_context = context.strip(); doc_cfg.meeting_context = context.strip()
    try:
        res = pipeline.run_all(audio_path=audio, glossary=_parse_glossary(glossary_text) or None,
                               on_progress=cb, stt_cfg=stt_cfg, diar_cfg=diar_cfg, refine_cfg=ref_cfg, doc_cfg=doc_cfg,
                               cal_cfg=_cal_cfg(meeting_date, meeting_time, tz),
                               speaker_names=parse_speaker_names(speaker_names_text))
    except PipelineError as e:
        return (f"<p><b>Cannot process this file [{e.code}]:</b> {report_html._e(e.user_message)}</p>", "", [], "",
                f"**Cannot process this file:** {e.user_message}")
    except Exception as e:
        pipeline.free_all()
        tb = traceback.format_exc(limit=3)
        return (f"<p><b>Unexpected error:</b> {report_html._e(type(e).__name__)}: {report_html._e(str(e)[:400])}</p>"
                f"<pre style='white-space:pre-wrap;font-size:.75rem'>{report_html._e(tb)}</pre>", "", [], "",
                "**Unexpected error** — details in the report panel.")
    run_dir = res["run_dir"]
    rec = res["record"]
    numbered = Path(run_dir) / "refined_transcript_numbered.txt"
    transcript = numbered.read_text(encoding="utf-8") if numbered.exists() else ""
    return (report_html.with_style(report_html.render_fragment(rec)), transcript, _files(run_dir), run_dir,
            _status_md(res["summary"]))


def update(run_dir, speaker_names_text, meeting_date, meeting_time, tz):
    if not run_dir or not (Path(run_dir) / "meeting_record.json").exists():
        return "<p>Process a recording first.</p>", [], "**Nothing to update.**"
    try:
        fin = pipeline.finalize(run_dir, _cal_cfg(meeting_date, meeting_time, tz), parse_speaker_names(speaker_names_text))
        pipeline.make_zip(run_dir)
        cal = fin["calendar"]
        msg = (f"**Updated.** Calendar {cal.get('status')}: {cal.get('n_events', 0)} event(s), {cal.get('resolved', 0)} due "
               f"date(s); meeting date {cal.get('meeting_date')} ({cal.get('meeting_date_source')}).")
        return report_html.with_style(report_html.render_fragment(fin["record"])), _files(run_dir), msg
    except Exception as e:
        return f"<p><b>Update failed:</b> {report_html._e(type(e).__name__)}: {report_html._e(str(e)[:300])}</p>", [], "**Update failed.**"


def ask(run_dir, query):
    if not run_dir or not (Path(run_dir) / "meeting_record.json").exists():
        return "<p>Process a recording first.</p>"
    try:
        return report_html.with_style(report_html.render_search(_load_record(run_dir), query))
    except Exception as e:
        return f"<p>Search failed: {report_html._e(type(e).__name__)}: {report_html._e(str(e)[:200])}</p>"


def build_app():
    import gradio as gr
    defaults = config.CAL_CFG
    blocks_kw = _kw(gr.Blocks.__init__, title="AI Meeting Assistant")
    with gr.Blocks(**blocks_kw) as demo:
        gr.Markdown("# 🎙️ AI Meeting Assistant\nTranscribe → speakers → verified minutes, decisions and action items → "
                    "calendar. Every item cites the transcript lines it came from; click a citation to jump to the line.")
        run_state = gr.State("")
        with gr.Tab("Process"):
            with gr.Row():
                with gr.Column(scale=1):
                    audio_in = _mk(gr.Audio, sources=["upload"], type="filepath", label="Meeting recording")
                    glossary_in = _mk(gr.Textbox, label="Glossary (names, products, jargon; comma-separated)", lines=2,
                                      placeholder="spellings you want enforced, e.g. attendee names and product names")
                    context_in = _mk(gr.Textbox, label="Meeting context (optional; detected automatically if empty)")
                    with gr.Row():
                        date_in = _mk(gr.Textbox, label="Meeting date (YYYY-MM-DD; optional)", value=defaults.meeting_date)
                        time_in = _mk(gr.Textbox, label="Start time (HH:MM; optional)", value=defaults.meeting_time)
                    tz_in = _mk(gr.Textbox, label="Time zone (IANA, e.g. Asia/Kolkata; empty = system)", value=defaults.timezone)
                    with gr.Row():
                        spk_n = _mk(gr.Number, label="Number of speakers (0 = auto)", value=config.DIAR_CFG.num_speakers, precision=0)
                        diar_in = _mk(gr.Checkbox, label="Speaker diarization (pyannote 3.1)", value=config.DIAR_CFG.enabled)
                        asr2_in = _mk(gr.Checkbox, label="Parakeet cross-check", value=config.CFG.use_asr2)
                    names_in = _mk(gr.Textbox, label="Speaker names (optional; e.g. SPEAKER_00=Priya, SPEAKER_01=Rahul)", lines=2)
                    with gr.Row():
                        go = _mk(gr.Button, value="Process meeting", variant="primary")
                        upd = _mk(gr.Button, value="Update names / date / calendar")
                    status = gr.Markdown("")
                with gr.Column(scale=2):
                    report = gr.HTML("<p>The report appears here.</p>")
                    files = _mk(gr.File, label="Downloads (.zip, .ics, .html, .md, .json)", file_count="multiple")
                    transcript = _mk(gr.Textbox, label="Refined transcript (numbered, with speakers)", lines=10)
        with gr.Tab("Search the meeting"):
            q = _mk(gr.Textbox, label="Question or keywords", placeholder="who owns the runbook? when is the release?")
            ask_btn = _mk(gr.Button, value="Find evidence", variant="primary")
            answer = gr.HTML("")
        def _process_ui(audio, glossary_text, context, meeting_date, meeting_time, tz, num_speakers, use_diar, use_asr2,
                        speaker_names_text, progress=gr.Progress()):
            return process(audio, glossary_text, context, meeting_date, meeting_time, tz, num_speakers, use_diar, use_asr2,
                           speaker_names_text, progress=progress)

        go.click(fn=_process_ui, inputs=[audio_in, glossary_in, context_in, date_in, time_in, tz_in, spk_n, diar_in, asr2_in, names_in],
                 outputs=[report, transcript, files, run_state, status])
        upd.click(fn=update, inputs=[run_state, names_in, date_in, time_in, tz_in], outputs=[report, files, status])
        ask_btn.click(fn=ask, inputs=[run_state, q], outputs=[answer])
        q.submit(fn=ask, inputs=[run_state, q], outputs=[answer])
    return demo


def launch(share: bool = True, server_port: int | None = None, inline: bool = False):
    """(Re)launch the app. Closes a previous instance first, so the cell can be re-run any number of times."""
    global _APP
    import gradio as gr
    if _APP is not None:
        try:
            _APP.close()
        except Exception:
            pass
        _APP = None
    try:
        gr.close_all()
    except Exception:
        pass
    demo = build_app()
    qkw = _kw(demo.queue, default_concurrency_limit=1, max_size=8)
    demo.queue(**qkw)
    css_theme = {}
    if "theme" in inspect.signature(demo.launch).parameters:
        css_theme["theme"] = gr.themes.Soft()
    lkw = _kw(demo.launch, share=share, inline=inline, prevent_thread_lock=True, show_error=True,
              server_port=server_port, quiet=True, **css_theme)
    demo.launch(**lkw)
    _APP = demo
    return demo


def close():
    global _APP
    if _APP is not None:
        try:
            _APP.close()
        finally:
            _APP = None


In [ ]:
# 3. Load the modules (fresh every time this cell runs), apply the compatibility shims, run the model-free self-tests
#    and check what this runtime can do. Nothing here raises: problems are reported and the affected step degrades.
SMOKE_TEST_WHISPER = True  #@param {type:"boolean"}

import importlib, sys, time, json, shutil
import numpy as np

MODULES = ["config", "errors", "compat", "stages.textutil", "stages.ingest", "stages.stt", "stages.diarize", "stages.asr2",
           "pipeline_stt", "stages.refine", "stages.document", "stages.calendar_ics", "report_html", "pipeline", "app"]
# release models held by a previous import before replacing the modules (otherwise GPU memory leaks on re-runs)
if "pipeline" in sys.modules:
    try:
        sys.modules["pipeline"].free_all()
    except Exception:
        pass
if "app" in sys.modules:
    try:
        sys.modules["app"].close()
    except Exception:
        pass
for m in MODULES + ["stages"]:
    sys.modules.pop(m, None)
importlib.invalidate_caches()

import config, errors, compat
print("compatibility shims:", compat.apply_all())
import pipeline_stt, pipeline, report_html, app
from config import CFG as STT_CFG, DIAR_CFG, REFINE_CFG, DOC_CFG, CAL_CFG, SR
from errors import PipelineError
from stages import ingest, stt, diarize, refine, document, calendar_ics

print(f"STT: {STT_CFG.model_size} on {STT_CFG.device} ({STT_CFG.compute_type}) | diarization: {DIAR_CFG.model_id} "
      f"(enabled={DIAR_CFG.enabled}) | cross-check: {STT_CFG.asr2_model} (enabled={STT_CFG.use_asr2})")
print(f"Refiner: {REFINE_CFG.model_id} | Documenter: {DOC_CFG.model_id}\n")

SELF_TESTS = {}
for name, fn in (("Stage 2 validators", refine.self_test), ("Stage 3 verifier", document.self_test),
                 ("diarization assignment", diarize.self_test), ("calendar / .ics", calendar_ics.self_test)):
    print(f"=== {name} self-test (no model) ===")
    try:
        SELF_TESTS[name] = fn()
    except Exception as e:
        SELF_TESTS[name] = -1
        print(f"self-test crashed: {type(e).__name__}: {e}")
    print()
bad = {k: v for k, v in SELF_TESTS.items() if v}
print("SELF-TESTS:", "all passed" if not bad else f"PROBLEMS {bad} - results of the affected stage are not trustworthy")

# ---- what this runtime can do (diagnostics only)
CAPS = {}
try:
    import ctranslate2
    CAPS["gpu_for_whisper"] = ctranslate2.get_cuda_device_count() > 0
except Exception:
    CAPS["gpu_for_whisper"] = False
try:
    compat.import_pyannote(); CAPS["pyannote"] = "importable"
except Exception as e:
    CAPS["pyannote"] = f"NOT importable ({type(e).__name__}: {str(e)[:120]})"
CAPS["hf_token"] = bool(config.get_hf_token())
if CAPS["hf_token"]:
    try:
        import huggingface_hub
        if hasattr(huggingface_hub, "auth_check"):
            for repo in (DIAR_CFG.model_id, "pyannote/segmentation-3.0"):
                try:
                    huggingface_hub.auth_check(repo, token=config.get_hf_token()); CAPS[repo] = "access OK"
                except Exception as e:
                    CAPS[repo] = f"NO ACCESS ({type(e).__name__}) -> accept the conditions at https://hf.co/{repo}"
    except Exception:
        pass
try:
    import onnx_asr  # noqa: F401
    CAPS["parakeet_onnx"] = "importable"
except Exception as e:
    CAPS["parakeet_onnx"] = f"NOT importable ({type(e).__name__})"
try:
    import gradio
    CAPS["gradio"] = gradio.__version__
except Exception as e:
    CAPS["gradio"] = f"NOT importable ({type(e).__name__})"
for k, v in CAPS.items():
    print(f"  {k:<36} {v}")

if SMOKE_TEST_WHISPER:
    t0 = time.time()
    try:
        m = stt.get_model(STT_CFG)
        segs, _ = m.transcribe(np.zeros(SR * 2, dtype="float32"), language=STT_CFG.language); list(segs)
        print(f"\nOK: speech model {STT_CFG.model_size} loaded and ran a test inference in {time.time() - t0:.0f}s")
    except Exception as e:
        print(f"\nspeech model not usable here: {type(e).__name__}: {str(e)[:300]}")
        print("-> on a CPU runtime set MA_STT_MODEL_SIZE=small (or config.apply_overrides(STT_CFG, {'model_size': 'small'})).")
    finally:
        stt.free_gpu()


## Inputs

The recording is the only required input. Everything else is optional and has a documented fallback. Re-run this cell to start a new run folder.

In [ ]:
# 4. Inputs — every value here is optional except the recording. Nothing about a meeting is hard-coded anywhere.
AUDIO_PATH = ""        #@param {type:"string"}
#@markdown ↑ path to a recording (e.g. on Drive). Empty → an upload button appears (Colab).
GLOSSARY = ""          #@param {type:"string"}
#@markdown ↑ comma-separated names / products / jargon with the spelling you want (feeds Whisper, Stage 2 and Stage 3).
MEETING_CONTEXT = ""   #@param {type:"string"}
#@markdown ↑ e.g. "hospital operations review"; empty → detected from the transcript.
MEETING_DATE = ""      #@param {type:"string"}
MEETING_TIME = ""      #@param {type:"string"}
TIMEZONE = ""          #@param {type:"string"}
#@markdown ↑ YYYY-MM-DD, HH:MM (24 h) and an IANA zone such as Asia/Kolkata. Empty → recording metadata → date in the
#@markdown file name → processing date (the source is recorded), and the system time zone.
NUM_SPEAKERS = 0       #@param {type:"integer"}
USE_DIARIZATION = True #@param {type:"boolean"}
USE_PARAKEET_CROSS_CHECK = True  #@param {type:"boolean"}
SPEAKER_NAMES = ""     #@param {type:"string"}
#@markdown ↑ optional display names, e.g. `SPEAKER_00=Priya, SPEAKER_01=Rahul` (can also be set later, Stage 4).
REFERENCE_TXT = ""     #@param {type:"string"}
#@markdown ↑ optional human transcript of the same recording, for WER before/after refinement.
RESUME_FROM = ""       #@param {type:"string"}
#@markdown ↑ optional stage1_record.json (skips Stage 1) or stage2_record.json (skips Stages 1-2) of an earlier run.
DOWNLOAD_FILES = False #@param {type:"boolean"}

import copy, os
from pathlib import Path

GLOSSARY_LIST = [g.strip() for g in GLOSSARY.split(",") if g.strip()]
RUN_STT = copy.deepcopy(STT_CFG); config.apply_overrides(RUN_STT, {"use_asr2": USE_PARAKEET_CROSS_CHECK})
RUN_DIAR = copy.deepcopy(DIAR_CFG); config.apply_overrides(RUN_DIAR, {"enabled": USE_DIARIZATION, "num_speakers": NUM_SPEAKERS})
RUN_REF = copy.deepcopy(REFINE_CFG); RUN_DOC = copy.deepcopy(DOC_CFG); RUN_CAL = copy.deepcopy(CAL_CFG)
if MEETING_CONTEXT.strip():
    RUN_REF.meeting_context = RUN_DOC.meeting_context = MEETING_CONTEXT.strip()
config.apply_overrides(RUN_CAL, {"meeting_date": MEETING_DATE.strip(), "meeting_time": MEETING_TIME.strip(),
                                 "timezone": TIMEZONE.strip()})
SPEAKER_NAME_MAP = app.parse_speaker_names(SPEAKER_NAMES)

AUDIO = AUDIO_PATH.strip() or None
RESUME = RESUME_FROM.strip() or None
if RESUME and not Path(RESUME).exists():
    print("RESUME_FROM does not exist, ignoring:", RESUME); RESUME = None
if AUDIO and not Path(AUDIO).exists():
    print("AUDIO_PATH does not exist:", AUDIO); AUDIO = None
if AUDIO is None and RESUME is None:
    try:
        from google.colab import files as _colab_files
        up = _colab_files.upload()
        if up:
            name = next(iter(up)); AUDIO = str(CLIPS_DIR / os.path.basename(name))
            Path(AUDIO).write_bytes(up[name])
    except Exception as e:
        print("No recording given (set AUDIO_PATH, or upload in Colab):", type(e).__name__)

RUN_DIR = pipeline.new_run_dir(OUT_DIR)
RUN = {"stage1_record": None, "stage2_record": None, "record": None}
if RESUME:
    import json as _json
    _d = _json.loads(Path(RESUME).read_text(encoding="utf-8"))
    key = "stage2_record" if "refined_transcript" in _d else "stage1_record" if "raw_transcript" in _d else None
    if key:
        RUN[key] = str(RUN_DIR / f"{key}.json"); shutil.copy(RESUME, RUN[key])
    else:
        print("RESUME_FROM is not a stage1/stage2 record, ignoring.")
READY = bool(AUDIO or RUN["stage1_record"] or RUN["stage2_record"])


def offer_download(*paths):
    "List the files; offer them for download in Colab if DOWNLOAD_FILES is ticked."
    for p in paths:
        p = Path(p)
        if p.exists():
            print(f"  {p.name:<34} {p.stat().st_size / 1024:8.1f} KB")
    if DOWNLOAD_FILES:
        try:
            from google.colab import files as _f
            for p in paths:
                if Path(p).exists():
                    _f.download(str(p))
        except Exception:
            pass


def progress(stage, p, msg=""):
    print(f"\r  {stage:<28} {p:5.0%} {msg}", end="\n" if p >= 1 else "")


print("audio:", AUDIO, "| glossary:", GLOSSARY_LIST, "| resume:", RESUME, "| run folder:", RUN_DIR, "| ready:", READY)


## Stage 1 — Transcribe, diarize, cross-check

Validates the file with a clear message, converts to 16 kHz mono, runs faster-whisper behind the hallucination firewall, assigns speakers word by word with pyannote 3.1, and aligns an independent Parakeet transcript to mark where the two recognisers disagree.

In [ ]:
# 5. Stage 1 — validate, transcribe (faster-whisper + hallucination firewall), diarize (pyannote 3.1), cross-check (Parakeet)
if not READY:
    print("Nothing to process: set AUDIO_PATH / upload a recording in the Inputs cell.")
elif RUN["stage1_record"] or RUN["stage2_record"]:
    print("Stage 1 skipped (resuming from", RUN["stage1_record"] or RUN["stage2_record"], ")")
else:
    t0 = time.time()
    try:
        raw, saved = pipeline.run_stage1(AUDIO, RUN_DIR, GLOSSARY_LIST or None, RUN_STT, on_progress=progress, diar_cfg=RUN_DIAR)
        RUN["stage1_record"] = saved["record"]
        d = raw.get("diarization") or {}
        a2 = raw.get("asr2") or {}
        print(f"\n{len(raw['segments'])} lines | {len(raw['text'].split())} words | {time.time() - t0:.0f}s | flags {raw['flags']}")
        print(f"speakers: {d.get('n_speakers', 0) if d.get('status') == 'ok' else 'not diarized'}"
              + (f" ({d.get('reason')})" if d.get("status") != "ok" else ""))
        if a2:
            print(f"Parakeet cross-check: {100 * a2.get('disagreement_rate', 0):.1f}% of Whisper words differ, "
                  f"{a2.get('critical_disagreements', 0)} on numbers/negations, {a2.get('seconds')}s")
        for w in raw["warnings"]:
            print("WARNING:", w)
        print("\n--- RAW TRANSCRIPT (numbered lines) ---")
        print("\n".join(raw["lines"][:400]) + ("\n..." if len(raw["lines"]) > 400 else ""))
        crit = [(s["id"], s["critical_low_conf"]) for s in raw["segments"] if s.get("critical_low_conf")]
        if crit:
            print("\nnumbers/negations/commitments Whisper was unsure about (never auto-edited):", crit[:30])
        if raw["dropped"]:
            print("\nsegments removed by the firewall:", [(x["drop_reason"], x["text"][:60]) for x in raw["dropped"]][:30])
        print("\nStage 1 files:")
        offer_download(saved["txt"], saved["json"], saved["record"])
    except PipelineError as e:
        print(f"Stage 1 stopped [{e.code}]: {e.user_message}")
        pipeline.free_all()
    except Exception as e:
        print(f"Stage 1 failed: {type(e).__name__}: {str(e)[:400]}")
        pipeline.free_all()


## Stage 2 — Refine terminology (LLM #1)

Domain discovery → chunked edit proposals (with the Parakeet alternatives shown to the model) → deterministic validators incl. LM rescoring → propagation → canonical casing → circuit breaker.

In [ ]:
# 6. Stage 2 — refine terminology (LLM #1 proposes, deterministic validators decide)
if RUN["stage2_record"]:
    print("Stage 2 skipped (resuming from a Stage 2 record).")
elif not RUN["stage1_record"]:
    print("Stage 2 skipped: no Stage 1 record (see the Stage 1 cell).")
else:
    t0 = time.time()
    try:
        R2, S1, DMG = pipeline.run_stage2(RUN["stage1_record"], RUN_DIR, GLOSSARY_LIST or None, RUN_REF, MODELS_DIR,
                                          on_progress=progress)
        RUN["stage2_record"] = str(RUN_DIR / "stage2_record.json")
        print(f"\n{R2['meta']['refiner']} | context: {R2['meta']['meeting_context']!r} | {R2['meta']['n_chunks']} chunk(s) | "
              f"{time.time() - t0:.0f}s | applied {len(R2['edits_applied'])} | rejected {len(R2['edits_rejected'])} | "
              f"edit ratio {100 * R2['edit_ratio']:.2f}% | damage {DMG['damage_count']} | status {R2['status']}\n")
        refine.print_report(R2, S1["segments"], RUN_REF)
        print("\nStage 2 files:")
        offer_download(RUN_DIR / "refined_transcript.txt", RUN_DIR / "refined_transcript_numbered.txt",
                       RUN_DIR / "edit_log.json", RUN["stage2_record"])
    except Exception as e:
        print(f"Stage 2 failed: {type(e).__name__}: {str(e)[:400]}")
        pipeline.free_all()


## Stage 3 — Minutes, decisions, action items (LLM #2)

Evidence-quoted proposals → verifier (quote verbatim in 1-3 consecutive lines incl. one neighbouring line, negation context, statement grounded in the quote, owner/deadline next to the quote, first-person rule, entity screen).

In [ ]:
# 7. Stage 3 — minutes, decisions, action items (LLM #2 proposes with evidence quotes, the verifier decides)
if not RUN["stage2_record"]:
    print("Stage 3 skipped: no Stage 2 record.")
else:
    t0 = time.time()
    try:
        R3, RECORD = pipeline.run_stage3(RUN["stage2_record"], RUN_DIR, RUN_DOC, MODELS_DIR, on_progress=progress)
        RUN["record"] = str(RUN_DIR / "meeting_record.json")
        m = R3["meta"]
        print(f"\n{m['documenter']} | {m['n_chunks']} chunk(s) | {time.time() - t0:.0f}s | status {R3['status']} | "
              f"decisions {len(R3['decisions'])} | proposals {len(R3['proposals_not_adopted'])} | tentative {len(R3['tentative'])} | "
              f"questions {len(R3['open_questions'])} | actions {len(R3['action_items'])} | dropped {m['n_dropped']} | "
              f"unspecified {m['n_adjusted']} | screened {m['n_screened']}\n")
        document.print_report(R3)
    except Exception as e:
        print(f"Stage 3 failed: {type(e).__name__}: {str(e)[:400]}")
        pipeline.free_all()


## Stage 4 — Calendar (.ics) and citation-grounded report

No model. Due dates are computed only from deadlines Stage 3 verified as spoken, relative to the meeting date; the rule used is stored with each date. A deadline that cannot be placed on a calendar gets no date.

In [ ]:
# 8. Stage 4 — calendar (.ics), due dates, Markdown and the citation-grounded HTML report (no model; re-run freely,
#    e.g. after filling SPEAKER_NAMES or MEETING_DATE in the Inputs cell — then run Inputs' config lines again or edit below)
from IPython.display import HTML, display
if not RUN["record"]:
    print("Stage 4 skipped: no meeting record (see the Stage 3 cell).")
else:
    try:
        FIN = pipeline.finalize(RUN_DIR, RUN_CAL, SPEAKER_NAME_MAP or None, on_progress=progress)
        cal = FIN["calendar"]
        print(f"\ncalendar: {cal.get('status')} | meeting date {cal.get('meeting_date')} ({cal.get('meeting_date_source')}) | "
              f"time zone {cal.get('timezone')} | {cal.get('n_events', 0)} event(s), {cal.get('n_todos', 0)} to-do(s) | "
              f"due dates resolved {cal.get('resolved', 0)}, not placeable {cal.get('unresolved', 0)}")
        for it in cal.get("items", []):
            d = it.get("due") or {}
            print(f"  - {it['task'][:60]:<60} deadline {it.get('deadline')!r:<22} -> {d.get('date', '—')} {d.get('time') or ''}"
                  f"  {('[' + d['rule'] + ']') if d else ''}")
        for p in cal.get("problems", []) or []:
            print("  ics problem:", p)
        display(HTML(report_html.with_style(report_html.render_fragment(FIN["record"]))))
        print("\nStage 3-4 files:")
        offer_download(RUN_DIR / "meeting_record.json", RUN_DIR / "meeting_record.md", RUN_DIR / "meeting_record.html",
                       RUN_DIR / "meeting_calendar.ics", RUN_DIR / "verification_log.json")
    except Exception as e:
        print(f"Stage 4 failed: {type(e).__name__}: {str(e)[:400]}")


## Evaluation

In [ ]:
# 9. Evaluation — WER raw -> refined against REFERENCE_TXT; Stage 3 precision/recall against GOLD (fill it in)
GOLD = {"decisions": [], "not_decisions": [], "action_items": []}   # action items: {"task", "owner", "deadline"}
if not RUN["record"] or not Path(RUN["record"]).exists():
    print("Evaluation skipped: no meeting record.")
else:
    try:
        rec = json.loads(Path(RUN["record"]).read_text(encoding="utf-8"))
        raw_text = (rec.get("raw_transcript") or {}).get("text", ""); ref_text = (rec.get("refined_transcript") or {}).get("text", "")
        if REFERENCE_TXT.strip() and Path(REFERENCE_TXT.strip()).exists():
            gold_txt = Path(REFERENCE_TXT.strip()).read_text(encoding="utf-8")
            w_raw, w_ref = refine.wer(gold_txt, raw_text), refine.wer(gold_txt, ref_text)
            print(f"WER raw -> refined : {100 * w_raw:.2f}% -> {100 * w_ref:.2f}%  "
                  f"({'better' if w_ref < w_raw else 'no change' if w_ref == w_raw else 'WORSE'})")
        else:
            print("REFERENCE_TXT not set - WER before/after skipped.")
        rt = rec.get("refined_transcript") or {}
        print(f"Stage 2: {len(rt.get('edits', []))} edits applied, {len(rt.get('rejected_edits', []))} rejected, "
              f"edit ratio {100 * float(rt.get('edit_ratio') or 0):.2f}%, damage {rt.get('damage_count')}, status {rt.get('status')}")
        if any(GOLD.values()):
            from rapidfuzz import fuzz
            _n = document._norm; _m = lambda a, b: fuzz.partial_ratio(_n(a), _n(b)) >= 80
            dec_txt = [d["decision"] + " " + d["evidence_quote"] for d in rec["decisions"]]
            tp = sum(any(_m(g, d) for d in dec_txt) for g in GOLD["decisions"])
            fp = sum(any(_m(g, d) for d in dec_txt) for g in GOLD["not_decisions"])
            print(f"decisions: recall {tp}/{len(GOLD['decisions'])} | precision {tp}/{len(dec_txt)} | "
                  f"proposals reported as decisions: {fp} (target 0)")
            hit = inv_o = inv_d = 0
            for g in GOLD["action_items"]:
                f = [a for a in rec["action_items"] if _m(g["task"], a["task"] + " " + a["evidence_quote"])]
                if f:
                    hit += 1
                    inv_o += (g.get("owner") == "unspecified" and f[0]["owner"] != "unspecified")
                    inv_d += (g.get("deadline") == "unspecified" and f[0]["deadline"] != "unspecified")
            print(f"action items: recall {hit}/{len(GOLD['action_items'])} | precision {hit}/{len(rec['action_items'])} | "
                  f"invented owners {inv_o} | invented deadlines {inv_d} (targets 0)")
        print("Stage 3 verifier:", (rec.get("meta") or {}).get("documenter_verification", {}))
        print("Diarization:", {k: (rec.get("diarization") or {}).get(k) for k in ("status", "n_speakers", "reason")})
        a2 = rec.get("asr2") or {}
        if a2:
            print(f"Parakeet cross-check: disagreement {100 * float(a2.get('disagreement_rate') or 0):.1f}% "
                  f"({a2.get('critical_disagreements')} critical)")
    except Exception as e:
        print(f"Evaluation failed: {type(e).__name__}: {str(e)[:300]}")


## Export

In [ ]:
# 10. Export — run summary (settings, models, status, timings) + one zip with every output and the source modules
import subprocess, zipfile
if not any(RUN_DIR.iterdir()):
    print("Nothing to export yet.")
else:
    try:
        rec = json.loads((RUN_DIR / "meeting_record.json").read_text(encoding="utf-8")) if (RUN_DIR / "meeting_record.json").exists() else {}
        meta = rec.get("meta", {})
        summary = {"run_dir": str(RUN_DIR), "audio_file": os.path.basename(AUDIO) if AUDIO else meta.get("audio_file"),
                   "models": meta.get("models"),
                   "status": {"stage2": (rec.get("refined_transcript") or {}).get("status"), "stage3": rec.get("status"),
                              "diarization": (rec.get("diarization") or {}).get("status"),
                              "calendar": (rec.get("calendar") or {}).get("status")},
                   "warnings": {"stage1": meta.get("warnings", []), "stage2": meta.get("refiner_warnings", []),
                                "stage3": meta.get("documenter_warnings", [])},
                   "settings": {"stt": vars(RUN_STT), "diarization": vars(RUN_DIAR), "refiner": vars(RUN_REF),
                                "documenter": vars(RUN_DOC), "calendar": vars(RUN_CAL)}}
        (RUN_DIR / "run_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
        print(json.dumps({k: v for k, v in summary.items() if k != "settings"}, indent=2, default=str))
        pins = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
        keep = ("faster-whisper", "ctranslate2", "transformers", "torch", "torchaudio", "accelerate", "bitsandbytes",
                "pyannote.audio", "huggingface-hub", "huggingface_hub", "onnx-asr", "onnx_asr", "onnxruntime", "gradio",
                "rapidfuzz", "jellyfish", "wordfreq", "spacy", "numpy")
        pins = "\n".join(l for l in pins.splitlines() if l.lower().split("==")[0] in keep)
        (RUN_DIR / "versions.txt").write_text(pins, encoding="utf-8"); print(pins)
        zip_path = OUT_DIR / f"{RUN_DIR.name}.zip"
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
            for f in sorted(RUN_DIR.iterdir()):
                if f.is_file():
                    z.write(f, f.name)
            for f in ["config.py", "errors.py", "compat.py", "pipeline_stt.py", "pipeline.py", "report_html.py", "app.py"]:
                if Path(f).exists():
                    z.write(f, "src/" + f)
            for f in sorted(Path("stages").glob("*.py")):
                z.write(f, "src/" + str(f))
        print("\nzip:", zip_path)
        offer_download(zip_path)
    except Exception as e:
        print(f"Export failed: {type(e).__name__}: {str(e)[:300]}")
# One-call alternative (what the UI uses):
# result = pipeline.run_all(AUDIO, glossary=GLOSSARY_LIST, stt_cfg=RUN_STT, diar_cfg=RUN_DIAR, refine_cfg=RUN_REF,
#                           doc_cfg=RUN_DOC, cal_cfg=RUN_CAL, speaker_names=SPEAKER_NAME_MAP, on_progress=progress)


## Web app

In [ ]:
# 11. Web app (Gradio) — citation-grounded report, transcript with speakers, search with verbatim cited lines, .ics download.
#     Re-running this cell closes the previous app first, so it can be run any number of times.
LAUNCH_UI = True   #@param {type:"boolean"}
SHARE_LINK = True  #@param {type:"boolean"}
if LAUNCH_UI:
    try:
        DEMO = app.launch(share=SHARE_LINK and IN_COLAB, inline=False)
        print("App running:", getattr(DEMO, "share_url", None) or getattr(DEMO, "local_url", None) or "see the link above")
    except Exception as e:
        print(f"UI not started: {type(e).__name__}: {str(e)[:300]}")
else:
    print("LAUNCH_UI is off.")


## Optional: multi-speaker evaluation clip

In [ ]:
# 12. (Optional) build a multi-speaker evaluation clip + reference transcript from the AMI meeting corpus.
BUILD_AMI_CLIP = False  #@param {type:"boolean"}
AMI_TURNS = 20          #@param {type:"integer"}
if BUILD_AMI_CLIP:
    import subprocess
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], capture_output=True)
        from datasets import load_dataset
        import soundfile as sf
        ds = load_dataset("edinburghcstr/ami", "ihm", split="test", streaming=True)
        chunks, ref, sr = [], [], None
        for i, sample in enumerate(ds):
            if i >= AMI_TURNS:
                break
            sr = sample["audio"]["sampling_rate"]; chunks.append(np.asarray(sample["audio"]["array"], dtype=np.float32))
            if sample.get("text"):
                ref.append(sample["text"])
        if chunks:
            out_wav, out_ref = CLIPS_DIR / "ami_meeting.wav", CLIPS_DIR / "ami_meeting_ref.txt"
            sf.write(str(out_wav), np.concatenate(chunks), sr); out_ref.write_text(" ".join(ref), encoding="utf-8")
            print(f"saved {out_wav} ({sum(len(c) for c in chunks) / sr:.1f}s) and {out_ref}")
            print("-> set AUDIO_PATH and REFERENCE_TXT to these in the Inputs cell and run again.")
        else:
            print("AMI stream returned no samples.")
    except Exception as e:
        print(f"AMI clip not built: {type(e).__name__}: {str(e)[:300]}")
else:
    print("BUILD_AMI_CLIP is off.")
